In [1]:
# ============================================================
# CELL 1 — PROJECT CONFIGURATION & DIRECTORY SETUP
# ============================================================

from pathlib import Path
import json
import os
import gc

# ------------------------------------------------------------
# 1. Project root
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd()

# ------------------------------------------------------------
# 2. Directory structure
# ------------------------------------------------------------

DIRS = {
    "data_raw": PROJECT_ROOT / "data" / "raw",
    "data_processed": PROJECT_ROOT / "data" / "processed",
    "data_chunks": PROJECT_ROOT / "data" / "chunks",
    "embeddings": PROJECT_ROOT / "embeddings",
    "vectorstore": PROJECT_ROOT / "vectorstore",
    "evaluation": PROJECT_ROOT / "evaluation",
    "results": PROJECT_ROOT / "results",
    "logs": PROJECT_ROOT / "logs",
    "checkpoints": PROJECT_ROOT / "checkpoints",
}

# Create directories safely
for directory in DIRS.values():
    directory.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 3. Dataset configuration
# ------------------------------------------------------------

DATASET_FILENAME = "synthetic_knowledge_items.csv"

DATASET_PATH = Path("E:/rag/synthetic_knowledge_items.csv")

# ------------------------------------------------------------
# 4. RAG baseline configuration
# ------------------------------------------------------------

CONFIG = {
    "project": {
        "name": "Synthetic IT Knowledge RAG",
        "dataset_name": "Synthetic IT-Related Knowledge Items",
        "dataset_version": "Version 3",
        "expected_records": 100,
        "expected_columns": 4,
    },

    "dataset": {
        "filename": DATASET_FILENAME,
        "raw_path": str(DATASET_PATH),
    },

    "chunking": {
        "chunk_size": 500,
        "chunk_overlap": 50,
    },

    "retrieval": {
        "method": "dense",
        "top_k": 5,
    },

    "generation": {
        "provider": "ollama",
        "base_url": "http://localhost:11434",
        "model": None,  # Set after checking installed Ollama models
    },
}

# ------------------------------------------------------------
# 5. Persistent configuration checkpoint
# ------------------------------------------------------------

CONFIG_CHECKPOINT = DIRS["checkpoints"] / "project_config.json"

try:
    with open(CONFIG_CHECKPOINT, "w", encoding="utf-8") as f:
        json.dump(CONFIG, f, indent=4)

    print("[SUCCESS] Project configuration saved.")
    
except Exception as e:
    print("[ERROR] Failed to save project configuration.")
    print(f"Error type: {type(e).__name__}")
    print(f"Error details: {e}")
    raise

# ------------------------------------------------------------
# 6. Environment summary
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("RAG PROJECT INITIALIZATION")
print("=" * 60)

print(f"Project root       : {PROJECT_ROOT}")
print(f"Dataset path       : {DATASET_PATH}")
print(f"Dataset expected   : {CONFIG['project']['expected_records']} records")
print(f"Expected columns   : {CONFIG['project']['expected_columns']}")
print(f"Chunk size         : {CONFIG['chunking']['chunk_size']}")
print(f"Chunk overlap      : {CONFIG['chunking']['chunk_overlap']}")
print(f"Retrieval method   : {CONFIG['retrieval']['method']}")
print(f"Top-K              : {CONFIG['retrieval']['top_k']}")
print(f"Ollama URL         : {CONFIG['generation']['base_url']}")

print("=" * 60)
print("[SUCCESS] Initialization completed.")

[SUCCESS] Project configuration saved.

RAG PROJECT INITIALIZATION
Project root       : e:\rag
Dataset path       : E:\rag\synthetic_knowledge_items.csv
Dataset expected   : 100 records
Expected columns   : 4
Chunk size         : 500
Chunk overlap      : 50
Retrieval method   : dense
Top-K              : 5
Ollama URL         : http://localhost:11434
[SUCCESS] Initialization completed.


In [2]:
# ============================================================
# CELL 2 — OLLAMA CONNECTION & MODEL DISCOVERY
# ============================================================

import requests
import json
from pathlib import Path

# ------------------------------------------------------------
# 1. Ollama configuration
# ------------------------------------------------------------

OLLAMA_BASE_URL = CONFIG["generation"]["base_url"]
OLLAMA_TAGS_URL = f"{OLLAMA_BASE_URL}/api/tags"

OLLAMA_CHECKPOINT = (
    DIRS["checkpoints"] / "ollama_environment.json"
)

# ------------------------------------------------------------
# 2. Check Ollama server
# ------------------------------------------------------------

print("=" * 60)
print("OLLAMA CONNECTION CHECK")
print("=" * 60)

try:
    response = requests.get(
        OLLAMA_TAGS_URL,
        timeout=10
    )

    response.raise_for_status()

    ollama_data = response.json()

    print("[SUCCESS] Ollama server is running.")
    print(f"[INFO] Ollama URL: {OLLAMA_BASE_URL}")

except requests.exceptions.ConnectionError:
    print("[ERROR] Could not connect to Ollama.")
    print(f"[ERROR] Expected Ollama at: {OLLAMA_BASE_URL}")
    print()
    print("Make sure Ollama is running on your computer.")
    raise

except requests.exceptions.Timeout:
    print("[ERROR] Ollama connection timed out.")
    print(f"[ERROR] URL: {OLLAMA_BASE_URL}")
    raise

except requests.exceptions.HTTPError as e:
    print("[ERROR] Ollama returned an HTTP error.")
    print(f"[ERROR] Status code: {response.status_code}")
    print(f"[ERROR] Details: {e}")
    raise

except Exception as e:
    print("[ERROR] Unexpected Ollama connection error.")
    print(f"[ERROR] Error type: {type(e).__name__}")
    print(f"[ERROR] Details: {e}")
    raise

# ------------------------------------------------------------
# 3. Extract installed models
# ------------------------------------------------------------

try:
    installed_models = ollama_data.get("models", [])

    if not installed_models:
        raise RuntimeError(
            "Ollama is running, but no local models were found."
        )

    model_names = [
        model.get("name")
        for model in installed_models
        if model.get("name")
    ]

    print(f"\n[INFO] Installed Ollama models: {len(model_names)}")

    for index, model_name in enumerate(model_names, start=1):
        print(f"  {index}. {model_name}")

except Exception as e:
    print("[ERROR] Failed to inspect Ollama models.")
    print(f"[ERROR] Error type: {type(e).__name__}")
    print(f"[ERROR] Details: {e}")
    raise

# ------------------------------------------------------------
# 4. Display model information
# ------------------------------------------------------------

print("\n" + "-" * 60)
print("MODEL INFORMATION")
print("-" * 60)

for model in installed_models:
    name = model.get("name", "Unknown")
    size = model.get("size", 0)

    # Convert bytes → GB
    size_gb = size / (1024 ** 3) if size else 0

    print(f"Model : {name}")
    print(f"Size  : {size_gb:.2f} GB")
    print()

# ------------------------------------------------------------
# 5. Save Ollama environment checkpoint
# ------------------------------------------------------------

ollama_environment = {
    "base_url": OLLAMA_BASE_URL,
    "server_status": "connected",
    "installed_models": model_names,
    "model_details": installed_models,
}

try:
    with open(
        OLLAMA_CHECKPOINT,
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            ollama_environment,
            f,
            indent=4
        )

    print("[SUCCESS] Ollama environment checkpoint saved.")
    print(f"[INFO] Checkpoint: {OLLAMA_CHECKPOINT}")

except Exception as e:
    print("[ERROR] Failed to save Ollama checkpoint.")
    print(f"[ERROR] Error type: {type(e).__name__}")
    print(f"[ERROR] Details: {e}")
    raise

print("=" * 60)
print("[SUCCESS] Ollama environment is ready.")
print("=" * 60)

OLLAMA CONNECTION CHECK
[SUCCESS] Ollama server is running.
[INFO] Ollama URL: http://localhost:11434

[INFO] Installed Ollama models: 5
  1. qwen3:8b
  2. qwen3.8:27b-mtp-q4_K_M
  3. qwen3.8:27b
  4. gemma4:12b
  5. gemma4:26b

------------------------------------------------------------
MODEL INFORMATION
------------------------------------------------------------
Model : qwen3:8b
Size  : 4.87 GB

Model : qwen3.8:27b-mtp-q4_K_M
Size  : 16.52 GB

Model : qwen3.8:27b
Size  : 16.52 GB

Model : gemma4:12b
Size  : 7.04 GB

Model : gemma4:26b
Size  : 16.75 GB

[SUCCESS] Ollama environment checkpoint saved.
[INFO] Checkpoint: e:\rag\checkpoints\ollama_environment.json
[SUCCESS] Ollama environment is ready.


In [3]:
# ============================================================
# CELL 3 — DATASET LOADING & INITIAL VALIDATION
# ============================================================

import pandas as pd
import json
from pathlib import Path

# ------------------------------------------------------------
# 1. Verify dataset path
# ------------------------------------------------------------

print("=" * 60)
print("DATASET LOADING & VALIDATION")
print("=" * 60)

try:
    if not DATASET_PATH.exists():
        raise FileNotFoundError(
            f"Dataset not found at: {DATASET_PATH}"
        )

    if not DATASET_PATH.is_file():
        raise FileNotFoundError(
            f"Dataset path is not a file: {DATASET_PATH}"
        )

    print(f"[SUCCESS] Dataset found:")
    print(f"         {DATASET_PATH}")

except Exception as e:
    print("[ERROR] Dataset verification failed.")
    print(f"Error type: {type(e).__name__}")
    print(f"Error details: {e}")
    raise


# ------------------------------------------------------------
# 2. Load dataset
# ------------------------------------------------------------

try:
    df_raw = pd.read_csv(DATASET_PATH)

    if df_raw.empty:
        raise ValueError("Dataset is empty.")

    print("\n[SUCCESS] Dataset loaded successfully.")

except Exception as e:
    print("[ERROR] Failed to load dataset.")
    print(f"Error type: {type(e).__name__}")
    print(f"Error details: {e}")
    raise


# ------------------------------------------------------------
# 3. Basic dimensions
# ------------------------------------------------------------

try:
    n_rows, n_columns = df_raw.shape

    print("\n" + "-" * 60)
    print("DATASET DIMENSIONS")
    print("-" * 60)

    print(f"Rows    : {n_rows:,}")
    print(f"Columns : {n_columns}")

except Exception as e:
    print("[ERROR] Failed to inspect dataset dimensions.")
    print(f"Error type: {type(e).__name__}")
    print(f"Error details: {e}")
    raise


# ------------------------------------------------------------
# 4. Validate expected dimensions
# ------------------------------------------------------------

EXPECTED_ROWS = CONFIG["project"]["expected_records"]
EXPECTED_COLUMNS = CONFIG["project"]["expected_columns"]

if n_rows != EXPECTED_ROWS:
    print(
        f"[WARNING] Expected {EXPECTED_ROWS} rows "
        f"but found {n_rows}."
    )
else:
    print(f"[SUCCESS] Record count matches expected value: {EXPECTED_ROWS}")

if n_columns != EXPECTED_COLUMNS:
    print(
        f"[WARNING] Expected {EXPECTED_COLUMNS} columns "
        f"but found {n_columns}."
    )
else:
    print(
        f"[SUCCESS] Column count matches expected value: "
        f"{EXPECTED_COLUMNS}"
    )


# ------------------------------------------------------------
# 5. Exact column names
# ------------------------------------------------------------

print("\n" + "-" * 60)
print("COLUMN INFORMATION")
print("-" * 60)

for index, column in enumerate(df_raw.columns, start=1):
    print(f"{index}. {column}")


# ------------------------------------------------------------
# 6. Data types
# ------------------------------------------------------------

print("\n" + "-" * 60)
print("DATA TYPES")
print("-" * 60)

print(df_raw.dtypes)


# ------------------------------------------------------------
# 7. Missing-value analysis
# ------------------------------------------------------------

print("\n" + "-" * 60)
print("MISSING VALUE ANALYSIS")
print("-" * 60)

missing_counts = df_raw.isna().sum()

missing_table = pd.DataFrame({
    "column": missing_counts.index,
    "missing_count": missing_counts.values,
    "missing_percentage": (
        missing_counts.values / len(df_raw) * 100
    )
})

print(missing_table.to_string(index=False))


# ------------------------------------------------------------
# 8. Duplicate analysis
# ------------------------------------------------------------

try:
    duplicate_count = df_raw.duplicated().sum()

    print("\n" + "-" * 60)
    print("DUPLICATE ANALYSIS")
    print("-" * 60)

    print(f"Duplicate rows: {duplicate_count:,}")

except Exception as e:
    print("[ERROR] Duplicate analysis failed.")
    print(f"Error type: {type(e).__name__}")
    print(f"Error details: {e}")
    raise


# ------------------------------------------------------------
# 9. Sample records
# ------------------------------------------------------------

print("\n" + "-" * 60)
print("SAMPLE RECORDS")
print("-" * 60)

display(df_raw.head())


# ------------------------------------------------------------
# 10. Dataset memory usage
# ------------------------------------------------------------

memory_mb = df_raw.memory_usage(deep=True).sum() / (1024 ** 2)

print("\n" + "-" * 60)
print("MEMORY USAGE")
print("-" * 60)

print(f"DataFrame memory usage: {memory_mb:.2f} MB")


# ------------------------------------------------------------
# 11. Create dataset inspection checkpoint
# ------------------------------------------------------------

dataset_inspection = {
    "dataset_path": str(DATASET_PATH),
    "dataset_name": CONFIG["project"]["dataset_name"],
    "dataset_version": CONFIG["project"]["dataset_version"],
    "rows": int(n_rows),
    "columns": int(n_columns),
    "column_names": list(df_raw.columns),
    "data_types": {
        column: str(dtype)
        for column, dtype in df_raw.dtypes.items()
    },
    "missing_values": {
        column: int(count)
        for column, count in missing_counts.items()
    },
    "duplicate_rows": int(duplicate_count),
    "memory_mb": round(memory_mb, 4),
}

DATASET_INSPECTION_CHECKPOINT = (
    DIRS["checkpoints"] / "dataset_inspection.json"
)

try:
    with open(
        DATASET_INSPECTION_CHECKPOINT,
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            dataset_inspection,
            f,
            indent=4
        )

    print("\n[SUCCESS] Dataset inspection checkpoint saved.")
    print(
        f"[INFO] Checkpoint: "
        f"{DATASET_INSPECTION_CHECKPOINT}"
    )

except Exception as e:
    print("[ERROR] Failed to save dataset inspection checkpoint.")
    print(f"Error type: {type(e).__name__}")
    print(f"Error details: {e}")
    raise


# ------------------------------------------------------------
# 12. Final validation summary
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("DATASET VALIDATION SUMMARY")
print("=" * 60)

print(f"Records              : {n_rows:,}")
print(f"Columns              : {n_columns}")
print(f"Missing values       : {int(missing_counts.sum()):,}")
print(f"Duplicate rows       : {duplicate_count:,}")
print(f"Memory usage         : {memory_mb:.2f} MB")
print(f"Original CSV modified: NO")

print("=" * 60)
print("[SUCCESS] Dataset inspection completed.")
print("=" * 60)

DATASET LOADING & VALIDATION
[SUCCESS] Dataset found:
         E:\rag\synthetic_knowledge_items.csv

[SUCCESS] Dataset loaded successfully.

------------------------------------------------------------
DATASET DIMENSIONS
------------------------------------------------------------
Rows    : 100
Columns : 4
[SUCCESS] Record count matches expected value: 100
[SUCCESS] Column count matches expected value: 4

------------------------------------------------------------
COLUMN INFORMATION
------------------------------------------------------------
1. ki_topic
2. ki_text
3. alt_ki_text
4. bad_ki_text

------------------------------------------------------------
DATA TYPES
------------------------------------------------------------
ki_topic       str
ki_text        str
alt_ki_text    str
bad_ki_text    str
dtype: object

------------------------------------------------------------
MISSING VALUE ANALYSIS
------------------------------------------------------------
     column  missing_count 

,ki_topic,ki_text,alt_ki_text,bad_ki_text
0,Setting Up a Mobile Device for Company Email,**Setting Up a Mobile Device for Company Email...,"To set up a mobile device for company email, f...",# Setting Up a Mobile Device for Company Email...
1,Resetting a Forgotten PIN,**Resetting a Forgotten PIN**\n\nIf you have f...,"If you have forgotten your PIN, you can reset ...","# How to Resetting Your Forgot PIN \n\nSo, you..."
2,Configuring VPN Access for Remote Workers,**Configuring VPN Access for Remote Workers**\...,To configure VPN access for remote workers at ...,# How to Set Up VPN Access for Remote Workrs\n...
3,Troubleshooting Issues with Microsoft Office,**Troubleshooting Issues with Microsoft Office...,When troubleshooting issues with Microsoft Off...,# Troubleshooting Issues with Microsoft Office...
4,Setting Up a Conference Call on Cisco Webex,"To set up a conference call on Cisco Webex, fo...","To set up a conference call on Cisco Webex, fo...",# How To Set Up A Conference Call on Cisco Web...



------------------------------------------------------------
MEMORY USAGE
------------------------------------------------------------
DataFrame memory usage: 1.35 MB

[SUCCESS] Dataset inspection checkpoint saved.
[INFO] Checkpoint: e:\rag\checkpoints\dataset_inspection.json

DATASET VALIDATION SUMMARY
Records              : 100
Columns              : 4
Missing values       : 0
Duplicate rows       : 0
Memory usage         : 1.35 MB
Original CSV modified: NO
[SUCCESS] Dataset inspection completed.


In [4]:
# ============================================================
# CELL 4 — DATASET PROFILING & RAG FIELD DEFINITION
# ============================================================

import pandas as pd
import json
from pathlib import Path

print("=" * 60)
print("DATASET PROFILING & RAG FIELD DEFINITION")
print("=" * 60)


# ------------------------------------------------------------
# 1. Define the dataset schema
# ------------------------------------------------------------

RAG_SCHEMA = {
    "document_id_source": "row_index",

    "topic_column": "ki_topic",

    "primary_text_column": "ki_text",

    "alternative_text_column": "alt_ki_text",

    "bad_text_column": "bad_ki_text",
}


# ------------------------------------------------------------
# 2. Validate required columns
# ------------------------------------------------------------

required_columns = [
    RAG_SCHEMA["topic_column"],
    RAG_SCHEMA["primary_text_column"],
    RAG_SCHEMA["alternative_text_column"],
    RAG_SCHEMA["bad_text_column"],
]

try:
    missing_columns = [
        column
        for column in required_columns
        if column not in df_raw.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Required columns are missing: {missing_columns}"
        )

    print("[SUCCESS] All required dataset columns are present.")

except Exception as e:
    print("[ERROR] Dataset schema validation failed.")
    print(f"Error type: {type(e).__name__}")
    print(f"Error details: {e}")
    raise


# ------------------------------------------------------------
# 3. Create a non-destructive profiling copy
# ------------------------------------------------------------

try:
    df_profile = df_raw.copy(deep=True)

    print("[SUCCESS] Profiling copy created.")
    print("[INFO] Original dataset remains unchanged.")

except Exception as e:
    print("[ERROR] Failed to create profiling copy.")
    print(f"Error type: {type(e).__name__}")
    print(f"Error details: {e}")
    raise


# ------------------------------------------------------------
# 4. Calculate character lengths
# ------------------------------------------------------------

try:
    df_profile["topic_char_length"] = (
        df_profile[RAG_SCHEMA["topic_column"]]
        .astype(str)
        .str.len()
    )

    df_profile["primary_char_length"] = (
        df_profile[RAG_SCHEMA["primary_text_column"]]
        .astype(str)
        .str.len()
    )

    df_profile["alternative_char_length"] = (
        df_profile[RAG_SCHEMA["alternative_text_column"]]
        .astype(str)
        .str.len()
    )

    df_profile["bad_char_length"] = (
        df_profile[RAG_SCHEMA["bad_text_column"]]
        .astype(str)
        .str.len()
    )

    print("[SUCCESS] Text-length profiling completed.")

except Exception as e:
    print("[ERROR] Text-length calculation failed.")
    print(f"Error type: {type(e).__name__}")
    print(f"Error details: {e}")
    raise


# ------------------------------------------------------------
# 5. Generate descriptive statistics
# ------------------------------------------------------------

length_columns = [
    "topic_char_length",
    "primary_char_length",
    "alternative_char_length",
    "bad_char_length",
]

try:
    length_statistics = (
        df_profile[length_columns]
        .describe()
        .round(2)
    )

    print("\n" + "-" * 60)
    print("TEXT LENGTH STATISTICS")
    print("-" * 60)

    display(length_statistics)

except Exception as e:
    print("[ERROR] Failed to calculate text statistics.")
    print(f"Error type: {type(e).__name__}")
    print(f"Error details: {e}")
    raise


# ------------------------------------------------------------
# 6. Display min/max examples
# ------------------------------------------------------------

try:
    print("\n" + "-" * 60)
    print("TEXT LENGTH RANGE")
    print("-" * 60)

    for column in length_columns:
        print(
            f"{column:25s}: "
            f"min={df_profile[column].min():,} | "
            f"max={df_profile[column].max():,} | "
            f"mean={df_profile[column].mean():,.2f}"
        )

except Exception as e:
    print("[ERROR] Failed to display length ranges.")
    print(f"Error type: {type(e).__name__}")
    print(f"Error details: {e}")
    raise


# ------------------------------------------------------------
# 7. Check whether text fields are actually distinct
# ------------------------------------------------------------

try:
    primary_equals_alternative = (
        df_raw[RAG_SCHEMA["primary_text_column"]]
        ==
        df_raw[RAG_SCHEMA["alternative_text_column"]]
    ).sum()

    primary_equals_bad = (
        df_raw[RAG_SCHEMA["primary_text_column"]]
        ==
        df_raw[RAG_SCHEMA["bad_text_column"]]
    ).sum()

    alternative_equals_bad = (
        df_raw[RAG_SCHEMA["alternative_text_column"]]
        ==
        df_raw[RAG_SCHEMA["bad_text_column"]]
    ).sum()

    print("\n" + "-" * 60)
    print("TEXT VERSION COMPARISON")
    print("-" * 60)

    print(
        f"Primary == Alternative : "
        f"{primary_equals_alternative:,} / {len(df_raw):,}"
    )

    print(
        f"Primary == Bad         : "
        f"{primary_equals_bad:,} / {len(df_raw):,}"
    )

    print(
        f"Alternative == Bad     : "
        f"{alternative_equals_bad:,} / {len(df_raw):,}"
    )

except Exception as e:
    print("[ERROR] Text version comparison failed.")
    print(f"Error type: {type(e).__name__}")
    print(f"Error details: {e}")
    raise


# ------------------------------------------------------------
# 8. Inspect topics
# ------------------------------------------------------------

try:
    unique_topics = df_raw[
        RAG_SCHEMA["topic_column"]
    ].nunique()

    print("\n" + "-" * 60)
    print("TOPIC ANALYSIS")
    print("-" * 60)

    print(f"Total records : {len(df_raw):,}")
    print(f"Unique topics : {unique_topics:,}")

    if unique_topics != len(df_raw):
        print(
            "[WARNING] Multiple records share the same topic."
        )
    else:
        print(
            "[SUCCESS] Every knowledge item has a unique topic."
        )

except Exception as e:
    print("[ERROR] Topic analysis failed.")
    print(f"Error type: {type(e).__name__}")
    print(f"Error details: {e}")
    raise


# ------------------------------------------------------------
# 9. Define baseline RAG document strategy
# ------------------------------------------------------------

DOCUMENT_STRATEGY = {
    "baseline_corpus": "primary",
    "topic_field": RAG_SCHEMA["topic_column"],
    "content_field": RAG_SCHEMA["primary_text_column"],

    "excluded_from_baseline": [
        RAG_SCHEMA["alternative_text_column"],
        RAG_SCHEMA["bad_text_column"],
    ],

    "alternative_text_usage": (
        "Reserved for controlled experiments/evaluation."
    ),

    "bad_text_usage": (
        "Reserved for robustness/adversarial experiments "
        "and evaluation."
    ),
}


# ------------------------------------------------------------
# 10. Save schema and profiling checkpoint
# ------------------------------------------------------------

PROFILE_CHECKPOINT = (
    DIRS["checkpoints"] / "dataset_profile.json"
)

profile_checkpoint = {
    "schema": RAG_SCHEMA,
    "document_strategy": DOCUMENT_STRATEGY,

    "record_count": int(len(df_raw)),
    "unique_topics": int(unique_topics),

    "text_statistics": {
        column: {
            metric: float(length_statistics.loc[metric, column])
            for metric in length_statistics.index
        }
        for column in length_statistics.columns
    },

    "version_comparison": {
        "primary_equals_alternative": int(
            primary_equals_alternative
        ),
        "primary_equals_bad": int(
            primary_equals_bad
        ),
        "alternative_equals_bad": int(
            alternative_equals_bad
        ),
    },
}

try:
    with open(
        PROFILE_CHECKPOINT,
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            profile_checkpoint,
            f,
            indent=4
        )

    print("\n[SUCCESS] Dataset profile checkpoint saved.")
    print(f"[INFO] Checkpoint: {PROFILE_CHECKPOINT}")

except Exception as e:
    print("[ERROR] Failed to save dataset profile checkpoint.")
    print(f"Error type: {type(e).__name__}")
    print(f"Error details: {e}")
    raise


# ------------------------------------------------------------
# 11. Final strategy summary
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("BASELINE RAG DOCUMENT STRATEGY")
print("=" * 60)

print(f"Topic field       : {RAG_SCHEMA['topic_column']}")
print(f"Primary text      : {RAG_SCHEMA['primary_text_column']}")
print(f"Alternative text  : {RAG_SCHEMA['alternative_text_column']}")
print(f"Bad text          : {RAG_SCHEMA['bad_text_column']}")

print("\nBaseline corpus:")
print("  ki_topic + ki_text")

print("\nExcluded from baseline:")
print("  alt_ki_text")
print("  bad_ki_text")

print("\n[SUCCESS] Dataset profiling completed.")
print("=" * 60)

DATASET PROFILING & RAG FIELD DEFINITION
[SUCCESS] All required dataset columns are present.
[SUCCESS] Profiling copy created.
[INFO] Original dataset remains unchanged.
[SUCCESS] Text-length profiling completed.

------------------------------------------------------------
TEXT LENGTH STATISTICS
------------------------------------------------------------


,topic_char_length,primary_char_length,alternative_char_length,bad_char_length
count,100.00,100.00,100.00,100.00
mean,41.55,2581.34,2440.64,2921.60
std,9.77,361.60,372.04,319.16
min,25.00,1606.00,1526.00,2326.00
25%,33.00,2327.25,2240.50,2756.25
50%,41.00,2573.50,2415.00,2923.00
75%,50.00,2824.25,2660.50,3108.00
max,64.00,3730.00,3456.00,3904.00



------------------------------------------------------------
TEXT LENGTH RANGE
------------------------------------------------------------
topic_char_length        : min=25 | max=64 | mean=41.55
primary_char_length      : min=1,606 | max=3,730 | mean=2,581.34
alternative_char_length  : min=1,526 | max=3,456 | mean=2,440.64
bad_char_length          : min=2,326 | max=3,904 | mean=2,921.60

------------------------------------------------------------
TEXT VERSION COMPARISON
------------------------------------------------------------
Primary == Alternative : 0 / 100
Primary == Bad         : 0 / 100
Alternative == Bad     : 0 / 100

------------------------------------------------------------
TOPIC ANALYSIS
------------------------------------------------------------
Total records : 100
Unique topics : 98
[WARNING] Multiple records share the same topic.

[SUCCESS] Dataset profile checkpoint saved.
[INFO] Checkpoint: e:\rag\checkpoints\dataset_profile.json

BASELINE RAG DOCUMENT STRATEGY


In [5]:
# ============================================================
# CELL 5 — RAG DOCUMENT CREATION & PERSISTENCE
# ============================================================

import json
import re
import gc
from pathlib import Path

print("=" * 60)
print("RAG DOCUMENT CREATION")
print("=" * 60)


# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

TOPIC_COLUMN = RAG_SCHEMA["topic_column"]
TEXT_COLUMN = RAG_SCHEMA["primary_text_column"]

DOCUMENTS_CHECKPOINT = (
    DIRS["data_processed"] / "rag_documents.json"
)


# ------------------------------------------------------------
# 2. Text cleaning function
# ------------------------------------------------------------

def clean_document_text(text):
    """
    Perform conservative text cleaning.

    The purpose is to remove accidental formatting noise
    while preserving the actual knowledge content.
    """

    if text is None:
        return ""

    text = str(text)

    # Normalize Windows line endings
    text = text.replace("\r\n", "\n")
    text = text.replace("\r", "\n")

    # Remove trailing whitespace from each line
    text = "\n".join(
        line.rstrip()
        for line in text.split("\n")
    )

    # Collapse excessive blank lines
    text = re.sub(
        r"\n{3,}",
        "\n\n",
        text
    )

    # Remove unnecessary leading/trailing whitespace
    text = text.strip()

    return text


# ------------------------------------------------------------
# 3. Validate source data before document creation
# ------------------------------------------------------------

try:
    if df_raw.empty:
        raise ValueError(
            "Source DataFrame is empty."
        )

    if TOPIC_COLUMN not in df_raw.columns:
        raise KeyError(
            f"Topic column '{TOPIC_COLUMN}' not found."
        )

    if TEXT_COLUMN not in df_raw.columns:
        raise KeyError(
            f"Text column '{TEXT_COLUMN}' not found."
        )

    if df_raw[TOPIC_COLUMN].isna().any():
        raise ValueError(
            "Topic column contains missing values."
        )

    if df_raw[TEXT_COLUMN].isna().any():
        raise ValueError(
            "Primary text column contains missing values."
        )

    print("[SUCCESS] Source dataset validation passed.")

except Exception as e:
    print("[ERROR] Source validation failed.")
    print(f"Error type: {type(e).__name__}")
    print(f"Error details: {e}")
    raise


# ------------------------------------------------------------
# 4. Create RAG documents
# ------------------------------------------------------------

rag_documents = []

try:

    for row_index, row in df_raw.iterrows():

        topic = str(row[TOPIC_COLUMN]).strip()
        original_text = str(row[TEXT_COLUMN])

        cleaned_text = clean_document_text(
            original_text
        )

        # Validate cleaned content
        if not topic:
            raise ValueError(
                f"Empty topic at row {row_index}."
            )

        if not cleaned_text:
            raise ValueError(
                f"Empty document text at row {row_index}."
            )

        document_id = f"doc_{row_index + 1:03d}"

        document = {
            "document_id": document_id,

            "metadata": {
                "topic": topic,
                "source": CONFIG["project"]["dataset_name"],
                "dataset_version": (
                    CONFIG["project"]["dataset_version"]
                ),
                "original_row_index": int(row_index),
            },

            "text": cleaned_text,
        }

        rag_documents.append(document)

    print(
        f"[SUCCESS] Created {len(rag_documents):,} "
        f"RAG documents."
    )

except Exception as e:
    print("[ERROR] RAG document creation failed.")
    print(f"Error type: {type(e).__name__}")
    print(f"Error details: {e}")
    raise


# ------------------------------------------------------------
# 5. Validate generated documents
# ------------------------------------------------------------

try:

    if len(rag_documents) != len(df_raw):
        raise ValueError(
            "Document count does not match source "
            "record count."
        )

    document_ids = [
        document["document_id"]
        for document in rag_documents
    ]

    if len(document_ids) != len(set(document_ids)):
        raise ValueError(
            "Duplicate document IDs detected."
        )

    for document in rag_documents:

        required_keys = {
            "document_id",
            "metadata",
            "text",
        }

        if not required_keys.issubset(
            document.keys()
        ):
            raise ValueError(
                f"Invalid document structure: "
                f"{document.get('document_id')}"
            )

        if not document["text"].strip():
            raise ValueError(
                f"Empty text in "
                f"{document['document_id']}"
            )

        if not document["metadata"]["topic"].strip():
            raise ValueError(
                f"Empty topic in "
                f"{document['document_id']}"
            )

    print("[SUCCESS] Document validation passed.")
    print(f"[INFO] Valid documents: {len(rag_documents):,}")

except Exception as e:
    print("[ERROR] Document validation failed.")
    print(f"Error type: {type(e).__name__}")
    print(f"Error details: {e}")
    raise


# ------------------------------------------------------------
# 6. Calculate document statistics
# ------------------------------------------------------------

try:

    document_lengths = [
        len(document["text"])
        for document in rag_documents
    ]

    total_characters = sum(
        document_lengths
    )

    average_length = (
        total_characters / len(document_lengths)
    )

    min_length = min(document_lengths)
    max_length = max(document_lengths)

    print("\n" + "-" * 60)
    print("PREPARED DOCUMENT STATISTICS")
    print("-" * 60)

    print(
        f"Documents           : "
        f"{len(rag_documents):,}"
    )

    print(
        f"Total characters    : "
        f"{total_characters:,}"
    )

    print(
        f"Average characters  : "
        f"{average_length:,.2f}"
    )

    print(
        f"Minimum characters  : "
        f"{min_length:,}"
    )

    print(
        f"Maximum characters  : "
        f"{max_length:,}"
    )

except Exception as e:
    print("[ERROR] Document statistics failed.")
    print(f"Error type: {type(e).__name__}")
    print(f"Error details: {e}")
    raise


# ------------------------------------------------------------
# 7. Preview prepared documents
# ------------------------------------------------------------

print("\n" + "-" * 60)
print("DOCUMENT PREVIEW")
print("-" * 60)

for document in rag_documents[:3]:

    print(
        f"\nDocument ID : "
        f"{document['document_id']}"
    )

    print(
        f"Topic       : "
        f"{document['metadata']['topic']}"
    )

    preview = document["text"][:500]

    print("Text preview:")
    print(preview)

    if len(document["text"]) > 500:
        print("...")


# ------------------------------------------------------------
# 8. Save documents using temporary file
# ------------------------------------------------------------

TEMP_DOCUMENTS_CHECKPOINT = (
    DOCUMENTS_CHECKPOINT.with_suffix(".tmp")
)

try:

    with open(
        TEMP_DOCUMENTS_CHECKPOINT,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            rag_documents,
            f,
            ensure_ascii=False,
            indent=2
        )

    # Replace old checkpoint only after successful write
    TEMP_DOCUMENTS_CHECKPOINT.replace(
        DOCUMENTS_CHECKPOINT
    )

    print(
        "\n[SUCCESS] RAG documents checkpoint saved."
    )

    print(
        f"[INFO] Path: {DOCUMENTS_CHECKPOINT}"
    )

except Exception as e:

    # Remove incomplete temporary file
    if TEMP_DOCUMENTS_CHECKPOINT.exists():
        TEMP_DOCUMENTS_CHECKPOINT.unlink(
            missing_ok=True
        )

    print(
        "[ERROR] Failed to save RAG document checkpoint."
    )

    print(
        f"Error type: {type(e).__name__}"
    )

    print(
        f"Error details: {e}"
    )

    raise


# ------------------------------------------------------------
# 9. Final integrity summary
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("RAG DOCUMENT PREPARATION COMPLETE")
print("=" * 60)

print(
    f"Source records       : {len(df_raw):,}"
)

print(
    f"Prepared documents   : {len(rag_documents):,}"
)

print(
    f"Unique document IDs  : "
    f"{len(set(document_ids)):,}"
)

print(
    f"Baseline text field  : {TEXT_COLUMN}"
)

print(
    "Alternative text     : NOT included"
)

print(
    "Bad text             : NOT included"
)

print(
    "Original dataset     : UNCHANGED"
)

print("=" * 60)
print("[SUCCESS] Document layer is ready.")
print("=" * 60)


# ------------------------------------------------------------
# 10. Memory cleanup
# ------------------------------------------------------------

del df_profile
gc.collect()

print("[INFO] Temporary profiling objects released.")

RAG DOCUMENT CREATION
[SUCCESS] Source dataset validation passed.
[SUCCESS] Created 100 RAG documents.
[SUCCESS] Document validation passed.
[INFO] Valid documents: 100

------------------------------------------------------------
PREPARED DOCUMENT STATISTICS
------------------------------------------------------------
Documents           : 100
Total characters    : 258,134
Average characters  : 2,581.34
Minimum characters  : 1,606
Maximum characters  : 3,730

------------------------------------------------------------
DOCUMENT PREVIEW
------------------------------------------------------------

Document ID : doc_001
Topic       : Setting Up a Mobile Device for Company Email
Text preview:
**Setting Up a Mobile Device for Company Email**

**Prerequisites:**

* Mobile device with a supported operating system (iOS, Android, or Windows)
* Company email account credentials
* Mobile device management (MDM) profile installed (if required by company policy)

**Step 1: Ensure Mobile Device Ma

In [6]:
# ============================================================
# CELL 6 — TOKENIZATION & CHUNKING FEASIBILITY ANALYSIS
# ============================================================

import json
import gc
from pathlib import Path

print("=" * 60)
print("TOKENIZATION & CHUNKING FEASIBILITY ANALYSIS")
print("=" * 60)


# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

DOCUMENTS_CHECKPOINT = (
    DIRS["data_processed"] / "rag_documents.json"
)

CHUNK_SIZE = CONFIG["chunking"]["chunk_size"]
CHUNK_OVERLAP = CONFIG["chunking"]["chunk_overlap"]

TOKEN_ANALYSIS_CHECKPOINT = (
    DIRS["checkpoints"] / "tokenization_analysis.json"
)


# ------------------------------------------------------------
# 2. Load persisted documents
# ------------------------------------------------------------

try:

    if not DOCUMENTS_CHECKPOINT.exists():
        raise FileNotFoundError(
            f"Document checkpoint not found: "
            f"{DOCUMENTS_CHECKPOINT}"
        )

    with open(
        DOCUMENTS_CHECKPOINT,
        "r",
        encoding="utf-8"
    ) as f:

        persisted_documents = json.load(f)

    if not persisted_documents:
        raise ValueError(
            "Persisted document list is empty."
        )

    print(
        f"[SUCCESS] Loaded {len(persisted_documents):,} "
        "persisted RAG documents."
    )

except Exception as e:

    print(
        "[ERROR] Failed to load persisted RAG documents."
    )

    print(
        f"Error type: {type(e).__name__}"
    )

    print(
        f"Error details: {e}"
    )

    raise


# ------------------------------------------------------------
# 3. Load tokenizer
# ------------------------------------------------------------

tokenizer = None
TOKENIZER_NAME = None

try:

    # tiktoken provides a stable tokenizer for
    # token-count analysis.
    import tiktoken

    try:
        tokenizer = tiktoken.get_encoding("cl100k_base")
        TOKENIZER_NAME = "cl100k_base"

    except Exception:

        tokenizer = tiktoken.get_encoding(
            "o200k_base"
        )
        TOKENIZER_NAME = "o200k_base"

    print(
        f"[SUCCESS] Tokenizer loaded: "
        f"{TOKENIZER_NAME}"
    )

except ImportError:

    print(
        "[WARNING] tiktoken is not installed."
    )

    print(
        "[INFO] Installing/loading a tokenizer will "
        "be handled before chunk generation."
    )

except Exception as e:

    print(
        "[WARNING] Tokenizer initialization failed."
    )

    print(
        f"Error type: {type(e).__name__}"
    )

    print(
        f"Error details: {e}"
    )


# ------------------------------------------------------------
# 4. Token count analysis
# ------------------------------------------------------------

try:

    if tokenizer is None:

        raise RuntimeError(
            "No tokenizer is available. "
            "Cannot perform token-level analysis."
        )

    token_statistics = []

    for document in persisted_documents:

        text = document["text"]

        token_ids = tokenizer.encode(
            text,
            disallowed_special=()
        )

        token_count = len(token_ids)

        character_count = len(text)

        token_statistics.append({
            "document_id": document["document_id"],
            "topic": document["metadata"]["topic"],
            "character_count": character_count,
            "token_count": token_count,
            "characters_per_token": (
                character_count / token_count
                if token_count > 0
                else 0
            ),
        })

    print(
        "[SUCCESS] Token counts calculated for all "
        f"{len(token_statistics):,} documents."
    )

except Exception as e:

    print(
        "[ERROR] Token analysis failed."
    )

    print(
        f"Error type: {type(e).__name__}"
    )

    print(
        f"Error details: {e}"
    )

    raise


# ------------------------------------------------------------
# 5. Statistical analysis
# ------------------------------------------------------------

try:

    token_counts = [
        item["token_count"]
        for item in token_statistics
    ]

    character_counts = [
        item["character_count"]
        for item in token_statistics
    ]

    ratios = [
        item["characters_per_token"]
        for item in token_statistics
    ]

    token_counts_sorted = sorted(token_counts)

    def percentile(values, percentage):

        if not values:
            return 0

        position = (
            (len(values) - 1)
            * percentage
        )

        lower = int(position)
        upper = min(
            lower + 1,
            len(values) - 1
        )

        fraction = position - lower

        return (
            values[lower]
            + (
                values[upper]
                - values[lower]
            ) * fraction
        )

    token_analysis = {
        "document_count": len(token_counts),

        "tokenizer": TOKENIZER_NAME,

        "chunk_size": CHUNK_SIZE,

        "chunk_overlap": CHUNK_OVERLAP,

        "token_statistics": {
            "minimum": min(token_counts),
            "maximum": max(token_counts),
            "mean": sum(token_counts)
            / len(token_counts),
            "median": percentile(
                token_counts_sorted,
                0.50
            ),
            "p25": percentile(
                token_counts_sorted,
                0.25
            ),
            "p75": percentile(
                token_counts_sorted,
                0.75
            ),
        },

        "character_statistics": {
            "minimum": min(character_counts),
            "maximum": max(character_counts),
            "mean": sum(character_counts)
            / len(character_counts),
        },

        "characters_per_token": {
            "minimum": min(ratios),
            "maximum": max(ratios),
            "mean": sum(ratios)
            / len(ratios),
        },
    }

except Exception as e:

    print(
        "[ERROR] Failed to calculate token statistics."
    )

    print(
        f"Error type: {type(e).__name__}"
    )

    print(
        f"Error details: {e}"
    )

    raise


# ------------------------------------------------------------
# 6. Estimate chunk counts
# ------------------------------------------------------------

try:

    estimated_chunks = []

    for item in token_statistics:

        token_count = item["token_count"]

        if token_count <= CHUNK_SIZE:
            chunks_needed = 1

        else:

            effective_step = (
                CHUNK_SIZE - CHUNK_OVERLAP
            )

            chunks_needed = (
                1
                + (
                    token_count
                    - CHUNK_SIZE
                    + effective_step
                    - 1
                )
                // effective_step
            )

        estimated_chunks.append({
            "document_id": item["document_id"],
            "token_count": token_count,
            "estimated_chunks": int(
                chunks_needed
            ),
        })

    total_estimated_chunks = sum(
        item["estimated_chunks"]
        for item in estimated_chunks
    )

    token_analysis[
        "estimated_chunk_count"
    ] = total_estimated_chunks

except Exception as e:

    print(
        "[ERROR] Chunk estimation failed."
    )

    print(
        f"Error type: {type(e).__name__}"
    )

    print(
        f"Error details: {e}"
    )

    raise


# ------------------------------------------------------------
# 7. Display results
# ------------------------------------------------------------

print("\n" + "-" * 60)
print("TOKENIZATION RESULTS")
print("-" * 60)

print(
    f"Tokenizer             : "
    f"{TOKENIZER_NAME}"
)

print(
    f"Documents             : "
    f"{len(token_counts):,}"
)

print(
    f"Minimum tokens        : "
    f"{min(token_counts):,}"
)

print(
    f"Maximum tokens        : "
    f"{max(token_counts):,}"
)

print(
    f"Average tokens        : "
    f"{sum(token_counts) / len(token_counts):,.2f}"
)

print(
    f"Median tokens         : "
    f"{token_analysis['token_statistics']['median']:,.0f}"
)

print(
    f"Chunk size            : "
    f"{CHUNK_SIZE} tokens"
)

print(
    f"Chunk overlap         : "
    f"{CHUNK_OVERLAP} tokens"
)

print(
    f"Estimated total chunks: "
    f"{total_estimated_chunks:,}"
)


# ------------------------------------------------------------
# 8. Document-level token distribution
# ------------------------------------------------------------

print("\n" + "-" * 60)
print("DOCUMENT TOKEN DISTRIBUTION")
print("-" * 60)

for item in token_statistics[:10]:

    print(
        f"{item['document_id']} | "
        f"{item['token_count']:,} tokens | "
        f"{item['character_count']:,} chars | "
        f"{item['characters_per_token']:.2f} chars/token"
    )


# ------------------------------------------------------------
# 9. Save analysis checkpoint
# ------------------------------------------------------------

token_analysis[
    "document_token_statistics"
] = token_statistics

token_analysis[
    "estimated_document_chunks"
] = estimated_chunks

TEMP_TOKEN_CHECKPOINT = (
    TOKEN_ANALYSIS_CHECKPOINT.with_suffix(
        ".tmp"
    )
)

try:

    with open(
        TEMP_TOKEN_CHECKPOINT,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            token_analysis,
            f,
            indent=4
        )

    TEMP_TOKEN_CHECKPOINT.replace(
        TOKEN_ANALYSIS_CHECKPOINT
    )

    print(
        "\n[SUCCESS] Tokenization analysis "
        "checkpoint saved."
    )

    print(
        f"[INFO] Path: "
        f"{TOKEN_ANALYSIS_CHECKPOINT}"
    )

except Exception as e:

    if TEMP_TOKEN_CHECKPOINT.exists():
        TEMP_TOKEN_CHECKPOINT.unlink(
            missing_ok=True
        )

    print(
        "[ERROR] Failed to save tokenization "
        "checkpoint."
    )

    print(
        f"Error type: {type(e).__name__}"
    )

    print(
        f"Error details: {e}"
    )

    raise


# ------------------------------------------------------------
# 10. Final status
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("TOKENIZATION ANALYSIS COMPLETE")
print("=" * 60)

print(
    "[SUCCESS] No documents were modified."
)

print(
    "[SUCCESS] No chunks were generated yet."
)

print(
    "[SUCCESS] Tokenization analysis persisted."
)

print("=" * 60)


# ------------------------------------------------------------
# 11. Memory cleanup
# ------------------------------------------------------------

del token_ids
gc.collect()

print("[INFO] Temporary tokenization objects released.")

TOKENIZATION & CHUNKING FEASIBILITY ANALYSIS
[SUCCESS] Loaded 100 persisted RAG documents.
[SUCCESS] Tokenizer loaded: cl100k_base
[SUCCESS] Token counts calculated for all 100 documents.

------------------------------------------------------------
TOKENIZATION RESULTS
------------------------------------------------------------
Tokenizer             : cl100k_base
Documents             : 100
Minimum tokens        : 335
Maximum tokens        : 776
Average tokens        : 537.08
Median tokens         : 537
Chunk size            : 500 tokens
Chunk overlap         : 50 tokens
Estimated total chunks: 164

------------------------------------------------------------
DOCUMENT TOKEN DISTRIBUTION
------------------------------------------------------------
doc_001 | 535 tokens | 2,568 chars | 4.80 chars/token
doc_002 | 421 tokens | 1,873 chars | 4.45 chars/token
doc_003 | 528 tokens | 2,607 chars | 4.94 chars/token
doc_004 | 558 tokens | 2,924 chars | 5.24 chars/token
doc_005 | 523 tokens | 2,

In [7]:
# ============================================================
# CELL 7 — CORRECTED TOKEN-AWARE CHUNK GENERATION
# ============================================================

import os
import json
import gc
import tempfile
from pathlib import Path
from statistics import mean, median

print("=" * 60)
print("CORRECTED TOKEN-AWARE CHUNK GENERATION")
print("=" * 60)

# ------------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------------

PROJECT_ROOT = Path(r"E:\rag")

DOCUMENTS_PATH = (
    PROJECT_ROOT / "data" / "processed" / "rag_documents.json"
)

CHUNKS_DIR = PROJECT_ROOT / "data" / "processed"
CHUNKS_PATH = CHUNKS_DIR / "rag_chunks.json"

CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
CHUNK_CHECKPOINT_PATH = (
    CHECKPOINT_DIR / "chunk_generation.json"
)

CHUNK_SIZE = 500
CHUNK_OVERLAP = 50

TOKENIZER_NAME = "cl100k_base"

# ------------------------------------------------------------
# HELPER — ATOMIC JSON SAVE
# ------------------------------------------------------------

def atomic_json_save(data, output_path):

    output_path = Path(output_path)
    output_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    temp_path = None

    try:

        with tempfile.NamedTemporaryFile(
            mode="w",
            encoding="utf-8",
            dir=output_path.parent,
            delete=False,
            suffix=".tmp"
        ) as tmp_file:

            json.dump(
                data,
                tmp_file,
                ensure_ascii=False,
                indent=2
            )

            tmp_file.flush()
            os.fsync(tmp_file.fileno())

            temp_path = Path(tmp_file.name)

        os.replace(
            temp_path,
            output_path
        )

    except Exception:

        if (
            temp_path is not None
            and temp_path.exists()
        ):
            try:
                temp_path.unlink()
            except Exception:
                pass

        raise


# ------------------------------------------------------------
# STEP 1 — LOAD PERSISTED DOCUMENTS
# ------------------------------------------------------------

try:

    if not DOCUMENTS_PATH.exists():

        raise FileNotFoundError(
            f"Persisted document file not found:\n"
            f"{DOCUMENTS_PATH}"
        )

    with open(
        DOCUMENTS_PATH,
        "r",
        encoding="utf-8"
    ) as f:

        documents = json.load(f)

    if not isinstance(documents, list):

        raise ValueError(
            "Persisted documents must be a list."
        )

    if len(documents) != 100:

        raise ValueError(
            f"Expected 100 documents, "
            f"found {len(documents)}."
        )

    print(
        f"[SUCCESS] Loaded {len(documents)} "
        f"persisted RAG documents."
    )

    # --------------------------------------------------------
    # STEP 2 — LOAD TOKENIZER
    # --------------------------------------------------------

    try:

        import tiktoken

    except ImportError as e:

        raise ImportError(
            "tiktoken is required.\n"
            "Install with: pip install tiktoken"
        ) from e

    tokenizer = tiktoken.get_encoding(
        TOKENIZER_NAME
    )

    print(
        f"[SUCCESS] Tokenizer loaded: "
        f"{TOKENIZER_NAME}"
    )

    # --------------------------------------------------------
    # STEP 3 — VALIDATE CONFIGURATION
    # --------------------------------------------------------

    if CHUNK_SIZE <= 0:

        raise ValueError(
            "CHUNK_SIZE must be greater than zero."
        )

    if CHUNK_OVERLAP < 0:

        raise ValueError(
            "CHUNK_OVERLAP cannot be negative."
        )

    if CHUNK_OVERLAP >= CHUNK_SIZE:

        raise ValueError(
            "CHUNK_OVERLAP must be smaller "
            "than CHUNK_SIZE."
        )

    STEP_SIZE = CHUNK_SIZE - CHUNK_OVERLAP

    print()
    print("-" * 60)
    print("CHUNKING CONFIGURATION")
    print("-" * 60)
    print(
        f"Tokenizer       : {TOKENIZER_NAME}"
    )
    print(
        f"Chunk size      : {CHUNK_SIZE} tokens"
    )
    print(
        f"Chunk overlap   : {CHUNK_OVERLAP} tokens"
    )
    print(
        f"Step size       : {STEP_SIZE} tokens"
    )

    # --------------------------------------------------------
    # STEP 4 — GENERATE CORRECTED CHUNKS
    # --------------------------------------------------------

    chunks = []

    for doc in documents:

        document_id = doc.get(
            "document_id"
        )

        document_text = doc.get(
            "text",
            ""
        )

        document_metadata = doc.get(
            "metadata",
            {}
        )

        if not document_id:

            raise ValueError(
                "Document missing document_id."
            )

        if not isinstance(
            document_text,
            str
        ):

            raise ValueError(
                f"{document_id}: text is not a string."
            )

        if not document_text.strip():

            raise ValueError(
                f"{document_id}: empty document text."
            )

        # Encode complete document.
        token_ids = tokenizer.encode(
            document_text
        )

        total_tokens = len(token_ids)

        if total_tokens == 0:

            raise ValueError(
                f"{document_id}: zero tokens."
            )

        chunk_number = 0
        start_token = 0

        while start_token < total_tokens:

            end_token = min(
                start_token + CHUNK_SIZE,
                total_tokens
            )

            chunk_token_ids = token_ids[
                start_token:end_token
            ]

            chunk_text = tokenizer.decode(
                chunk_token_ids
            ).strip()

            if not chunk_text:

                raise ValueError(
                    f"{document_id}: "
                    f"generated empty chunk."
                )

            chunk_number += 1

            chunk_id = (
                f"{document_id}"
                f"_chunk_{chunk_number:03d}"
            )

            chunks.append({
                "chunk_id": chunk_id,

                "document_id": document_id,

                "text": chunk_text,

                "metadata": {
                    **document_metadata,

                    "chunk_number": chunk_number,

                    "start_token": start_token,

                    "end_token": end_token,

                    "token_count": len(
                        chunk_token_ids
                    ),

                    "chunk_size": CHUNK_SIZE,

                    "chunk_overlap": CHUNK_OVERLAP,

                    "tokenizer": TOKENIZER_NAME
                }
            })

            # ------------------------------------------------
            # CRITICAL FIX
            #
            # If this chunk reached the end of the document,
            # stop immediately.
            #
            # This prevents tiny trailing fragments such as
            # 3 tokens from being created.
            # ------------------------------------------------

            if end_token >= total_tokens:

                break

            # Move forward while preserving overlap.
            start_token += STEP_SIZE

    # --------------------------------------------------------
    # STEP 5 — VALIDATE GENERATED CHUNKS
    # --------------------------------------------------------

    if not chunks:

        raise ValueError(
            "No chunks were generated."
        )

    chunk_ids = [
        chunk["chunk_id"]
        for chunk in chunks
    ]

    if len(chunk_ids) != len(
        set(chunk_ids)
    ):

        raise ValueError(
            "Duplicate chunk IDs detected."
        )

    source_document_ids = {
        doc["document_id"]
        for doc in documents
    }

    chunk_document_ids = {
        chunk["document_id"]
        for chunk in chunks
    }

    if not chunk_document_ids.issubset(
        source_document_ids
    ):

        raise ValueError(
            "Chunks reference unknown documents."
        )

    for chunk in chunks:

        token_count = chunk[
            "metadata"
        ]["token_count"]

        if token_count <= 0:

            raise ValueError(
                f"{chunk['chunk_id']}: "
                f"invalid token count."
            )

        if token_count > CHUNK_SIZE:

            raise ValueError(
                f"{chunk['chunk_id']}: "
                f"exceeds chunk size."
            )

        if not chunk["text"].strip():

            raise ValueError(
                f"{chunk['chunk_id']}: "
                f"empty text."
            )

    # --------------------------------------------------------
    # STEP 6 — CHECK FOR UNNECESSARY TINY FINAL CHUNKS
    # --------------------------------------------------------

    tiny_chunks = [
        chunk
        for chunk in chunks
        if chunk["metadata"]["token_count"] < 50
    ]

    # A tiny chunk is acceptable if the entire document itself
    # is tiny. Our documents are much larger, so a tiny chunk
    # indicates a generation problem.

    if tiny_chunks:

        raise ValueError(
            f"Found {len(tiny_chunks)} "
            f"unexpected tiny chunks (<50 tokens)."
        )

    print(
        "[SUCCESS] No unnecessary tiny trailing "
        "chunks detected."
    )

    # --------------------------------------------------------
    # STEP 7 — DOCUMENT CHUNK DISTRIBUTION
    # --------------------------------------------------------

    chunks_per_document = {}

    for chunk in chunks:

        doc_id = chunk["document_id"]

        chunks_per_document[doc_id] = (
            chunks_per_document.get(
                doc_id,
                0
            ) + 1
        )

    one_chunk_documents = sum(
        1
        for count in chunks_per_document.values()
        if count == 1
    )

    two_chunk_documents = sum(
        1
        for count in chunks_per_document.values()
        if count == 2
    )

    three_plus_documents = sum(
        1
        for count in chunks_per_document.values()
        if count >= 3
    )

    # --------------------------------------------------------
    # STEP 8 — TOKEN STATISTICS
    # --------------------------------------------------------

    chunk_token_counts = [
        chunk["metadata"]["token_count"]
        for chunk in chunks
    ]

    # --------------------------------------------------------
    # STEP 9 — ATOMICALLY SAVE CHUNKS
    # --------------------------------------------------------

    atomic_json_save(
        chunks,
        CHUNKS_PATH
    )

    print()
    print(
        "[SUCCESS] Corrected chunk file saved atomically."
    )

    print(
        f"[INFO] Path: {CHUNKS_PATH}"
    )

    # --------------------------------------------------------
    # STEP 10 — SAVE CHECKPOINT
    # --------------------------------------------------------

    checkpoint = {

        "stage":
            "token_aware_chunk_generation",

        "status":
            "success",

        "generation_version":
            "corrected_final_chunk_handling",

        "source_documents":
            len(documents),

        "generated_chunks":
            len(chunks),

        "tokenizer":
            TOKENIZER_NAME,

        "chunk_size_tokens":
            CHUNK_SIZE,

        "chunk_overlap_tokens":
            CHUNK_OVERLAP,

        "step_size_tokens":
            STEP_SIZE,

        "chunk_token_statistics": {

            "minimum":
                min(chunk_token_counts),

            "maximum":
                max(chunk_token_counts),

            "mean":
                mean(chunk_token_counts),

            "median":
                median(chunk_token_counts)
        },

        "document_distribution": {

            "one_chunk_documents":
                one_chunk_documents,

            "two_chunk_documents":
                two_chunk_documents,

            "three_plus_chunk_documents":
                three_plus_documents,

            "total_documents":
                len(documents)
        },

        "tiny_chunks":
            len(tiny_chunks),

        "source_documents_path":
            str(DOCUMENTS_PATH),

        "chunks_path":
            str(CHUNKS_PATH)
    }

    atomic_json_save(
        checkpoint,
        CHUNK_CHECKPOINT_PATH
    )

    # --------------------------------------------------------
    # STEP 11 — DISPLAY RESULTS
    # --------------------------------------------------------

    print()
    print("=" * 60)
    print("CORRECTED CHUNK GENERATION RESULTS")
    print("=" * 60)

    print(
        f"Source documents       : {len(documents)}"
    )

    print(
        f"Generated chunks       : {len(chunks)}"
    )

    print(
        f"Tokenizer              : {TOKENIZER_NAME}"
    )

    print(
        f"Chunk size             : "
        f"{CHUNK_SIZE} tokens"
    )

    print(
        f"Chunk overlap          : "
        f"{CHUNK_OVERLAP} tokens"
    )

    print(
        f"Step size              : "
        f"{STEP_SIZE} tokens"
    )

    print()
    print(
        f"Minimum chunk tokens   : "
        f"{min(chunk_token_counts)}"
    )

    print(
        f"Maximum chunk tokens   : "
        f"{max(chunk_token_counts)}"
    )

    print(
        f"Average chunk tokens   : "
        f"{mean(chunk_token_counts):.2f}"
    )

    print(
        f"Median chunk tokens    : "
        f"{median(chunk_token_counts):.0f}"
    )

    print()
    print(
        f"1-chunk documents      : "
        f"{one_chunk_documents}"
    )

    print(
        f"2-chunk documents      : "
        f"{two_chunk_documents}"
    )

    print(
        f"3+ chunk documents     : "
        f"{three_plus_documents}"
    )

    print()
    print("-" * 60)
    print("FIRST 10 CORRECTED CHUNKS")
    print("-" * 60)

    for chunk in chunks[:10]:

        preview = (
            chunk["text"][:100]
            .replace("\n", " ")
        )

        print(
            f"{chunk['chunk_id']} | "
            f"{chunk['metadata']['token_count']} "
            f"tokens | {preview}..."
        )

    print()
    print(
        "[SUCCESS] Chunk generation checkpoint saved."
    )

    print(
        f"[INFO] Checkpoint: "
        f"{CHUNK_CHECKPOINT_PATH}"
    )

    # --------------------------------------------------------
    # CLEANUP
    # --------------------------------------------------------

    del documents
    del chunks
    del tokenizer
    del token_ids
    del chunk_token_counts
    del chunks_per_document
    del tiny_chunks

    gc.collect()

    print()
    print(
        "[INFO] Temporary chunk-generation "
        "objects released."
    )

except Exception as e:

    print()
    print("=" * 60)
    print("[ERROR] CORRECTED CHUNK GENERATION FAILED")
    print("=" * 60)

    print(
        f"Error type    : {type(e).__name__}"
    )

    print(
        f"Error message : {e}"
    )

    print()
    print(
        "[INFO] Source documents were not modified."
    )

    print(
        "[INFO] Existing chunks were not accepted "
        "as a successful generation."
    )

    raise

CORRECTED TOKEN-AWARE CHUNK GENERATION
[SUCCESS] Loaded 100 persisted RAG documents.
[SUCCESS] Tokenizer loaded: cl100k_base

------------------------------------------------------------
CHUNKING CONFIGURATION
------------------------------------------------------------
Tokenizer       : cl100k_base
Chunk size      : 500 tokens
Chunk overlap   : 50 tokens
Step size       : 450 tokens
[SUCCESS] No unnecessary tiny trailing chunks detected.

[SUCCESS] Corrected chunk file saved atomically.
[INFO] Path: E:\rag\data\processed\rag_chunks.json

CORRECTED CHUNK GENERATION RESULTS
Source documents       : 100
Generated chunks       : 164
Tokenizer              : cl100k_base
Chunk size             : 500 tokens
Chunk overlap          : 50 tokens
Step size              : 450 tokens

Minimum chunk tokens   : 53
Maximum chunk tokens   : 500
Average chunk tokens   : 347.00
Median chunk tokens    : 474

1-chunk documents      : 36
2-chunk documents      : 64
3+ chunk documents     : 0

--------------

In [8]:
# ============================================================
# CELL 8 — CHUNK QUALITY & INTEGRITY ANALYSIS
# ============================================================

import os
import json
import gc
import tempfile
from pathlib import Path
from collections import Counter, defaultdict
from statistics import mean, median

print("=" * 60)
print("CHUNK QUALITY & INTEGRITY ANALYSIS")
print("=" * 60)

# ------------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------------

PROJECT_ROOT = Path(r"E:\rag")

CHUNKS_PATH = PROJECT_ROOT / "data" / "processed" / "rag_chunks.json"
ANALYSIS_PATH = PROJECT_ROOT / "checkpoints" / "chunk_quality_analysis.json"

EXPECTED_DOCUMENTS = 100
EXPECTED_CHUNKS = 189

CHUNK_SIZE = 500
CHUNK_OVERLAP = 50

# ------------------------------------------------------------
# HELPER — ATOMIC JSON SAVE
# ------------------------------------------------------------

def atomic_json_save(data, output_path):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    temp_path = None

    try:
        with tempfile.NamedTemporaryFile(
            mode="w",
            encoding="utf-8",
            dir=output_path.parent,
            delete=False,
            suffix=".tmp"
        ) as tmp_file:

            json.dump(
                data,
                tmp_file,
                ensure_ascii=False,
                indent=2
            )

            tmp_file.flush()
            os.fsync(tmp_file.fileno())

            temp_path = Path(tmp_file.name)

        os.replace(temp_path, output_path)

    except Exception:
        if temp_path is not None and temp_path.exists():
            try:
                temp_path.unlink()
            except Exception:
                pass
        raise


# ------------------------------------------------------------
# STEP 1 — LOAD CHUNKS
# ------------------------------------------------------------

try:

    if not CHUNKS_PATH.exists():
        raise FileNotFoundError(
            f"Chunk file not found: {CHUNKS_PATH}"
        )

    with open(CHUNKS_PATH, "r", encoding="utf-8") as f:
        chunks = json.load(f)

    if not isinstance(chunks, list):
        raise ValueError("Chunk file must contain a list.")

    print(f"[SUCCESS] Loaded {len(chunks)} persisted chunks.")

    # --------------------------------------------------------
    # STEP 2 — BASIC STRUCTURAL VALIDATION
    # --------------------------------------------------------

    required_keys = {
        "chunk_id",
        "document_id",
        "text",
        "metadata"
    }

    chunk_ids = []
    document_ids = []

    for chunk in chunks:

        if not isinstance(chunk, dict):
            raise ValueError("Invalid chunk object detected.")

        if not required_keys.issubset(chunk.keys()):
            raise ValueError(
                f"Missing required keys in chunk: "
                f"{chunk.get('chunk_id', 'UNKNOWN')}"
            )

        if not isinstance(chunk["text"], str):
            raise ValueError(
                f"Non-string text in {chunk['chunk_id']}"
            )

        if not chunk["text"].strip():
            raise ValueError(
                f"Empty text in {chunk['chunk_id']}"
            )

        if not chunk["document_id"]:
            raise ValueError(
                f"Missing document_id in {chunk['chunk_id']}"
            )

        chunk_ids.append(chunk["chunk_id"])
        document_ids.append(chunk["document_id"])

    # Unique chunk IDs
    if len(chunk_ids) != len(set(chunk_ids)):
        raise ValueError("Duplicate chunk IDs detected.")

    print("[SUCCESS] Structural validation passed.")
    print("[SUCCESS] Chunk IDs are unique.")

    # --------------------------------------------------------
    # STEP 3 — TOKEN STATISTICS
    # --------------------------------------------------------

    token_counts = []
    character_counts = []

    for chunk in chunks:

        metadata = chunk["metadata"]

        token_count = metadata.get("token_count")

        if not isinstance(token_count, int):
            raise ValueError(
                f"Invalid token_count in {chunk['chunk_id']}"
            )

        if token_count <= 0:
            raise ValueError(
                f"Non-positive token_count in {chunk['chunk_id']}"
            )

        if token_count > CHUNK_SIZE:
            raise ValueError(
                f"{chunk['chunk_id']} exceeds "
                f"{CHUNK_SIZE} token limit."
            )

        token_counts.append(token_count)
        character_counts.append(len(chunk["text"]))

    # --------------------------------------------------------
    # STEP 4 — DOCUMENT → CHUNK DISTRIBUTION
    # --------------------------------------------------------

    chunks_per_document = Counter(document_ids)

    distribution = Counter(
        chunks_per_document.values()
    )

    one_chunk_documents = distribution.get(1, 0)
    two_chunk_documents = distribution.get(2, 0)
    three_plus_documents = sum(
        count
        for chunk_count, count in distribution.items()
        if chunk_count >= 3
    )

    # --------------------------------------------------------
    # STEP 5 — IDENTIFY VERY SMALL CHUNKS
    # --------------------------------------------------------

    tiny_threshold = 50
    small_threshold = 100

    tiny_chunks = [
        chunk
        for chunk in chunks
        if chunk["metadata"]["token_count"] < tiny_threshold
    ]

    small_chunks = [
        chunk
        for chunk in chunks
        if chunk["metadata"]["token_count"] < small_threshold
    ]

    # --------------------------------------------------------
    # STEP 6 — IDENTIFY THE SHORTEST CHUNKS
    # --------------------------------------------------------

    sorted_chunks = sorted(
        chunks,
        key=lambda x: x["metadata"]["token_count"]
    )

    shortest_chunks = []

    for chunk in sorted_chunks[:10]:

        shortest_chunks.append({
            "chunk_id": chunk["chunk_id"],
            "document_id": chunk["document_id"],
            "chunk_number": chunk["metadata"]["chunk_number"],
            "token_count": chunk["metadata"]["token_count"],
            "character_count": len(chunk["text"]),
            "text_preview": chunk["text"][:200]
                .replace("\n", " ")
        })

    # --------------------------------------------------------
    # STEP 7 — CHECK DUPLICATE CHUNK TEXT
    # --------------------------------------------------------

    text_counter = Counter(
        chunk["text"]
        for chunk in chunks
    )

    duplicate_text_groups = [
        {
            "text_preview": text[:200].replace("\n", " "),
            "occurrences": count
        }
        for text, count in text_counter.items()
        if count > 1
    ]

    # --------------------------------------------------------
    # STEP 8 — CHECK CHUNK NUMBERING
    # --------------------------------------------------------

    numbering_issues = []

    chunks_by_document = defaultdict(list)

    for chunk in chunks:
        chunks_by_document[
            chunk["document_id"]
        ].append(chunk)

    for document_id, document_chunks in chunks_by_document.items():

        document_chunks.sort(
            key=lambda x: x["metadata"]["chunk_number"]
        )

        expected_numbers = list(
            range(1, len(document_chunks) + 1)
        )

        actual_numbers = [
            chunk["metadata"]["chunk_number"]
            for chunk in document_chunks
        ]

        if actual_numbers != expected_numbers:
            numbering_issues.append({
                "document_id": document_id,
                "expected": expected_numbers,
                "actual": actual_numbers
            })

    if numbering_issues:
        raise ValueError(
            f"Chunk numbering issues found in "
            f"{len(numbering_issues)} documents."
        )

    print("[SUCCESS] Chunk numbering is consistent.")

    # --------------------------------------------------------
    # STEP 9 — CHECK TOKEN BOUNDARIES
    # --------------------------------------------------------

    boundary_issues = []

    for chunk in chunks:

        metadata = chunk["metadata"]

        start_token = metadata.get("start_token")
        end_token = metadata.get("end_token")
        token_count = metadata.get("token_count")

        if start_token is None or end_token is None:
            boundary_issues.append({
                "chunk_id": chunk["chunk_id"],
                "issue": "Missing token boundary"
            })
            continue

        if end_token <= start_token:
            boundary_issues.append({
                "chunk_id": chunk["chunk_id"],
                "issue": "Invalid token boundary"
            })
            continue

        if (end_token - start_token) != token_count:
            boundary_issues.append({
                "chunk_id": chunk["chunk_id"],
                "issue": "Boundary/token count mismatch"
            })

    if boundary_issues:
        raise ValueError(
            f"Found {len(boundary_issues)} token boundary issues."
        )

    print("[SUCCESS] Token boundaries are consistent.")

    # --------------------------------------------------------
    # STEP 10 — CHECK SLIDING-WINDOW OVERLAP
    # --------------------------------------------------------

    overlap_issues = []

    for document_id, document_chunks in chunks_by_document.items():

        document_chunks.sort(
            key=lambda x: x["metadata"]["chunk_number"]
        )

        for i in range(len(document_chunks) - 1):

            current = document_chunks[i]
            following = document_chunks[i + 1]

            current_end = current["metadata"]["end_token"]
            next_start = following["metadata"]["start_token"]

            actual_gap = next_start - current_end

            # For non-final chunks, expected movement is:
            # next_start = current_start + 450
            #
            # Therefore:
            # next_start - current_end = -50
            #
            # Negative 50 means 50-token overlap.

            if actual_gap != -CHUNK_OVERLAP:
                overlap_issues.append({
                    "document_id": document_id,
                    "current_chunk": current["chunk_id"],
                    "next_chunk": following["chunk_id"],
                    "expected_gap": -CHUNK_OVERLAP,
                    "actual_gap": actual_gap
                })

    if overlap_issues:
        raise ValueError(
            f"Found {len(overlap_issues)} overlap issues."
        )

    print("[SUCCESS] Token overlap is consistent.")

    # --------------------------------------------------------
    # STEP 11 — CHECK DOCUMENT COVERAGE
    # --------------------------------------------------------

    unique_documents = set(document_ids)

    if len(unique_documents) != EXPECTED_DOCUMENTS:
        raise ValueError(
            f"Expected {EXPECTED_DOCUMENTS} unique documents, "
            f"found {len(unique_documents)}."
        )

    print(
        f"[SUCCESS] All {EXPECTED_DOCUMENTS} source documents "
        f"are represented."
    )

    # --------------------------------------------------------
    # STEP 12 — SUMMARY STATISTICS
    # --------------------------------------------------------

    print()
    print("-" * 60)
    print("CHUNK QUALITY RESULTS")
    print("-" * 60)

    print(f"Documents represented     : {len(unique_documents)}")
    print(f"Total chunks              : {len(chunks)}")

    print()
    print("TOKEN DISTRIBUTION")
    print(f"Minimum                   : {min(token_counts)}")
    print(f"Maximum                   : {max(token_counts)}")
    print(f"Mean                      : {mean(token_counts):.2f}")
    print(f"Median                    : {median(token_counts):.0f}")

    print()
    print("CHARACTER DISTRIBUTION")
    print(f"Minimum                   : {min(character_counts)}")
    print(f"Maximum                   : {max(character_counts)}")
    print(f"Mean                      : {mean(character_counts):.2f}")
    print(f"Median                    : {median(character_counts):.0f}")

    print()
    print("DOCUMENT → CHUNK DISTRIBUTION")
    print(f"1 chunk                   : {one_chunk_documents}")
    print(f"2 chunks                  : {two_chunk_documents}")
    print(f"3+ chunks                 : {three_plus_documents}")

    print()
    print("SMALL CHUNK ANALYSIS")
    print(
        f"< {tiny_threshold} tokens          : "
        f"{len(tiny_chunks)}"
    )
    print(
        f"< {small_threshold} tokens         : "
        f"{len(small_chunks)}"
    )

    print()
    print("DUPLICATE TEXT")
    print(
        f"Duplicate text groups     : "
        f"{len(duplicate_text_groups)}"
    )

    # --------------------------------------------------------
    # STEP 13 — DISPLAY SHORTEST CHUNKS
    # --------------------------------------------------------

    print()
    print("-" * 60)
    print("SHORTEST 10 CHUNKS")
    print("-" * 60)

    for item in shortest_chunks:

        print(
            f"{item['chunk_id']} | "
            f"{item['token_count']} tokens | "
            f"{item['character_count']} chars | "
            f"{item['text_preview']}..."
        )

    # --------------------------------------------------------
    # STEP 14 — CREATE PERSISTED ANALYSIS
    # --------------------------------------------------------

    analysis = {
        "stage": "chunk_quality_analysis",
        "status": "success",

        "configuration": {
            "chunk_size_tokens": CHUNK_SIZE,
            "chunk_overlap_tokens": CHUNK_OVERLAP,
            "expected_documents": EXPECTED_DOCUMENTS,
            "expected_chunks": EXPECTED_CHUNKS
        },

        "corpus": {
            "documents_represented": len(unique_documents),
            "total_chunks": len(chunks)
        },

        "token_statistics": {
            "minimum": min(token_counts),
            "maximum": max(token_counts),
            "mean": mean(token_counts),
            "median": median(token_counts)
        },

        "character_statistics": {
            "minimum": min(character_counts),
            "maximum": max(character_counts),
            "mean": mean(character_counts),
            "median": median(character_counts)
        },

        "document_chunk_distribution": {
            "one_chunk_documents": one_chunk_documents,
            "two_chunk_documents": two_chunk_documents,
            "three_plus_chunk_documents": three_plus_documents
        },

        "small_chunk_analysis": {
            "tiny_threshold_tokens": tiny_threshold,
            "small_threshold_tokens": small_threshold,
            "tiny_chunks_count": len(tiny_chunks),
            "small_chunks_count": len(small_chunks)
        },

        "duplicate_text_groups": len(
            duplicate_text_groups
        ),

        "integrity_checks": {
            "unique_chunk_ids": True,
            "all_documents_represented": True,
            "token_boundaries_valid": True,
            "token_counts_within_limit": True,
            "overlap_consistent": True,
            "chunk_numbering_consistent": True
        },

        "shortest_chunks": shortest_chunks,

        "paths": {
            "source_chunks": str(CHUNKS_PATH),
            "analysis_checkpoint": str(ANALYSIS_PATH)
        }
    }

    # --------------------------------------------------------
    # STEP 15 — SAVE ANALYSIS
    # --------------------------------------------------------

    atomic_json_save(
        analysis,
        ANALYSIS_PATH
    )

    print()
    print("[SUCCESS] Chunk quality analysis checkpoint saved.")
    print(f"[INFO] Path: {ANALYSIS_PATH}")

    print()
    print("=" * 60)
    print("CHUNK QUALITY ANALYSIS COMPLETE")
    print("=" * 60)
    print("[SUCCESS] No chunks were modified.")
    print("[SUCCESS] No source documents were modified.")
    print("[SUCCESS] Integrity checks passed.")

    # --------------------------------------------------------
    # CLEANUP
    # --------------------------------------------------------

    del chunks
    del token_counts
    del character_counts
    del chunks_per_document
    del distribution
    del tiny_chunks
    del small_chunks
    del sorted_chunks
    del shortest_chunks
    del text_counter
    del duplicate_text_groups
    del chunks_by_document
    del analysis

    gc.collect()

    print("[INFO] Temporary analysis objects released.")

except Exception as e:

    print()
    print("=" * 60)
    print("[ERROR] CHUNK QUALITY ANALYSIS FAILED")
    print("=" * 60)
    print(f"Error type    : {type(e).__name__}")
    print(f"Error message : {e}")
    print()
    print("[INFO] Chunk data was not modified.")
    print("[INFO] No invalid analysis checkpoint was accepted.")
    raise

CHUNK QUALITY & INTEGRITY ANALYSIS
[SUCCESS] Loaded 164 persisted chunks.
[SUCCESS] Structural validation passed.
[SUCCESS] Chunk IDs are unique.
[SUCCESS] Chunk numbering is consistent.
[SUCCESS] Token boundaries are consistent.
[SUCCESS] Token overlap is consistent.
[SUCCESS] All 100 source documents are represented.

------------------------------------------------------------
CHUNK QUALITY RESULTS
------------------------------------------------------------
Documents represented     : 100
Total chunks              : 164

TOKEN DISTRIBUTION
Minimum                   : 53
Maximum                   : 500
Mean                      : 347.00
Median                    : 474

CHARACTER DISTRIBUTION
Minimum                   : 292
Maximum                   : 2701
Mean                      : 1670.66
Median                    : 2196

DOCUMENT → CHUNK DISTRIBUTION
1 chunk                   : 36
2 chunks                  : 64
3+ chunks                 : 0

SMALL CHUNK ANALYSIS
< 50 tokens      

In [9]:
# ============================================================
# CELL 9 — EMBEDDING ENVIRONMENT & MODEL DISCOVERY
# ============================================================

import os
import json
import gc
import platform
import subprocess
import importlib.util
from pathlib import Path

print("=" * 60)
print("EMBEDDING ENVIRONMENT & MODEL DISCOVERY")
print("=" * 60)

# ------------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------------

PROJECT_ROOT = Path(r"E:\rag")

CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
ENVIRONMENT_PATH = (
    CHECKPOINT_DIR / "embedding_environment.json"
)

OLLAMA_URL = "http://localhost:11434"

# ------------------------------------------------------------
# HELPER — ATOMIC JSON SAVE
# ------------------------------------------------------------

def atomic_json_save(data, output_path):

    import tempfile

    output_path = Path(output_path)
    output_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    temp_path = None

    try:

        with tempfile.NamedTemporaryFile(
            mode="w",
            encoding="utf-8",
            dir=output_path.parent,
            delete=False,
            suffix=".tmp"
        ) as tmp_file:

            json.dump(
                data,
                tmp_file,
                ensure_ascii=False,
                indent=2
            )

            tmp_file.flush()
            os.fsync(tmp_file.fileno())

            temp_path = Path(tmp_file.name)

        os.replace(
            temp_path,
            output_path
        )

    except Exception:

        if temp_path is not None and temp_path.exists():

            try:
                temp_path.unlink()
            except Exception:
                pass

        raise


# ------------------------------------------------------------
# RESULT CONTAINERS
# ------------------------------------------------------------

environment = {
    "stage": "embedding_environment_discovery",
    "status": "success",
    "python": {},
    "gpu": {},
    "packages": {},
    "ollama": {},
    "candidate_routes": []
}


try:

    # ========================================================
    # STEP 1 — PYTHON ENVIRONMENT
    # ========================================================

    print()
    print("-" * 60)
    print("PYTHON ENVIRONMENT")
    print("-" * 60)

    environment["python"] = {
        "version": platform.python_version(),
        "implementation": platform.python_implementation(),
        "platform": platform.platform()
    }

    print(
        f"Python version       : "
        f"{platform.python_version()}"
    )

    print(
        f"Platform             : "
        f"{platform.platform()}"
    )

    # ========================================================
    # STEP 2 — CHECK IMPORTANT PACKAGES
    # ========================================================

    print()
    print("-" * 60)
    print("EMBEDDING-RELATED PACKAGES")
    print("-" * 60)

    packages_to_check = [
        "torch",
        "sentence_transformers",
        "transformers",
        "numpy",
        "requests"
    ]

    for package_name in packages_to_check:

        installed = (
            importlib.util.find_spec(package_name)
            is not None
        )

        version = None

        if installed:

            try:

                module = __import__(
                    package_name
                )

                version = getattr(
                    module,
                    "__version__",
                    "unknown"
                )

            except Exception:

                version = "installed"

        environment["packages"][
            package_name
        ] = {
            "installed": installed,
            "version": version
        }

        status = "INSTALLED" if installed else "NOT INSTALLED"

        print(
            f"{package_name:<22}: "
            f"{status}"
            + (
                f" ({version})"
                if installed
                else ""
            )
        )

    # ========================================================
    # STEP 3 — PYTORCH / CUDA
    # ========================================================

    print()
    print("-" * 60)
    print("GPU / CUDA ENVIRONMENT")
    print("-" * 60)

    torch_available = (
        importlib.util.find_spec("torch")
        is not None
    )

    if torch_available:

        import torch

        cuda_available = torch.cuda.is_available()

        environment["gpu"]["torch_cuda_available"] = (
            cuda_available
        )

        environment["gpu"]["torch_version"] = (
            torch.__version__
        )

        print(
            f"PyTorch version      : "
            f"{torch.__version__}"
        )

        print(
            f"CUDA available       : "
            f"{cuda_available}"
        )

        if cuda_available:

            gpu_count = torch.cuda.device_count()

            environment["gpu"]["device_count"] = (
                gpu_count
            )

            print(
                f"CUDA devices         : "
                f"{gpu_count}"
            )

            for i in range(gpu_count):

                gpu_name = torch.cuda.get_device_name(i)

                total_memory = (
                    torch.cuda.get_device_properties(i)
                    .total_memory
                )

                total_memory_gb = (
                    total_memory
                    / (1024 ** 3)
                )

                gpu_info = {
                    "index": i,
                    "name": gpu_name,
                    "total_memory_gb": round(
                        total_memory_gb,
                        2
                    )
                }

                environment["gpu"].setdefault(
                    "devices",
                    []
                ).append(gpu_info)

                print()
                print(
                    f"GPU {i}               : "
                    f"{gpu_name}"
                )

                print(
                    f"VRAM                  : "
                    f"{total_memory_gb:.2f} GB"
                )

        else:

            environment["gpu"]["device_count"] = 0

            print(
                "[WARNING] CUDA is not available."
            )

    else:

        environment["gpu"][
            "torch_cuda_available"
        ] = False

        environment["gpu"][
            "device_count"
        ] = 0

        print(
            "[WARNING] PyTorch is not installed."
        )

    # ========================================================
    # STEP 4 — NVIDIA-SMI CHECK
    # ========================================================

    print()
    print("-" * 60)
    print("NVIDIA-SMI")
    print("-" * 60)

    nvidia_smi_available = False
    nvidia_smi_output = None

    try:

        result = subprocess.run(
            ["nvidia-smi"],
            capture_output=True,
            text=True,
            timeout=10
        )

        if result.returncode == 0:

            nvidia_smi_available = True
            nvidia_smi_output = (
                result.stdout.strip()
            )

            print(
                "[SUCCESS] nvidia-smi is available."
            )

            first_lines = (
                nvidia_smi_output
                .splitlines()[:10]
            )

            for line in first_lines:
                print(line)

        else:

            print(
                "[WARNING] nvidia-smi returned "
                f"code {result.returncode}."
            )

    except FileNotFoundError:

        print(
            "[WARNING] nvidia-smi was not found."
        )

    except Exception as e:

        print(
            f"[WARNING] nvidia-smi check failed: {e}"
        )

    environment["gpu"][
        "nvidia_smi_available"
    ] = nvidia_smi_available

    # ========================================================
    # STEP 5 — OLLAMA CONNECTION
    # ========================================================

    print()
    print("-" * 60)
    print("OLLAMA EMBEDDING ENVIRONMENT")
    print("-" * 60)

    ollama_available = False
    ollama_models = []
    ollama_error = None

    try:

        import requests

        response = requests.get(
            f"{OLLAMA_URL}/api/tags",
            timeout=10
        )

        response.raise_for_status()

        ollama_data = response.json()

        ollama_available = True

        for model in ollama_data.get(
            "models",
            []
        ):

            model_name = model.get(
                "name"
            )

            if model_name:

                ollama_models.append({
                    "name": model_name,
                    "size_bytes": model.get(
                        "size"
                    ),
                    "parameter_size": (
                        model.get(
                            "details",
                            {}
                        ).get(
                            "parameter_size"
                        )
                    ),
                    "quantization": (
                        model.get(
                            "details",
                            {}
                        ).get(
                            "quantization_level"
                        )
                    )
                })

        print(
            "[SUCCESS] Ollama server is reachable."
        )

        print(
            f"Installed models      : "
            f"{len(ollama_models)}"
        )

        for model in ollama_models:

            print(
                f"  - {model['name']}"
            )

    except Exception as e:

        ollama_error = str(e)

        print(
            "[WARNING] Ollama embedding environment "
            f"check failed: {e}"
        )

    environment["ollama"] = {
        "url": OLLAMA_URL,
        "available": ollama_available,
        "models": ollama_models,
        "error": ollama_error
    }

    # ========================================================
    # STEP 6 — IDENTIFY EMBEDDING ROUTES
    # ========================================================

    print()
    print("-" * 60)
    print("CANDIDATE EMBEDDING ROUTES")
    print("-" * 60)

    routes = []

    # Sentence Transformers
    if (
        environment["packages"]
        ["sentence_transformers"]
        ["installed"]
    ):

        routes.append({
            "route": "sentence_transformers",
            "available": True,
            "description":
                "Local Hugging Face/Sentence-Transformers "
                "embedding model with optional CUDA."
        })

        print(
            "[AVAILABLE] Sentence Transformers"
        )

    else:

        print(
            "[UNAVAILABLE] Sentence Transformers"
        )

    # Transformers
    if (
        environment["packages"]
        ["transformers"]
        ["installed"]
    ):

        routes.append({
            "route": "transformers",
            "available": True,
            "description":
                "Direct Transformer-based embedding "
                "implementation."
        })

        print(
            "[AVAILABLE] Transformers"
        )

    else:

        print(
            "[UNAVAILABLE] Transformers"
        )

    # Ollama
    if ollama_available:

        routes.append({
            "route": "ollama",
            "available": True,
            "description":
                "Ollama-based local embedding API."
        })

        print(
            "[AVAILABLE] Ollama"
        )

    else:

        print(
            "[UNAVAILABLE] Ollama"
        )

    environment["candidate_routes"] = routes

    # ========================================================
    # STEP 7 — RESEARCH BASELINE RECOMMENDATION
    # ========================================================

    print()
    print("-" * 60)
    print("EMBEDDING STAGE STATUS")
    print("-" * 60)

    print(
        "[INFO] No embeddings have been generated."
    )

    print(
        "[INFO] No vector index has been created."
    )

    print(
        "[INFO] Embedding model selection will be "
        "performed in the next stage."
    )

    # ========================================================
    # STEP 8 — SAVE ENVIRONMENT CHECKPOINT
    # ========================================================

    atomic_json_save(
        environment,
        ENVIRONMENT_PATH
    )

    print()
    print(
        "[SUCCESS] Embedding environment checkpoint saved."
    )

    print(
        f"[INFO] Path: {ENVIRONMENT_PATH}"
    )

    print()
    print("=" * 60)
    print("EMBEDDING ENVIRONMENT DISCOVERY COMPLETE")
    print("=" * 60)

    # --------------------------------------------------------
    # CLEANUP
    # --------------------------------------------------------

    gc.collect()

    print(
        "[INFO] Temporary environment objects released."
    )

except Exception as e:

    print()
    print("=" * 60)
    print("[ERROR] EMBEDDING ENVIRONMENT DISCOVERY FAILED")
    print("=" * 60)

    print(
        f"Error type    : {type(e).__name__}"
    )

    print(
        f"Error message : {e}"
    )

    print()
    print(
        "[INFO] No dataset or chunk files were modified."
    )

    raise

EMBEDDING ENVIRONMENT & MODEL DISCOVERY

------------------------------------------------------------
PYTHON ENVIRONMENT
------------------------------------------------------------
Python version       : 3.11.15
Platform             : Windows-10-10.0.26200-SP0

------------------------------------------------------------
EMBEDDING-RELATED PACKAGES
------------------------------------------------------------
torch                 : INSTALLED (2.5.1)


e:\anaconda3\envs\imgds\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


sentence_transformers : INSTALLED (6.0.1)
transformers          : INSTALLED (5.16.1)
numpy                 : INSTALLED (2.0.1)
requests              : INSTALLED (2.34.2)

------------------------------------------------------------
GPU / CUDA ENVIRONMENT
------------------------------------------------------------
PyTorch version      : 2.5.1
CUDA available       : True
CUDA devices         : 1

GPU 0               : NVIDIA GeForce RTX 4060 Ti
VRAM                  : 16.00 GB

------------------------------------------------------------
NVIDIA-SMI
------------------------------------------------------------
[SUCCESS] nvidia-smi is available.
Sat Sep  5 23:42:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 616.56                 KMD Version: 616.56        CUDA UMD Version: 13.4     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model 

In [10]:
# ============================================================
# CELL 10 — INSTALL & VERIFY SENTENCE TRANSFORMERS
# ============================================================

import os
import sys
import gc
import json
import subprocess
import importlib.util
from pathlib import Path

print("=" * 60)
print("SENTENCE TRANSFORMERS INSTALLATION & VERIFICATION")
print("=" * 60)

PROJECT_ROOT = Path(r"E:\rag")

CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

VERIFICATION_PATH = (
    CHECKPOINT_DIR /
    "embedding_framework_verification.json"
)

PACKAGE_NAME = "sentence-transformers"

# ------------------------------------------------------------
# HELPER — ATOMIC JSON SAVE
# ------------------------------------------------------------

def atomic_json_save(data, output_path):

    import tempfile

    output_path = Path(output_path)
    output_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    temp_path = None

    try:

        with tempfile.NamedTemporaryFile(
            mode="w",
            encoding="utf-8",
            dir=output_path.parent,
            delete=False,
            suffix=".tmp"
        ) as tmp_file:

            json.dump(
                data,
                tmp_file,
                ensure_ascii=False,
                indent=2
            )

            tmp_file.flush()
            os.fsync(tmp_file.fileno())

            temp_path = Path(
                tmp_file.name
            )

        os.replace(
            temp_path,
            output_path
        )

    except Exception:

        if (
            temp_path is not None
            and temp_path.exists()
        ):
            try:
                temp_path.unlink()
            except Exception:
                pass

        raise


try:

    # ========================================================
    # STEP 1 — CHECK PACKAGE
    # ========================================================

    print()
    print("-" * 60)
    print("PACKAGE STATUS")
    print("-" * 60)

    package_installed = (
        importlib.util.find_spec(
            "sentence_transformers"
        ) is not None
    )

    if package_installed:

        print(
            "[INFO] sentence-transformers "
            "is already installed."
        )

    else:

        print(
            "[INFO] sentence-transformers "
            "is not installed."
        )

        print(
            "[INFO] Installing required package..."
        )

        result = subprocess.run(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-U",
                PACKAGE_NAME
            ],
            capture_output=True,
            text=True
        )

        if result.returncode != 0:

            print(
                result.stdout
            )

            print(
                result.stderr
            )

            raise RuntimeError(
                "sentence-transformers installation failed."
            )

        print(
            "[SUCCESS] sentence-transformers "
            "installation completed."
        )

    # --------------------------------------------------------
    # IMPORTANT:
    # Refresh package discovery after installation.
    # --------------------------------------------------------

    importlib.invalidate_caches()

    if (
        importlib.util.find_spec(
            "sentence_transformers"
        ) is None
    ):

        raise ImportError(
            "sentence-transformers could not be "
            "imported after installation."
        )

    # ========================================================
    # STEP 2 — IMPORT FRAMEWORK
    # ========================================================

    from sentence_transformers import (
        SentenceTransformer
    )

    import sentence_transformers

    framework_version = getattr(
        sentence_transformers,
        "__version__",
        "unknown"
    )

    print(
        f"[SUCCESS] Sentence Transformers loaded."
    )

    print(
        f"Version              : "
        f"{framework_version}"
    )

    # ========================================================
    # STEP 3 — VERIFY PYTORCH
    # ========================================================

    print()
    print("-" * 60)
    print("PYTORCH / CUDA VERIFICATION")
    print("-" * 60)

    import torch

    cuda_available = torch.cuda.is_available()

    print(
        f"PyTorch version      : "
        f"{torch.__version__}"
    )

    print(
        f"CUDA available       : "
        f"{cuda_available}"
    )

    if not cuda_available:

        raise RuntimeError(
            "CUDA is not available. "
            "Embedding generation should not proceed "
            "until the GPU environment is verified."
        )

    device_count = torch.cuda.device_count()

    if device_count < 1:

        raise RuntimeError(
            "PyTorch reports zero CUDA devices."
        )

    gpu_name = torch.cuda.get_device_name(0)

    gpu_properties = (
        torch.cuda.get_device_properties(0)
    )

    total_vram_gb = (
        gpu_properties.total_memory
        / (1024 ** 3)
    )

    print(
        f"GPU                  : "
        f"{gpu_name}"
    )

    print(
        f"VRAM                 : "
        f"{total_vram_gb:.2f} GB"
    )

    # ========================================================
    # STEP 4 — FRAMEWORK VERIFICATION
    # ========================================================

    print()
    print("-" * 60)
    print("FRAMEWORK VERIFICATION")
    print("-" * 60)

    verification = {
        "stage":
            "embedding_framework_verification",

        "status":
            "success",

        "framework": {
            "name":
                "sentence-transformers",

            "version":
                framework_version
        },

        "pytorch": {
            "version":
                torch.__version__,

            "cuda_available":
                cuda_available,

            "device_count":
                device_count
        },

        "gpu": {
            "name":
                gpu_name,

            "vram_gb":
                round(
                    total_vram_gb,
                    2
                )
        },

        "next_stage": {
            "action":
                "select_and_download_embedding_model",

            "embeddings_generated":
                False,

            "vector_index_created":
                False
        }
    }

    atomic_json_save(
        verification,
        VERIFICATION_PATH
    )

    print(
        "[SUCCESS] Embedding framework verified."
    )

    print(
        f"[INFO] Verification checkpoint: "
        f"{VERIFICATION_PATH}"
    )

    print()
    print("=" * 60)
    print("EMBEDDING FRAMEWORK VERIFICATION COMPLETE")
    print("=" * 60)

    # --------------------------------------------------------
    # CLEANUP
    # --------------------------------------------------------

    gc.collect()

    print(
        "[INFO] Temporary objects released."
    )

except Exception as e:

    print()
    print("=" * 60)
    print("[ERROR] EMBEDDING FRAMEWORK SETUP FAILED")
    print("=" * 60)

    print(
        f"Error type    : {type(e).__name__}"
    )

    print(
        f"Error message : {e}"
    )

    print()
    print(
        "[INFO] No embeddings were generated."
    )

    print(
        "[INFO] No vector index was created."
    )

    raise

SENTENCE TRANSFORMERS INSTALLATION & VERIFICATION

------------------------------------------------------------
PACKAGE STATUS
------------------------------------------------------------
[INFO] sentence-transformers is already installed.
[SUCCESS] Sentence Transformers loaded.
Version              : 6.0.1

------------------------------------------------------------
PYTORCH / CUDA VERIFICATION
------------------------------------------------------------
PyTorch version      : 2.5.1
CUDA available       : True
GPU                  : NVIDIA GeForce RTX 4060 Ti
VRAM                 : 16.00 GB

------------------------------------------------------------
FRAMEWORK VERIFICATION
------------------------------------------------------------
[SUCCESS] Embedding framework verified.
[INFO] Verification checkpoint: E:\rag\checkpoints\embedding_framework_verification.json

EMBEDDING FRAMEWORK VERIFICATION COMPLETE
[INFO] Temporary objects released.


In [11]:
# ============================================================
# CELL 11 — EMBEDDING MODEL SELECTION & VERIFICATION
# ============================================================

import os
import json
import gc
import tempfile
from pathlib import Path

print("=" * 60)
print("EMBEDDING MODEL SELECTION & VERIFICATION")
print("=" * 60)

# ------------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------------

PROJECT_ROOT = Path(r"E:\rag")

CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
MODEL_CHECKPOINT_PATH = (
    CHECKPOINT_DIR / "embedding_model.json"
)

MODEL_NAME = "BAAI/bge-small-en-v1.5"
DEVICE = "cuda"

# ------------------------------------------------------------
# HELPER — ATOMIC JSON SAVE
# ------------------------------------------------------------

def atomic_json_save(data, output_path):

    output_path = Path(output_path)
    output_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    temp_path = None

    try:

        with tempfile.NamedTemporaryFile(
            mode="w",
            encoding="utf-8",
            dir=output_path.parent,
            delete=False,
            suffix=".tmp"
        ) as tmp_file:

            json.dump(
                data,
                tmp_file,
                ensure_ascii=False,
                indent=2
            )

            tmp_file.flush()
            os.fsync(tmp_file.fileno())

            temp_path = Path(
                tmp_file.name
            )

        os.replace(
            temp_path,
            output_path
        )

    except Exception:

        if (
            temp_path is not None
            and temp_path.exists()
        ):
            try:
                temp_path.unlink()
            except Exception:
                pass

        raise


try:

    # ========================================================
    # STEP 1 — IMPORT DEPENDENCIES
    # ========================================================

    import torch
    from sentence_transformers import SentenceTransformer

    print()
    print("-" * 60)
    print("DEPENDENCY VERIFICATION")
    print("-" * 60)

    print(
        f"PyTorch              : "
        f"{torch.__version__}"
    )

    print(
        f"CUDA available       : "
        f"{torch.cuda.is_available()}"
    )

    if not torch.cuda.is_available():

        raise RuntimeError(
            "CUDA is unavailable. "
            "Cannot use the configured GPU device."
        )

    # ========================================================
    # STEP 2 — DISPLAY MODEL CONFIGURATION
    # ========================================================

    print()
    print("-" * 60)
    print("MODEL CONFIGURATION")
    print("-" * 60)

    print(
        f"Model                : "
        f"{MODEL_NAME}"
    )

    print(
        f"Device               : "
        f"{DEVICE}"
    )

    # ========================================================
    # STEP 3 — LOAD MODEL
    # ========================================================

    print()
    print("-" * 60)
    print("MODEL LOADING")
    print("-" * 60)

    print(
        "[INFO] Loading embedding model..."
    )

    model = SentenceTransformer(
        MODEL_NAME,
        device=DEVICE
    )

    print(
        "[SUCCESS] Embedding model loaded."
    )

    # ========================================================
    # STEP 4 — MODEL INFORMATION
    # ========================================================

    embedding_dimension = (
        model.get_sentence_embedding_dimension()
    )

    max_sequence_length = (
        model.max_seq_length
    )

    print()
    print("-" * 60)
    print("MODEL INFORMATION")
    print("-" * 60)

    print(
        f"Embedding dimension  : "
        f"{embedding_dimension}"
    )

    print(
        f"Max sequence length  : "
        f"{max_sequence_length}"
    )

    if embedding_dimension is None:

        raise ValueError(
            "Could not determine embedding dimension."
        )

    if embedding_dimension <= 0:

        raise ValueError(
            "Invalid embedding dimension."
        )

    # ========================================================
    # STEP 5 — TEST EMBEDDING
    # ========================================================

    print()
    print("-" * 60)
    print("TEST EMBEDDING")
    print("-" * 60)

    test_text = (
        "A VPN provides secure remote access "
        "to company resources."
    )

    print(
        "[INFO] Generating test embedding..."
    )

    test_embedding = model.encode(
        [test_text],
        convert_to_tensor=True,
        normalize_embeddings=True,
        show_progress_bar=False
    )

    # --------------------------------------------------------
    # Validate tensor shape
    # --------------------------------------------------------

    if test_embedding.ndim != 2:

        raise ValueError(
            f"Unexpected embedding shape: "
            f"{tuple(test_embedding.shape)}"
        )

    if test_embedding.shape[0] != 1:

        raise ValueError(
            "Test embedding batch dimension "
            "is incorrect."
        )

    if test_embedding.shape[1] != embedding_dimension:

        raise ValueError(
            "Embedding dimension mismatch."
        )

    # --------------------------------------------------------
    # Validate finite values
    # --------------------------------------------------------

    if not torch.isfinite(
        test_embedding
    ).all():

        raise ValueError(
            "Embedding contains NaN or infinite values."
        )

    # --------------------------------------------------------
    # Validate normalization
    # --------------------------------------------------------

    embedding_norm = (
        torch.linalg.vector_norm(
            test_embedding[0]
        ).item()
    )

    print(
        "[SUCCESS] Test embedding generated."
    )

    print(
        f"Test shape           : "
        f"{tuple(test_embedding.shape)}"
    )

    print(
        f"Vector norm          : "
        f"{embedding_norm:.6f}"
    )

    if abs(embedding_norm - 1.0) > 1e-4:

        raise ValueError(
            "Normalized embedding norm is not "
            "approximately 1.0."
        )

    print(
        "[SUCCESS] Embedding normalization verified."
    )

    # ========================================================
    # STEP 6 — GPU MEMORY INFORMATION
    # ========================================================

    allocated_mb = (
        torch.cuda.memory_allocated()
        / (1024 ** 2)
    )

    reserved_mb = (
        torch.cuda.memory_reserved()
        / (1024 ** 2)
    )

    print()
    print("-" * 60)
    print("GPU MEMORY")
    print("-" * 60)

    print(
        f"Allocated            : "
        f"{allocated_mb:.2f} MB"
    )

    print(
        f"Reserved             : "
        f"{reserved_mb:.2f} MB"
    )

    # ========================================================
    # STEP 7 — SAVE MODEL CHECKPOINT
    # ========================================================

    model_checkpoint = {

        "stage":
            "embedding_model_selection",

        "status":
            "success",

        "model": {

            "name":
                MODEL_NAME,

            "framework":
                "sentence-transformers",

            "embedding_dimension":
                embedding_dimension,

            "max_sequence_length":
                max_sequence_length,

            "device":
                DEVICE,

            "normalized_embeddings":
                True
        },

        "verification": {

            "test_embedding_success":
                True,

            "finite_values":
                True,

            "expected_vector_norm":
                1.0,

            "observed_vector_norm":
                embedding_norm
        },

        "next_stage": {

            "action":
                "generate_embeddings",

            "embeddings_generated":
                False,

            "vector_index_created":
                False
        }
    }

    atomic_json_save(
        model_checkpoint,
        MODEL_CHECKPOINT_PATH
    )

    print()
    print(
        "[SUCCESS] Embedding model checkpoint saved."
    )

    print(
        f"[INFO] Path: "
        f"{MODEL_CHECKPOINT_PATH}"
    )

    # ========================================================
    # STEP 8 — CLEANUP
    # ========================================================

    del test_embedding
    del model

    gc.collect()

    if torch.cuda.is_available():

        torch.cuda.empty_cache()

    print()
    print(
        "[INFO] Test model released from memory."
    )

    print()
    print("=" * 60)
    print("EMBEDDING MODEL VERIFICATION COMPLETE")
    print("=" * 60)

except Exception as e:

    print()
    print("=" * 60)
    print("[ERROR] EMBEDDING MODEL VERIFICATION FAILED")
    print("=" * 60)

    print(
        f"Error type    : {type(e).__name__}"
    )

    print(
        f"Error message : {e}"
    )

    print()
    print(
        "[INFO] No production embeddings were generated."
    )

    print(
        "[INFO] No vector index was created."
    )

    raise

EMBEDDING MODEL SELECTION & VERIFICATION

------------------------------------------------------------
DEPENDENCY VERIFICATION
------------------------------------------------------------
PyTorch              : 2.5.1
CUDA available       : True

------------------------------------------------------------
MODEL CONFIGURATION
------------------------------------------------------------
Model                : BAAI/bge-small-en-v1.5
Device               : cuda

------------------------------------------------------------
MODEL LOADING
------------------------------------------------------------
[INFO] Loading embedding model...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5262.71it/s]


[SUCCESS] Embedding model loaded.

------------------------------------------------------------
MODEL INFORMATION
------------------------------------------------------------
Embedding dimension  : 384
Max sequence length  : 512

------------------------------------------------------------
TEST EMBEDDING
------------------------------------------------------------
[INFO] Generating test embedding...


C:\Users\sbmis\AppData\Local\Temp\ipykernel_21080\2982726458.py:163: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  model.get_sentence_embedding_dimension()


[SUCCESS] Test embedding generated.
Test shape           : (1, 384)
Vector norm          : 1.000000
[SUCCESS] Embedding normalization verified.

------------------------------------------------------------
GPU MEMORY
------------------------------------------------------------
Allocated            : 135.39 MB
Reserved             : 162.00 MB

[SUCCESS] Embedding model checkpoint saved.
[INFO] Path: E:\rag\checkpoints\embedding_model.json

[INFO] Test model released from memory.

EMBEDDING MODEL VERIFICATION COMPLETE


In [12]:
# ============================================================
# CELL 12 — PRODUCTION EMBEDDING GENERATION
# ============================================================

import os
import json
import gc
import time
import tempfile
from pathlib import Path
from datetime import datetime

print("=" * 60)
print("PRODUCTION EMBEDDING GENERATION")
print("=" * 60)

# ------------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------------

PROJECT_ROOT = Path(r"E:\rag")

CHUNKS_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "rag_chunks.json"
)

EMBEDDING_DIR = (
    PROJECT_ROOT
    / "data"
    / "embeddings"
)

EMBEDDINGS_PATH = (
    EMBEDDING_DIR
    / "rag_embeddings.json"
)

PROGRESS_PATH = (
    EMBEDDING_DIR
    / "embedding_progress.json"
)

CHECKPOINT_DIR = (
    PROJECT_ROOT
    / "checkpoints"
)

FINAL_CHECKPOINT_PATH = (
    CHECKPOINT_DIR
    / "embedding_generation.json"
)

MODEL_NAME = "BAAI/bge-small-en-v1.5"
DEVICE = "cuda"

BATCH_SIZE = 32

NORMALIZE_EMBEDDINGS = True

EXPECTED_CHUNKS = 164
EXPECTED_DIMENSION = 384

# ------------------------------------------------------------
# HELPER — ATOMIC JSON SAVE
# ------------------------------------------------------------

def atomic_json_save(data, output_path):

    output_path = Path(output_path)

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    temp_path = None

    try:

        with tempfile.NamedTemporaryFile(
            mode="w",
            encoding="utf-8",
            dir=output_path.parent,
            delete=False,
            suffix=".tmp"
        ) as tmp_file:

            json.dump(
                data,
                tmp_file,
                ensure_ascii=False
            )

            tmp_file.flush()
            os.fsync(tmp_file.fileno())

            temp_path = Path(
                tmp_file.name
            )

        os.replace(
            temp_path,
            output_path
        )

    except Exception:

        if (
            temp_path is not None
            and temp_path.exists()
        ):

            try:
                temp_path.unlink()
            except Exception:
                pass

        raise


# ------------------------------------------------------------
# HELPER — LOAD JSON SAFELY
# ------------------------------------------------------------

def load_json(path):

    path = Path(path)

    if not path.exists():
        return None

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        return json.load(f)


# ------------------------------------------------------------
# HELPER — VALIDATE EMBEDDING RECORD
# ------------------------------------------------------------

def validate_embedding_record(record):

    required_keys = {
        "chunk_id",
        "document_id",
        "embedding",
        "embedding_dimension"
    }

    if not isinstance(
        record,
        dict
    ):
        return False

    if not required_keys.issubset(
        record.keys()
    ):
        return False

    if not record["chunk_id"]:
        return False

    if not record["document_id"]:
        return False

    embedding = record["embedding"]

    if not isinstance(
        embedding,
        list
    ):
        return False

    if len(embedding) != EXPECTED_DIMENSION:
        return False

    if record["embedding_dimension"] != EXPECTED_DIMENSION:
        return False

    # Validate finite numeric values.
    for value in embedding:

        if not isinstance(
            value,
            (int, float)
        ):
            return False

        if not (
            float("-inf")
            < float(value)
            < float("inf")
        ):
            return False

    return True


try:

    # ========================================================
    # STEP 1 — IMPORT DEPENDENCIES
    # ========================================================

    import torch
    from sentence_transformers import SentenceTransformer

    print()
    print("-" * 60)
    print("ENVIRONMENT")
    print("-" * 60)

    if not torch.cuda.is_available():

        raise RuntimeError(
            "CUDA is unavailable."
        )

    print(
        f"Model                : {MODEL_NAME}"
    )

    print(
        f"Device               : {DEVICE}"
    )

    print(
        f"GPU                  : "
        f"{torch.cuda.get_device_name(0)}"
    )

    print(
        f"Batch size           : {BATCH_SIZE}"
    )

    print(
        f"Normalization        : "
        f"{NORMALIZE_EMBEDDINGS}"
    )

    # ========================================================
    # STEP 2 — LOAD CHUNKS
    # ========================================================

    print()
    print("-" * 60)
    print("LOADING CHUNKS")
    print("-" * 60)

    if not CHUNKS_PATH.exists():

        raise FileNotFoundError(
            f"Chunk file not found:\n"
            f"{CHUNKS_PATH}"
        )

    chunks = load_json(
        CHUNKS_PATH
    )

    if not isinstance(
        chunks,
        list
    ):

        raise ValueError(
            "Chunk file must contain a list."
        )

    if len(chunks) != EXPECTED_CHUNKS:

        raise ValueError(
            f"Expected {EXPECTED_CHUNKS} chunks, "
            f"found {len(chunks)}."
        )

    print(
        f"[SUCCESS] Loaded {len(chunks)} chunks."
    )

    # Validate chunk IDs.
    chunk_ids = [
        chunk["chunk_id"]
        for chunk in chunks
    ]

    if len(chunk_ids) != len(
        set(chunk_ids)
    ):

        raise ValueError(
            "Duplicate chunk IDs found."
        )

    print(
        "[SUCCESS] Chunk IDs validated."
    )

    # ========================================================
    # STEP 3 — LOAD EMBEDDING MODEL
    # ========================================================

    print()
    print("-" * 60)
    print("LOADING EMBEDDING MODEL")
    print("-" * 60)

    model = SentenceTransformer(
        MODEL_NAME,
        device=DEVICE
    )

    actual_dimension = (
        model.get_embedding_dimension()
    )

    if actual_dimension != EXPECTED_DIMENSION:

        raise ValueError(
            f"Expected embedding dimension "
            f"{EXPECTED_DIMENSION}, "
            f"got {actual_dimension}."
        )

    print(
        "[SUCCESS] Embedding model loaded."
    )

    print(
        f"Embedding dimension  : "
        f"{actual_dimension}"
    )

    # ========================================================
    # STEP 4 — LOAD EXISTING PROGRESS
    # ========================================================

    print()
    print("-" * 60)
    print("CHECKING EXISTING PROGRESS")
    print("-" * 60)

    existing_embeddings = []

    if PROGRESS_PATH.exists():

        try:

            progress_data = load_json(
                PROGRESS_PATH
            )

            if (
                isinstance(
                    progress_data,
                    dict
                )
                and isinstance(
                    progress_data.get(
                        "embeddings"
                    ),
                    list
                )
            ):

                candidate_embeddings = (
                    progress_data["embeddings"]
                )

                # Validate every persisted record.
                valid_records = all(
                    validate_embedding_record(
                        record
                    )
                    for record
                    in candidate_embeddings
                )

                persisted_ids = [
                    record["chunk_id"]
                    for record
                    in candidate_embeddings
                ]

                unique_ids = (
                    len(persisted_ids)
                    == len(
                        set(persisted_ids)
                    )
                )

                expected_ids = set(
                    chunk_ids
                )

                persisted_id_set = set(
                    persisted_ids
                )

                ids_are_known = (
                    persisted_id_set
                    .issubset(
                        expected_ids
                    )
                )

                if (
                    valid_records
                    and unique_ids
                    and ids_are_known
                ):

                    existing_embeddings = (
                        candidate_embeddings
                    )

                    print(
                        f"[INFO] Valid progress found: "
                        f"{len(existing_embeddings)} "
                        f"embeddings."
                    )

                else:

                    print(
                        "[WARNING] Existing progress "
                        "failed validation."
                    )

                    print(
                        "[INFO] Starting fresh."
                    )

            else:

                print(
                    "[WARNING] Existing progress "
                    "format is invalid."
                )

                print(
                    "[INFO] Starting fresh."
                )

        except Exception as e:

            print(
                f"[WARNING] Could not load existing "
                f"progress: {e}"
            )

            print(
                "[INFO] Starting fresh."
            )

    else:

        print(
            "[INFO] No existing embedding progress found."
        )

    # --------------------------------------------------------
    # Build lookup of already completed chunks.
    # --------------------------------------------------------

    completed = {
        record["chunk_id"]: record
        for record
        in existing_embeddings
    }

    remaining_chunks = [
        chunk
        for chunk in chunks
        if chunk["chunk_id"]
        not in completed
    ]

    print(
        f"[INFO] Completed embeddings : "
        f"{len(completed)}"
    )

    print(
        f"[INFO] Remaining chunks      : "
        f"{len(remaining_chunks)}"
    )

    # ========================================================
    # STEP 5 — GENERATE EMBEDDINGS
    # ========================================================

    print()
    print("-" * 60)
    print("EMBEDDING GENERATION")
    print("-" * 60)

    generation_start = time.perf_counter()

    total_batches = (
        (
            len(remaining_chunks)
            + BATCH_SIZE
            - 1
        )
        // BATCH_SIZE
    )

    processed_this_run = 0

    for batch_start in range(
        0,
        len(remaining_chunks),
        BATCH_SIZE
    ):

        batch = remaining_chunks[
            batch_start:
            batch_start + BATCH_SIZE
        ]

        batch_texts = [
            chunk["text"]
            for chunk in batch
        ]

        batch_number = (
            batch_start // BATCH_SIZE
        ) + 1

        try:

            batch_embeddings = model.encode(
                batch_texts,
                batch_size=BATCH_SIZE,
                convert_to_numpy=True,
                normalize_embeddings=(
                    NORMALIZE_EMBEDDINGS
                ),
                show_progress_bar=False
            )

        except Exception as batch_error:

            raise RuntimeError(
                f"Embedding generation failed "
                f"at batch {batch_number}/"
                f"{total_batches}: "
                f"{batch_error}"
            ) from batch_error

        # ----------------------------------------------------
        # Validate batch shape.
        # ----------------------------------------------------

        if batch_embeddings.ndim != 2:

            raise ValueError(
                f"Batch {batch_number}: "
                f"unexpected embedding shape "
                f"{batch_embeddings.shape}"
            )

        if batch_embeddings.shape[0] != len(batch):

            raise ValueError(
                f"Batch {batch_number}: "
                f"embedding count mismatch."
            )

        if batch_embeddings.shape[1] != EXPECTED_DIMENSION:

            raise ValueError(
                f"Batch {batch_number}: "
                f"embedding dimension mismatch."
            )

        # ----------------------------------------------------
        # Validate finite values.
        # ----------------------------------------------------

        if not (
            torch.isfinite(
                torch.from_numpy(
                    batch_embeddings
                )
            ).all()
        ):

            raise ValueError(
                f"Batch {batch_number}: "
                f"NaN or infinite values detected."
            )

        # ----------------------------------------------------
        # Create embedding records.
        # ----------------------------------------------------

        batch_records = []

        for chunk, vector in zip(
            batch,
            batch_embeddings
        ):

            vector_list = (
                vector.astype(
                    "float32"
                ).tolist()
            )

            record = {

                "chunk_id":
                    chunk["chunk_id"],

                "document_id":
                    chunk["document_id"],

                "embedding":
                    vector_list,

                "embedding_dimension":
                    EXPECTED_DIMENSION,

                "model":
                    MODEL_NAME,

                "normalized":
                    NORMALIZE_EMBEDDINGS
            }

            if not validate_embedding_record(
                record
            ):

                raise ValueError(
                    f"Invalid embedding record "
                    f"for {chunk['chunk_id']}"
                )

            batch_records.append(
                record
            )

        # ----------------------------------------------------
        # Add batch to completed records.
        # ----------------------------------------------------

        for record in batch_records:

            completed[
                record["chunk_id"]
            ] = record

        processed_this_run += len(
            batch_records
        )

        # ----------------------------------------------------
        # Persist after every batch.
        # ----------------------------------------------------

        ordered_embeddings = [
            completed[chunk_id]
            for chunk_id in chunk_ids
            if chunk_id in completed
        ]

        progress_data = {

            "stage":
                "production_embedding_generation",

            "status":
                "in_progress",

            "model":
                MODEL_NAME,

            "embedding_dimension":
                EXPECTED_DIMENSION,

            "normalized":
                NORMALIZE_EMBEDDINGS,

            "batch_size":
                BATCH_SIZE,

            "total_chunks":
                len(chunks),

            "completed_chunks":
                len(ordered_embeddings),

            "remaining_chunks":
                len(chunks)
                - len(ordered_embeddings),

            "embeddings":
                ordered_embeddings
        }

        atomic_json_save(
            progress_data,
            PROGRESS_PATH
        )

        # ----------------------------------------------------
        # Progress display.
        # ----------------------------------------------------

        completed_count = len(
            ordered_embeddings
        )

        percentage = (
            completed_count
            / len(chunks)
            * 100
        )

        elapsed = (
            time.perf_counter()
            - generation_start
        )

        print(
            f"[BATCH {batch_number:02d}/"
            f"{total_batches:02d}] "
            f"Completed: "
            f"{completed_count}/"
            f"{len(chunks)} "
            f"({percentage:.1f}%) | "
            f"Elapsed: {elapsed:.2f}s"
        )

        # Release batch objects.
        del batch_embeddings
        del batch_texts
        del batch_records
        del batch

        gc.collect()

    # ========================================================
    # STEP 6 — FINAL VALIDATION
    # ========================================================

    print()
    print("-" * 60)
    print("FINAL EMBEDDING VALIDATION")
    print("-" * 60)

    final_embeddings = [
        completed[chunk_id]
        for chunk_id in chunk_ids
    ]

    if len(final_embeddings) != EXPECTED_CHUNKS:

        raise ValueError(
            f"Expected {EXPECTED_CHUNKS} "
            f"embeddings, found "
            f"{len(final_embeddings)}."
        )

    final_ids = [
        record["chunk_id"]
        for record in final_embeddings
    ]

    if final_ids != chunk_ids:

        raise ValueError(
            "Embedding order does not match "
            "chunk order."
        )

    for record in final_embeddings:

        if not validate_embedding_record(
            record
        ):

            raise ValueError(
                f"Final validation failed for "
                f"{record['chunk_id']}"
            )

    # --------------------------------------------------------
    # Verify normalized vectors.
    # --------------------------------------------------------

    sample_vectors = torch.tensor(
        [
            record["embedding"]
            for record
            in final_embeddings
        ],
        dtype=torch.float32
    )

    vector_norms = torch.linalg.vector_norm(
        sample_vectors,
        dim=1
    )

    max_norm_error = torch.max(
        torch.abs(
            vector_norms - 1.0
        )
    ).item()

    if (
        NORMALIZE_EMBEDDINGS
        and max_norm_error > 1e-4
    ):

        raise ValueError(
            f"Embedding normalization validation "
            f"failed. Maximum norm error: "
            f"{max_norm_error}"
        )

    print(
        "[SUCCESS] Embedding count validated."
    )

    print(
        "[SUCCESS] Embedding dimensions validated."
    )

    print(
        "[SUCCESS] Embedding values validated."
    )

    print(
        f"[SUCCESS] Maximum norm error: "
        f"{max_norm_error:.8f}"
    )

    # ========================================================
    # STEP 7 — SAVE FINAL EMBEDDING FILE
    # ========================================================

    final_data = {

        "stage":
            "production_embeddings",

        "status":
            "complete",

        "model":
            MODEL_NAME,

        "embedding_dimension":
            EXPECTED_DIMENSION,

        "normalized":
            NORMALIZE_EMBEDDINGS,

        "total_embeddings":
            len(final_embeddings),

        "embeddings":
            final_embeddings
    }

    atomic_json_save(
        final_data,
        EMBEDDINGS_PATH
    )

    print()
    print(
        "[SUCCESS] Final embedding file saved."
    )

    print(
        f"[INFO] Path: {EMBEDDINGS_PATH}"
    )

    # ========================================================
    # STEP 8 — SAVE FINAL CHECKPOINT
    # ========================================================

    total_time = (
        time.perf_counter()
        - generation_start
    )

    final_checkpoint = {

        "stage":
            "production_embedding_generation",

        "status":
            "success",

        "model":
            MODEL_NAME,

        "device":
            DEVICE,

        "gpu":
            torch.cuda.get_device_name(0),

        "batch_size":
            BATCH_SIZE,

        "total_chunks":
            len(chunks),

        "total_embeddings":
            len(final_embeddings),

        "embedding_dimension":
            EXPECTED_DIMENSION,

        "normalized":
            NORMALIZE_EMBEDDINGS,

        "max_norm_error":
            max_norm_error,

        "generation_time_seconds":
            total_time,

        "completed_at":
            datetime.now().isoformat(),

        "source_chunks":
            str(CHUNKS_PATH),

        "embeddings_path":
            str(EMBEDDINGS_PATH),

        "progress_path":
            str(PROGRESS_PATH)
    }

    atomic_json_save(
        final_checkpoint,
        FINAL_CHECKPOINT_PATH
    )

    print()
    print(
        "[SUCCESS] Final embedding checkpoint saved."
    )

    print(
        f"[INFO] Checkpoint: "
        f"{FINAL_CHECKPOINT_PATH}"
    )

    # ========================================================
    # STEP 9 — DISPLAY FINAL RESULTS
    # ========================================================

    print()
    print("=" * 60)
    print("EMBEDDING GENERATION COMPLETE")
    print("=" * 60)

    print(
        f"Chunks                  : "
        f"{len(chunks)}"
    )

    print(
        f"Embeddings              : "
        f"{len(final_embeddings)}"
    )

    print(
        f"Embedding dimension     : "
        f"{EXPECTED_DIMENSION}"
    )

    print(
        f"Model                   : "
        f"{MODEL_NAME}"
    )

    print(
        f"Normalized              : "
        f"{NORMALIZE_EMBEDDINGS}"
    )

    print(
        f"Generation time         : "
        f"{total_time:.2f} seconds"
    )

    print()
    print(
        "[SUCCESS] Production embeddings "
        "are fully validated."
    )

    print(
        "[SUCCESS] Embedding progress persisted."
    )

    print(
        "[SUCCESS] No vector index has been created yet."
    )

    # ========================================================
    # CLEANUP
    # ========================================================

    del chunks
    del final_embeddings
    del completed
    del remaining_chunks
    del sample_vectors
    del vector_norms
    del model

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print()
    print(
        "[INFO] Model and temporary GPU objects released."
    )

except Exception as e:

    print()
    print("=" * 60)
    print("[ERROR] PRODUCTION EMBEDDING GENERATION FAILED")
    print("=" * 60)

    print(
        f"Error type    : {type(e).__name__}"
    )

    print(
        f"Error message : {e}"
    )

    print()
    print(
        "[INFO] Previously persisted embedding "
        "progress was not discarded."
    )

    print(
        "[INFO] You can rerun this cell to resume "
        "from the last valid batch."
    )

    raise

PRODUCTION EMBEDDING GENERATION

------------------------------------------------------------
ENVIRONMENT
------------------------------------------------------------
Model                : BAAI/bge-small-en-v1.5
Device               : cuda
GPU                  : NVIDIA GeForce RTX 4060 Ti
Batch size           : 32
Normalization        : True

------------------------------------------------------------
LOADING CHUNKS
------------------------------------------------------------
[SUCCESS] Loaded 164 chunks.
[SUCCESS] Chunk IDs validated.

------------------------------------------------------------
LOADING EMBEDDING MODEL
------------------------------------------------------------


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 10456.34it/s]


[SUCCESS] Embedding model loaded.
Embedding dimension  : 384

------------------------------------------------------------
CHECKING EXISTING PROGRESS
------------------------------------------------------------
[INFO] Valid progress found: 164 embeddings.
[INFO] Completed embeddings : 164
[INFO] Remaining chunks      : 0

------------------------------------------------------------
EMBEDDING GENERATION
------------------------------------------------------------

------------------------------------------------------------
FINAL EMBEDDING VALIDATION
------------------------------------------------------------
[SUCCESS] Embedding count validated.
[SUCCESS] Embedding dimensions validated.
[SUCCESS] Embedding values validated.
[SUCCESS] Maximum norm error: 0.00000012

[SUCCESS] Final embedding file saved.
[INFO] Path: E:\rag\data\embeddings\rag_embeddings.json

[SUCCESS] Final embedding checkpoint saved.
[INFO] Checkpoint: E:\rag\checkpoints\embedding_generation.json

EMBEDDING GENERATION

In [13]:
# ============================================================
# CELL 13 — FAISS DENSE VECTOR INDEX CONSTRUCTION
# ============================================================

import os
import json
import gc
import tempfile
import importlib.util
from pathlib import Path
from datetime import datetime

print("=" * 60)
print("FAISS DENSE VECTOR INDEX CONSTRUCTION")
print("=" * 60)

# ------------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------------

PROJECT_ROOT = Path(r"E:\rag")

EMBEDDINGS_PATH = (
    PROJECT_ROOT
    / "data"
    / "embeddings"
    / "rag_embeddings.json"
)

INDEX_DIR = (
    PROJECT_ROOT
    / "data"
    / "vector_index"
)

FAISS_INDEX_PATH = (
    INDEX_DIR
    / "dense_index.faiss"
)

MANIFEST_PATH = (
    INDEX_DIR
    / "dense_index_manifest.json"
)

CHECKPOINT_PATH = (
    PROJECT_ROOT
    / "checkpoints"
    / "vector_index_generation.json"
)

MODEL_NAME = "BAAI/bge-small-en-v1.5"

EXPECTED_EMBEDDINGS = 164
EXPECTED_DIMENSION = 384

INDEX_TYPE = "IndexFlatIP"

# ------------------------------------------------------------
# HELPER — ATOMIC JSON SAVE
# ------------------------------------------------------------

def atomic_json_save(data, output_path):

    output_path = Path(output_path)

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    temp_path = None

    try:

        with tempfile.NamedTemporaryFile(
            mode="w",
            encoding="utf-8",
            dir=output_path.parent,
            delete=False,
            suffix=".tmp"
        ) as tmp_file:

            json.dump(
                data,
                tmp_file,
                ensure_ascii=False,
                indent=2
            )

            tmp_file.flush()
            os.fsync(tmp_file.fileno())

            temp_path = Path(
                tmp_file.name
            )

        os.replace(
            temp_path,
            output_path
        )

    except Exception:

        if (
            temp_path is not None
            and temp_path.exists()
        ):

            try:
                temp_path.unlink()
            except Exception:
                pass

        raise


# ------------------------------------------------------------
# HELPER — LOAD JSON
# ------------------------------------------------------------

def load_json(path):

    path = Path(path)

    if not path.exists():

        raise FileNotFoundError(
            f"File not found: {path}"
        )

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        return json.load(f)


try:

    # ========================================================
    # STEP 1 — CHECK FAISS INSTALLATION
    # ========================================================

    print()
    print("-" * 60)
    print("FAISS ENVIRONMENT")
    print("-" * 60)

    faiss_installed = (
        importlib.util.find_spec("faiss")
        is not None
    )

    if not faiss_installed:

        print(
            "[INFO] FAISS is not installed."
        )

        print(
            "[INFO] Installing faiss-cpu..."
        )

        import subprocess
        import sys

        result = subprocess.run(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "faiss-cpu"
            ],
            capture_output=True,
            text=True
        )

        if result.returncode != 0:

            print(result.stdout)
            print(result.stderr)

            raise RuntimeError(
                "FAISS installation failed."
            )

        print(
            "[SUCCESS] faiss-cpu installation completed."
        )

        importlib.invalidate_caches()

    import faiss

    print(
        "[SUCCESS] FAISS imported successfully."
    )

    print(
        f"FAISS version        : "
        f"{getattr(faiss, '__version__', 'unknown')}"
    )

    # ========================================================
    # STEP 2 — LOAD PERSISTED EMBEDDINGS
    # ========================================================

    print()
    print("-" * 60)
    print("LOADING EMBEDDINGS")
    print("-" * 60)

    embedding_data = load_json(
        EMBEDDINGS_PATH
    )

    if not isinstance(
        embedding_data,
        dict
    ):

        raise ValueError(
            "Embedding file must contain a dictionary."
        )

    embeddings = embedding_data.get(
        "embeddings"
    )

    if not isinstance(
        embeddings,
        list
    ):

        raise ValueError(
            "Embedding records are missing."
        )

    if len(embeddings) != EXPECTED_EMBEDDINGS:

        raise ValueError(
            f"Expected {EXPECTED_EMBEDDINGS} "
            f"embeddings, found {len(embeddings)}."
        )

    print(
        f"[SUCCESS] Loaded {len(embeddings)} "
        f"embedding records."
    )

    # ========================================================
    # STEP 3 — VALIDATE EMBEDDING METADATA
    # ========================================================

    print()
    print("-" * 60)
    print("EMBEDDING VALIDATION")
    print("-" * 60)

    embedding_ids = []
    document_ids = []

    for record in embeddings:

        required_keys = {
            "chunk_id",
            "document_id",
            "embedding",
            "embedding_dimension"
        }

        if not required_keys.issubset(
            record.keys()
        ):

            raise ValueError(
                f"Missing fields in embedding record."
            )

        vector = record["embedding"]

        if not isinstance(
            vector,
            list
        ):

            raise ValueError(
                f"{record['chunk_id']}: "
                f"embedding is not a list."
            )

        if len(vector) != EXPECTED_DIMENSION:

            raise ValueError(
                f"{record['chunk_id']}: "
                f"expected dimension "
                f"{EXPECTED_DIMENSION}, "
                f"got {len(vector)}."
            )

        embedding_ids.append(
            record["chunk_id"]
        )

        document_ids.append(
            record["document_id"]
        )

    if len(embedding_ids) != len(
        set(embedding_ids)
    ):

        raise ValueError(
            "Duplicate chunk IDs found."
        )

    print(
        "[SUCCESS] Embedding dimensions validated."
    )

    print(
        "[SUCCESS] Chunk IDs are unique."
    )

    # ========================================================
    # STEP 4 — CREATE NUMPY MATRIX
    # ========================================================

    print()
    print("-" * 60)
    print("BUILDING EMBEDDING MATRIX")
    print("-" * 60)

    import numpy as np

    embedding_matrix = np.asarray(
        [
            record["embedding"]
            for record in embeddings
        ],
        dtype=np.float32
    )

    if embedding_matrix.shape != (
        EXPECTED_EMBEDDINGS,
        EXPECTED_DIMENSION
    ):

        raise ValueError(
            f"Unexpected embedding matrix shape: "
            f"{embedding_matrix.shape}"
        )

    if not np.isfinite(
        embedding_matrix
    ).all():

        raise ValueError(
            "Embedding matrix contains "
            "NaN or infinite values."
        )

    print(
        f"[SUCCESS] Matrix shape: "
        f"{embedding_matrix.shape}"
    )

    # ========================================================
    # STEP 5 — VERIFY NORMALIZATION
    # ========================================================

    print()
    print("-" * 60)
    print("NORMALIZATION VERIFICATION")
    print("-" * 60)

    vector_norms = np.linalg.norm(
        embedding_matrix,
        axis=1
    )

    max_norm_error = float(
        np.max(
            np.abs(
                vector_norms - 1.0
            )
        )
    )

    print(
        f"Maximum norm error   : "
        f"{max_norm_error:.10f}"
    )

    if max_norm_error > 1e-4:

        raise ValueError(
            "Embeddings are not sufficiently normalized "
            "for the cosine/inner-product baseline."
        )

    print(
        "[SUCCESS] All embeddings are normalized."
    )

    # ========================================================
    # STEP 6 — CREATE EXACT INNER-PRODUCT INDEX
    # ========================================================

    print()
    print("-" * 60)
    print("CREATING FAISS INDEX")
    print("-" * 60)

    index = faiss.IndexFlatIP(
        EXPECTED_DIMENSION
    )

    if index.d != EXPECTED_DIMENSION:

        raise ValueError(
            f"FAISS index dimension mismatch: "
            f"{index.d}"
        )

    # Add all vectors.
    index.add(
        embedding_matrix
    )

    if index.ntotal != EXPECTED_EMBEDDINGS:

        raise ValueError(
            f"Expected {EXPECTED_EMBEDDINGS} "
            f"vectors in index, "
            f"found {index.ntotal}."
        )

    print(
        "[SUCCESS] FAISS index constructed."
    )

    print(
        f"Index type           : {INDEX_TYPE}"
    )

    print(
        f"Index dimension      : {index.d}"
    )

    print(
        f"Indexed vectors      : {index.ntotal}"
    )

    # ========================================================
    # STEP 7 — WRITE FAISS INDEX
    # ========================================================

    INDEX_DIR.mkdir(
        parents=True,
        exist_ok=True
    )

    temp_index_path = (
        INDEX_DIR
        / "dense_index.faiss.tmp"
    )

    try:

        faiss.write_index(
            index,
            str(temp_index_path)
        )

        if not temp_index_path.exists():

            raise IOError(
                "Temporary FAISS index was not created."
            )

        os.replace(
            temp_index_path,
            FAISS_INDEX_PATH
        )

    except Exception:

        if temp_index_path.exists():

            try:
                temp_index_path.unlink()
            except Exception:
                pass

        raise

    print(
        "[SUCCESS] FAISS index saved atomically."
    )

    print(
        f"[INFO] Path: {FAISS_INDEX_PATH}"
    )

    # ========================================================
    # STEP 8 — CREATE VECTOR POSITION MANIFEST
    # ========================================================

    print()
    print("-" * 60)
    print("CREATING VECTOR POSITION MANIFEST")
    print("-" * 60)

    vector_mapping = []

    for position, record in enumerate(
        embeddings
    ):

        vector_mapping.append({

            "vector_position":
                position,

            "chunk_id":
                record["chunk_id"],

            "document_id":
                record["document_id"]
        })

    manifest = {

        "index_type":
            INDEX_TYPE,

        "dimension":
            EXPECTED_DIMENSION,

        "metric":
            "inner_product",

        "cosine_equivalent":
            True,

        "normalized_embeddings":
            True,

        "model":
            MODEL_NAME,

        "total_vectors":
            EXPECTED_EMBEDDINGS,

        "vectors":
            vector_mapping
    }

    atomic_json_save(
        manifest,
        MANIFEST_PATH
    )

    print(
        "[SUCCESS] Vector position manifest saved."
    )

    print(
        f"[INFO] Path: {MANIFEST_PATH}"
    )

    # ========================================================
    # STEP 9 — VERIFY PERSISTED INDEX
    # ========================================================

    print()
    print("-" * 60)
    print("PERSISTED INDEX VERIFICATION")
    print("-" * 60)

    reloaded_index = faiss.read_index(
        str(FAISS_INDEX_PATH)
    )

    if reloaded_index.ntotal != EXPECTED_EMBEDDINGS:

        raise ValueError(
            "Reloaded FAISS index contains "
            "an unexpected number of vectors."
        )

    if reloaded_index.d != EXPECTED_DIMENSION:

        raise ValueError(
            "Reloaded FAISS index has "
            "an unexpected dimension."
        )

    reloaded_manifest = load_json(
        MANIFEST_PATH
    )

    if len(
        reloaded_manifest["vectors"]
    ) != EXPECTED_EMBEDDINGS:

        raise ValueError(
            "Manifest vector count mismatch."
        )

    # Check position ordering.
    for position, mapping in enumerate(
        reloaded_manifest["vectors"]
    ):

        if mapping["vector_position"] != position:

            raise ValueError(
                "Manifest vector positions "
                "are not sequential."
            )

        if mapping["chunk_id"] != embedding_ids[position]:

            raise ValueError(
                "Manifest order does not match "
                "embedding order."
            )

    print(
        "[SUCCESS] Persisted FAISS index reloaded."
    )

    print(
        "[SUCCESS] Index vector count verified."
    )

    print(
        "[SUCCESS] Index dimension verified."
    )

    print(
        "[SUCCESS] Vector-position mapping verified."
    )

    # ========================================================
    # STEP 10 — SAVE CHECKPOINT
    # ========================================================

    checkpoint = {

        "stage":
            "dense_vector_index_generation",

        "status":
            "success",

        "index_type":
            INDEX_TYPE,

        "metric":
            "inner_product",

        "cosine_equivalent":
            True,

        "normalized_embeddings":
            True,

        "model":
            MODEL_NAME,

        "dimension":
            EXPECTED_DIMENSION,

        "total_vectors":
            EXPECTED_EMBEDDINGS,

        "max_norm_error":
            max_norm_error,

        "faiss_index_path":
            str(FAISS_INDEX_PATH),

        "manifest_path":
            str(MANIFEST_PATH),

        "source_embeddings":
            str(EMBEDDINGS_PATH),

        "created_at":
            datetime.now().isoformat()
    }

    atomic_json_save(
        checkpoint,
        CHECKPOINT_PATH
    )

    print()
    print(
        "[SUCCESS] Vector-index checkpoint saved."
    )

    print(
        f"[INFO] Checkpoint: {CHECKPOINT_PATH}"
    )

    # ========================================================
    # STEP 11 — FINAL RESULTS
    # ========================================================

    print()
    print("=" * 60)
    print("VECTOR INDEX CONSTRUCTION COMPLETE")
    print("=" * 60)

    print(
        f"Index type            : {INDEX_TYPE}"
    )

    print(
        f"Dimension             : "
        f"{EXPECTED_DIMENSION}"
    )

    print(
        f"Vectors indexed       : "
        f"{EXPECTED_EMBEDDINGS}"
    )

    print(
        f"Similarity metric     : "
        f"Inner Product"
    )

    print(
        f"Cosine equivalent     : "
        f"Yes"
    )

    print()
    print(
        "[SUCCESS] Dense vector index is persistent."
    )

    print(
        "[SUCCESS] Vector-to-chunk mapping is persistent."
    )

    print(
        "[SUCCESS] Index was successfully reloaded."
    )

    print(
        "[INFO] Retrieval has not been executed yet."
    )

    # ========================================================
    # CLEANUP
    # ========================================================

    del embedding_data
    del embeddings
    del embedding_matrix
    del vector_norms
    del index
    del reloaded_index
    del manifest
    del reloaded_manifest
    del vector_mapping

    gc.collect()

    print()
    print(
        "[INFO] Temporary index-generation objects released."
    )

except Exception as e:

    print()
    print("=" * 60)
    print("[ERROR] VECTOR INDEX CONSTRUCTION FAILED")
    print("=" * 60)

    print(
        f"Error type    : {type(e).__name__}"
    )

    print(
        f"Error message : {e}"
    )

    print()
    print(
        "[INFO] Source embeddings were not modified."
    )

    print(
        "[INFO] Retrieval has not been started."
    )

    raise

FAISS DENSE VECTOR INDEX CONSTRUCTION

------------------------------------------------------------
FAISS ENVIRONMENT
------------------------------------------------------------
[SUCCESS] FAISS imported successfully.
FAISS version        : 1.15.0

------------------------------------------------------------
LOADING EMBEDDINGS
------------------------------------------------------------
[SUCCESS] Loaded 164 embedding records.

------------------------------------------------------------
EMBEDDING VALIDATION
------------------------------------------------------------
[SUCCESS] Embedding dimensions validated.
[SUCCESS] Chunk IDs are unique.

------------------------------------------------------------
BUILDING EMBEDDING MATRIX
------------------------------------------------------------
[SUCCESS] Matrix shape: (164, 384)

------------------------------------------------------------
NORMALIZATION VERIFICATION
------------------------------------------------------------
Maximum norm error

In [14]:
# ============================================================
# CELL 14 — QUERY EMBEDDING + DENSE RETRIEVAL TEST
# ============================================================

import os
import json
import gc
import tempfile
from pathlib import Path
from datetime import datetime

print("=" * 60)
print("QUERY EMBEDDING + DENSE RETRIEVAL TEST")
print("=" * 60)

# ------------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------------

PROJECT_ROOT = Path(r"E:\rag")

EMBEDDINGS_PATH = (
    PROJECT_ROOT
    / "data"
    / "embeddings"
    / "rag_embeddings.json"
)

CHUNKS_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "rag_chunks.json"
)

FAISS_INDEX_PATH = (
    PROJECT_ROOT
    / "data"
    / "vector_index"
    / "dense_index.faiss"
)

MANIFEST_PATH = (
    PROJECT_ROOT
    / "data"
    / "vector_index"
    / "dense_index_manifest.json"
)

RESULTS_DIR = (
    PROJECT_ROOT
    / "data"
    / "retrieval"
)

RESULT_PATH = (
    RESULTS_DIR
    / "dense_retrieval_test.json"
)

CHECKPOINT_PATH = (
    PROJECT_ROOT
    / "checkpoints"
    / "retrieval_test.json"
)

MODEL_NAME = "BAAI/bge-small-en-v1.5"

EXPECTED_DIMENSION = 384
EXPECTED_CHUNKS = 164
TOP_K = 5

# ------------------------------------------------------------
# TEST QUERY
# ------------------------------------------------------------

QUERY = (
    "What is the purpose of a firewall and how does it "
    "protect a network?"
)

print()
print(f"Query: {QUERY}")

# ------------------------------------------------------------
# HELPER — LOAD JSON
# ------------------------------------------------------------

def load_json(path):

    path = Path(path)

    if not path.exists():

        raise FileNotFoundError(
            f"File not found: {path}"
        )

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        return json.load(f)


# ------------------------------------------------------------
# HELPER — ATOMIC JSON SAVE
# ------------------------------------------------------------

def atomic_json_save(data, output_path):

    output_path = Path(output_path)

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    temp_path = None

    try:

        with tempfile.NamedTemporaryFile(
            mode="w",
            encoding="utf-8",
            dir=output_path.parent,
            delete=False,
            suffix=".tmp"
        ) as tmp_file:

            json.dump(
                data,
                tmp_file,
                ensure_ascii=False,
                indent=2
            )

            tmp_file.flush()
            os.fsync(tmp_file.fileno())

            temp_path = Path(
                tmp_file.name
            )

        os.replace(
            temp_path,
            output_path
        )

    except Exception:

        if (
            temp_path is not None
            and temp_path.exists()
        ):

            try:
                temp_path.unlink()
            except Exception:
                pass

        raise


try:

    # ========================================================
    # STEP 1 — VERIFY REQUIRED FILES
    # ========================================================

    print()
    print("-" * 60)
    print("VERIFYING PERSISTED RAG COMPONENTS")
    print("-" * 60)

    required_files = [
        EMBEDDINGS_PATH,
        CHUNKS_PATH,
        FAISS_INDEX_PATH,
        MANIFEST_PATH
    ]

    for path in required_files:

        if not path.exists():

            raise FileNotFoundError(
                f"Required file missing: {path}"
            )

        print(
            f"[OK] {path}"
        )

    # ========================================================
    # STEP 2 — IMPORT DEPENDENCIES
    # ========================================================

    print()
    print("-" * 60)
    print("LOADING RETRIEVAL DEPENDENCIES")
    print("-" * 60)

    import numpy as np
    import faiss

    from sentence_transformers import (
        SentenceTransformer
    )

    print(
        "[SUCCESS] NumPy loaded."
    )

    print(
        "[SUCCESS] FAISS loaded."
    )

    print(
        "[SUCCESS] Sentence Transformers loaded."
    )

    # ========================================================
    # STEP 3 — LOAD CHUNKS
    # ========================================================

    print()
    print("-" * 60)
    print("LOADING CHUNKS")
    print("-" * 60)

    chunk_data = load_json(
        CHUNKS_PATH
    )

    # Support either direct list or dictionary format.
    if isinstance(
        chunk_data,
        dict
    ):

        chunks = chunk_data.get(
            "chunks"
        )

    else:

        chunks = chunk_data

    if not isinstance(
        chunks,
        list
    ):

        raise ValueError(
            "Chunk data does not contain a valid chunk list."
        )

    if len(chunks) != EXPECTED_CHUNKS:

        raise ValueError(
            f"Expected {EXPECTED_CHUNKS} chunks, "
            f"found {len(chunks)}."
        )

    chunk_lookup = {}

    for chunk in chunks:

        chunk_id = chunk.get(
            "chunk_id"
        )

        if not chunk_id:

            raise ValueError(
                "Chunk without chunk_id found."
            )

        if chunk_id in chunk_lookup:

            raise ValueError(
                f"Duplicate chunk ID: {chunk_id}"
            )

        chunk_lookup[chunk_id] = chunk

    print(
        f"[SUCCESS] Loaded {len(chunks)} chunks."
    )

    print(
        f"[SUCCESS] Chunk lookup contains "
        f"{len(chunk_lookup)} unique IDs."
    )

    # ========================================================
    # STEP 4 — LOAD FAISS INDEX
    # ========================================================

    print()
    print("-" * 60)
    print("LOADING FAISS INDEX")
    print("-" * 60)

    index = faiss.read_index(
        str(FAISS_INDEX_PATH)
    )

    if index.ntotal != EXPECTED_CHUNKS:

        raise ValueError(
            f"FAISS index contains {index.ntotal} vectors; "
            f"expected {EXPECTED_CHUNKS}."
        )

    if index.d != EXPECTED_DIMENSION:

        raise ValueError(
            f"FAISS index dimension is {index.d}; "
            f"expected {EXPECTED_DIMENSION}."
        )

    print(
        "[SUCCESS] FAISS index loaded."
    )

    print(
        f"Indexed vectors      : {index.ntotal}"
    )

    print(
        f"Index dimension      : {index.d}"
    )

    # ========================================================
    # STEP 5 — LOAD MANIFEST
    # ========================================================

    print()
    print("-" * 60)
    print("LOADING VECTOR MANIFEST")
    print("-" * 60)

    manifest = load_json(
        MANIFEST_PATH
    )

    vector_mapping = manifest.get(
        "vectors"
    )

    if not isinstance(
        vector_mapping,
        list
    ):

        raise ValueError(
            "Vector manifest is missing vector mappings."
        )

    if len(vector_mapping) != EXPECTED_CHUNKS:

        raise ValueError(
            f"Manifest contains "
            f"{len(vector_mapping)} mappings; "
            f"expected {EXPECTED_CHUNKS}."
        )

    # Validate mapping.
    for position, mapping in enumerate(
        vector_mapping
    ):

        if mapping["vector_position"] != position:

            raise ValueError(
                "Vector positions are not sequential."
            )

        if mapping["chunk_id"] not in chunk_lookup:

            raise ValueError(
                f"Manifest references unknown chunk: "
                f"{mapping['chunk_id']}"
            )

    print(
        "[SUCCESS] Vector manifest validated."
    )

    # ========================================================
    # STEP 6 — LOAD EMBEDDING MODEL
    # ========================================================

    print()
    print("-" * 60)
    print("LOADING QUERY EMBEDDING MODEL")
    print("-" * 60)

    import torch

    device = (
        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    )

    print(
        f"Device               : {device}"
    )

    print(
        f"Model                : {MODEL_NAME}"
    )

    model = SentenceTransformer(
        MODEL_NAME,
        device=device
    )

    model_dimension = (
        model.get_embedding_dimension()
    )

    if model_dimension != EXPECTED_DIMENSION:

        raise ValueError(
            f"Model dimension is {model_dimension}; "
            f"expected {EXPECTED_DIMENSION}."
        )

    print(
        "[SUCCESS] Embedding model loaded."
    )

    print(
        f"Embedding dimension  : {model_dimension}"
    )

    # ========================================================
    # STEP 7 — EMBED QUERY
    # ========================================================

    print()
    print("-" * 60)
    print("EMBEDDING QUERY")
    print("-" * 60)

    query_embedding = model.encode(
        [QUERY],
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False
    )

    query_embedding = np.asarray(
        query_embedding,
        dtype=np.float32
    )

    if query_embedding.shape != (
        1,
        EXPECTED_DIMENSION
    ):

        raise ValueError(
            f"Unexpected query embedding shape: "
            f"{query_embedding.shape}"
        )

    if not np.isfinite(
        query_embedding
    ).all():

        raise ValueError(
            "Query embedding contains NaN or infinite values."
        )

    query_norm = float(
        np.linalg.norm(
            query_embedding[0]
        )
    )

    if abs(query_norm - 1.0) > 1e-4:

        raise ValueError(
            f"Query embedding is not normalized. "
            f"Norm = {query_norm}"
        )

    print(
        "[SUCCESS] Query embedded."
    )

    print(
        f"Shape                : "
        f"{query_embedding.shape}"
    )

    print(
        f"Query vector norm    : "
        f"{query_norm:.8f}"
    )

    # ========================================================
    # STEP 8 — PERFORM DENSE RETRIEVAL
    # ========================================================

    print()
    print("-" * 60)
    print("DENSE RETRIEVAL")
    print("-" * 60)

    scores, positions = index.search(
        query_embedding,
        TOP_K
    )

    scores = scores[0]
    positions = positions[0]

    if len(scores) != TOP_K:

        raise ValueError(
            f"Expected {TOP_K} retrieval results, "
            f"received {len(scores)}."
        )

    retrieval_results = []

    for rank, (
        score,
        position
    ) in enumerate(
        zip(scores, positions),
        start=1
    ):

        position = int(position)
        score = float(score)

        if position < 0:

            raise ValueError(
                "FAISS returned an invalid vector position."
            )

        if position >= len(vector_mapping):

            raise ValueError(
                "FAISS returned a position outside "
                "the manifest."
            )

        mapping = vector_mapping[position]

        chunk_id = mapping[
            "chunk_id"
        ]

        chunk = chunk_lookup.get(
            chunk_id
        )

        if chunk is None:

            raise ValueError(
                f"Chunk {chunk_id} not found."
            )

        retrieval_results.append({

            "rank":
                rank,

            "score":
                score,

            "vector_position":
                position,

            "chunk_id":
                chunk_id,

            "document_id":
                mapping["document_id"],

            "topic":
                chunk.get(
                    "topic",
                    chunk.get(
                        "metadata",
                        {}
                    ).get(
                        "topic"
                    )
                ),

            "text":
                chunk.get(
                    "text",
                    ""
                )
        })

    # ========================================================
    # STEP 9 — VALIDATE RETRIEVAL RESULTS
    # ========================================================

    print()
    print("-" * 60)
    print("VALIDATING RETRIEVAL RESULTS")
    print("-" * 60)

    result_chunk_ids = [
        result["chunk_id"]
        for result in retrieval_results
    ]

    if len(result_chunk_ids) != len(
        set(result_chunk_ids)
    ):

        raise ValueError(
            "Duplicate chunks returned in Top-K results."
        )

    result_scores = [
        result["score"]
        for result in retrieval_results
    ]

    if not all(
        np.isfinite(result_scores)
    ):

        raise ValueError(
            "Retrieval contains invalid similarity scores."
        )

    # Scores from FAISS should be in descending order.
    if any(
        result_scores[i]
        < result_scores[i + 1]
        for i in range(
            len(result_scores) - 1
        )
    ):

        raise ValueError(
            "Retrieval scores are not sorted "
            "in descending order."
        )

    print(
        "[SUCCESS] Top-K result count validated."
    )

    print(
        "[SUCCESS] Result chunk IDs are unique."
    )

    print(
        "[SUCCESS] Similarity scores are valid."
    )

    print(
        "[SUCCESS] Results are ordered by relevance score."
    )

    # ========================================================
    # STEP 10 — DISPLAY RETRIEVED RESULTS
    # ========================================================

    print()
    print("-" * 60)
    print(f"TOP-{TOP_K} RETRIEVED CHUNKS")
    print("-" * 60)

    for result in retrieval_results:

        print()
        print(
            f"Rank {result['rank']}"
        )

        print(
            f"Score       : "
            f"{result['score']:.6f}"
        )

        print(
            f"Chunk ID     : "
            f"{result['chunk_id']}"
        )

        print(
            f"Document ID  : "
            f"{result['document_id']}"
        )

        print(
            f"Topic        : "
            f"{result['topic']}"
        )

        text_preview = (
            result["text"]
            .replace("\n", " ")
            .strip()
        )

        if len(text_preview) > 350:

            text_preview = (
                text_preview[:350]
                + "..."
            )

        print(
            f"Text         : "
            f"{text_preview}"
        )

    # ========================================================
    # STEP 11 — BUILD PERSISTENT RESULT
    # ========================================================

    print()
    print("-" * 60)
    print("PERSISTING RETRIEVAL RESULTS")
    print("-" * 60)

    retrieval_record = {

        "stage":
            "dense_retrieval_test",

        "status":
            "success",

        "query":
            QUERY,

        "retrieval_method":
            "dense",

        "index_type":
            "IndexFlatIP",

        "similarity_metric":
            "inner_product",

        "cosine_equivalent":
            True,

        "embedding_model":
            MODEL_NAME,

        "embedding_dimension":
            EXPECTED_DIMENSION,

        "top_k":
            TOP_K,

        "query_embedding_norm":
            query_norm,

        "results":
            retrieval_results,

        "created_at":
            datetime.now().isoformat()
    }

    atomic_json_save(
        retrieval_record,
        RESULT_PATH
    )

    print(
        "[SUCCESS] Retrieval results saved."
    )

    print(
        f"[INFO] Result path: {RESULT_PATH}"
    )

    # ========================================================
    # STEP 12 — SAVE CHECKPOINT
    # ========================================================

    checkpoint = {

        "stage":
            "dense_retrieval_test",

        "status":
            "success",

        "query":
            QUERY,

        "top_k":
            TOP_K,

        "retrieved_results":
            len(retrieval_results),

        "embedding_model":
            MODEL_NAME,

        "embedding_dimension":
            EXPECTED_DIMENSION,

        "index_type":
            "IndexFlatIP",

        "similarity_metric":
            "inner_product",

        "result_path":
            str(RESULT_PATH),

        "created_at":
            datetime.now().isoformat()
    }

    atomic_json_save(
        checkpoint,
        CHECKPOINT_PATH
    )

    print(
        "[SUCCESS] Retrieval checkpoint saved."
    )

    print(
        f"[INFO] Checkpoint: {CHECKPOINT_PATH}"
    )

    # ========================================================
    # STEP 13 — CLEANUP
    # ========================================================

    del model
    del index
    del chunk_data
    del chunks
    del chunk_lookup
    del manifest
    del vector_mapping
    del query_embedding

    gc.collect()

    if torch.cuda.is_available():

        torch.cuda.empty_cache()

        print(
            "[INFO] CUDA cache released."
        )

    print()
    print("=" * 60)
    print("DENSE RETRIEVAL TEST COMPLETE")
    print("=" * 60)

    print(
        f"Query                  : {QUERY}"
    )

    print(
        f"Top-K                  : {TOP_K}"
    )

    print(
        f"Results returned       : "
        f"{len(retrieval_results)}"
    )

    print()
    print(
        "[SUCCESS] Query embedding completed."
    )

    print(
        "[SUCCESS] FAISS retrieval completed."
    )

    print(
        "[SUCCESS] Retrieval results persisted."
    )

    print(
        "[INFO] LLM generation has NOT been performed yet."
    )

except Exception as e:

    print()
    print("=" * 60)
    print("[ERROR] DENSE RETRIEVAL TEST FAILED")
    print("=" * 60)

    print(
        f"Error type    : {type(e).__name__}"
    )

    print(
        f"Error message : {e}"
    )

    print()
    print(
        "[INFO] Existing embeddings and FAISS index "
        "were not modified."
    )

    raise

QUERY EMBEDDING + DENSE RETRIEVAL TEST

Query: What is the purpose of a firewall and how does it protect a network?

------------------------------------------------------------
VERIFYING PERSISTED RAG COMPONENTS
------------------------------------------------------------
[OK] E:\rag\data\embeddings\rag_embeddings.json
[OK] E:\rag\data\processed\rag_chunks.json
[OK] E:\rag\data\vector_index\dense_index.faiss
[OK] E:\rag\data\vector_index\dense_index_manifest.json

------------------------------------------------------------
LOADING RETRIEVAL DEPENDENCIES
------------------------------------------------------------
[SUCCESS] NumPy loaded.
[SUCCESS] FAISS loaded.
[SUCCESS] Sentence Transformers loaded.

------------------------------------------------------------
LOADING CHUNKS
------------------------------------------------------------
[SUCCESS] Loaded 164 chunks.
[SUCCESS] Chunk lookup contains 164 unique IDs.

------------------------------------------------------------
LOADING FAIS

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9681.45it/s]


[SUCCESS] Embedding model loaded.
Embedding dimension  : 384

------------------------------------------------------------
EMBEDDING QUERY
------------------------------------------------------------
[SUCCESS] Query embedded.
Shape                : (1, 384)
Query vector norm    : 1.00000000

------------------------------------------------------------
DENSE RETRIEVAL
------------------------------------------------------------

------------------------------------------------------------
VALIDATING RETRIEVAL RESULTS
------------------------------------------------------------
[SUCCESS] Top-K result count validated.
[SUCCESS] Result chunk IDs are unique.
[SUCCESS] Similarity scores are valid.
[SUCCESS] Results are ordered by relevance score.

------------------------------------------------------------
TOP-5 RETRIEVED CHUNKS
------------------------------------------------------------

Rank 1
Score       : 0.716885
Chunk ID     : doc_008_chunk_002
Document ID  : doc_008
Topic        : S

In [15]:
# ============================================================
# CELL 15 — REUSABLE DENSE RETRIEVAL ENGINE
# ============================================================

import os
import json
import gc
import tempfile
from pathlib import Path
from datetime import datetime

print("=" * 60)
print("REUSABLE DENSE RETRIEVAL ENGINE")
print("=" * 60)

# ------------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------------

PROJECT_ROOT = Path(r"E:\rag")

CHUNKS_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "rag_chunks.json"
)

FAISS_INDEX_PATH = (
    PROJECT_ROOT
    / "data"
    / "vector_index"
    / "dense_index.faiss"
)

MANIFEST_PATH = (
    PROJECT_ROOT
    / "data"
    / "vector_index"
    / "dense_index_manifest.json"
)

CONFIG_PATH = (
    PROJECT_ROOT
    / "checkpoints"
    / "retrieval_engine.json"
)

MODEL_NAME = "BAAI/bge-small-en-v1.5"

EMBEDDING_DIMENSION = 384

DEFAULT_TOP_K = 5

INDEX_TYPE = "IndexFlatIP"

SIMILARITY_METRIC = "inner_product"

# ------------------------------------------------------------
# HELPER — LOAD JSON
# ------------------------------------------------------------

def load_json(path):

    path = Path(path)

    if not path.exists():

        raise FileNotFoundError(
            f"Required file not found: {path}"
        )

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        return json.load(f)


# ------------------------------------------------------------
# HELPER — ATOMIC JSON SAVE
# ------------------------------------------------------------

def atomic_json_save(data, output_path):

    output_path = Path(output_path)

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    temp_path = None

    try:

        with tempfile.NamedTemporaryFile(
            mode="w",
            encoding="utf-8",
            dir=output_path.parent,
            delete=False,
            suffix=".tmp"
        ) as tmp_file:

            json.dump(
                data,
                tmp_file,
                ensure_ascii=False,
                indent=2
            )

            tmp_file.flush()
            os.fsync(tmp_file.fileno())

            temp_path = Path(
                tmp_file.name
            )

        os.replace(
            temp_path,
            output_path
        )

    except Exception:

        if (
            temp_path is not None
            and temp_path.exists()
        ):

            try:
                temp_path.unlink()
            except Exception:
                pass

        raise


try:

    # ========================================================
    # STEP 1 — VERIFY REQUIRED FILES
    # ========================================================

    print()
    print("-" * 60)
    print("VERIFYING RETRIEVAL COMPONENTS")
    print("-" * 60)

    required_files = [
        CHUNKS_PATH,
        FAISS_INDEX_PATH,
        MANIFEST_PATH
    ]

    for path in required_files:

        if not path.exists():

            raise FileNotFoundError(
                f"Missing required component: {path}"
            )

        print(
            f"[OK] {path}"
        )

    # ========================================================
    # STEP 2 — LOAD DEPENDENCIES
    # ========================================================

    print()
    print("-" * 60)
    print("LOADING DEPENDENCIES")
    print("-" * 60)

    import numpy as np
    import faiss
    import torch

    from sentence_transformers import (
        SentenceTransformer
    )

    print(
        "[SUCCESS] NumPy loaded."
    )

    print(
        "[SUCCESS] FAISS loaded."
    )

    print(
        "[SUCCESS] PyTorch loaded."
    )

    print(
        "[SUCCESS] Sentence Transformers loaded."
    )

    # ========================================================
    # STEP 3 — LOAD CHUNKS
    # ========================================================

    print()
    print("-" * 60)
    print("LOADING CHUNKS")
    print("-" * 60)

    chunk_data = load_json(
        CHUNKS_PATH
    )

    if isinstance(
        chunk_data,
        dict
    ):

        chunks = chunk_data.get(
            "chunks"
        )

    else:

        chunks = chunk_data

    if not isinstance(
        chunks,
        list
    ):

        raise ValueError(
            "Invalid chunk structure."
        )

    chunk_lookup = {}

    for chunk in chunks:

        chunk_id = chunk.get(
            "chunk_id"
        )

        if not chunk_id:

            raise ValueError(
                "Chunk without chunk_id detected."
            )

        if chunk_id in chunk_lookup:

            raise ValueError(
                f"Duplicate chunk ID: {chunk_id}"
            )

        chunk_lookup[chunk_id] = chunk

    print(
        f"[SUCCESS] Loaded {len(chunks)} chunks."
    )

    print(
        f"[SUCCESS] Created lookup for "
        f"{len(chunk_lookup)} chunks."
    )

    # ========================================================
    # STEP 4 — LOAD FAISS INDEX
    # ========================================================

    print()
    print("-" * 60)
    print("LOADING FAISS INDEX")
    print("-" * 60)

    index = faiss.read_index(
        str(FAISS_INDEX_PATH)
    )

    if index.d != EMBEDDING_DIMENSION:

        raise ValueError(
            f"Index dimension mismatch: "
            f"{index.d} != {EMBEDDING_DIMENSION}"
        )

    if index.ntotal != len(chunks):

        raise ValueError(
            f"Index/chunk count mismatch: "
            f"{index.ntotal} != {len(chunks)}"
        )

    print(
        "[SUCCESS] FAISS index loaded."
    )

    print(
        f"Index type            : {INDEX_TYPE}"
    )

    print(
        f"Vectors               : {index.ntotal}"
    )

    print(
        f"Dimension             : {index.d}"
    )

    # ========================================================
    # STEP 5 — LOAD AND VALIDATE MANIFEST
    # ========================================================

    print()
    print("-" * 60)
    print("VALIDATING VECTOR MANIFEST")
    print("-" * 60)

    manifest = load_json(
        MANIFEST_PATH
    )

    vector_mapping = manifest.get(
        "vectors"
    )

    if not isinstance(
        vector_mapping,
        list
    ):

        raise ValueError(
            "Invalid vector manifest."
        )

    if len(vector_mapping) != index.ntotal:

        raise ValueError(
            "Manifest/index count mismatch."
        )

    for position, mapping in enumerate(
        vector_mapping
    ):

        if mapping["vector_position"] != position:

            raise ValueError(
                "Vector position mismatch."
            )

        if mapping["chunk_id"] not in chunk_lookup:

            raise ValueError(
                f"Manifest references unknown chunk: "
                f"{mapping['chunk_id']}"
            )

    print(
        "[SUCCESS] Vector manifest validated."
    )

    # ========================================================
    # STEP 6 — LOAD EMBEDDING MODEL
    # ========================================================

    print()
    print("-" * 60)
    print("LOADING QUERY EMBEDDING MODEL")
    print("-" * 60)

    device = (
        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    )

    print(
        f"Device                : {device}"
    )

    print(
        f"Model                 : {MODEL_NAME}"
    )

    model = SentenceTransformer(
        MODEL_NAME,
        device=device
    )

    model_dimension = (
        model.get_embedding_dimension()
    )

    if model_dimension != EMBEDDING_DIMENSION:

        raise ValueError(
            f"Model dimension mismatch: "
            f"{model_dimension} != "
            f"{EMBEDDING_DIMENSION}"
        )

    print(
        "[SUCCESS] Query embedding model loaded."
    )

    print(
        f"Embedding dimension   : "
        f"{model_dimension}"
    )

    # ========================================================
    # STEP 7 — DEFINE RETRIEVAL FUNCTION
    # ========================================================

    print()
    print("-" * 60)
    print("CREATING RETRIEVAL FUNCTION")
    print("-" * 60)

    def retrieve(
        query,
        top_k=DEFAULT_TOP_K
    ):
        """
        Perform dense retrieval for a natural-language query.

        Parameters
        ----------
        query : str
            User's natural-language question.

        top_k : int
            Number of chunks to retrieve.

        Returns
        -------
        list
            Ranked retrieval results.
        """

        if not isinstance(
            query,
            str
        ):

            raise TypeError(
                "Query must be a string."
            )

        query = query.strip()

        if not query:

            raise ValueError(
                "Query cannot be empty."
            )

        if not isinstance(
            top_k,
            int
        ):

            raise TypeError(
                "top_k must be an integer."
            )

        if top_k < 1:

            raise ValueError(
                "top_k must be at least 1."
            )

        if top_k > index.ntotal:

            raise ValueError(
                f"top_k={top_k} exceeds "
                f"indexed vectors={index.ntotal}."
            )

        # ----------------------------------------------------
        # Query embedding
        # ----------------------------------------------------

        query_embedding = model.encode(
            [query],
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=False
        )

        query_embedding = np.asarray(
            query_embedding,
            dtype=np.float32
        )

        if query_embedding.shape != (
            1,
            EMBEDDING_DIMENSION
        ):

            raise ValueError(
                "Unexpected query embedding shape."
            )

        if not np.isfinite(
            query_embedding
        ).all():

            raise ValueError(
                "Query embedding contains "
                "invalid numerical values."
            )

        # ----------------------------------------------------
        # FAISS search
        # ----------------------------------------------------

        scores, positions = index.search(
            query_embedding,
            top_k
        )

        scores = scores[0]
        positions = positions[0]

        results = []

        for rank, (
            score,
            position
        ) in enumerate(
            zip(scores, positions),
            start=1
        ):

            position = int(position)
            score = float(score)

            if position < 0:

                raise ValueError(
                    "FAISS returned invalid position."
                )

            mapping = vector_mapping[
                position
            ]

            chunk_id = mapping[
                "chunk_id"
            ]

            chunk = chunk_lookup.get(
                chunk_id
            )

            if chunk is None:

                raise ValueError(
                    f"Chunk not found: {chunk_id}"
                )

            metadata = chunk.get(
                "metadata",
                {}
            )

            results.append({

                "rank":
                    rank,

                "score":
                    score,

                "vector_position":
                    position,

                "chunk_id":
                    chunk_id,

                "document_id":
                    mapping[
                        "document_id"
                    ],

                "topic":
                    chunk.get(
                        "topic",
                        metadata.get(
                            "topic"
                        )
                    ),

                "text":
                    chunk.get(
                        "text",
                        ""
                    )
            })

        return results

    print(
        "[SUCCESS] Reusable retrieve() function created."
    )

    # ========================================================
    # STEP 8 — RUN FUNCTION VALIDATION TEST
    # ========================================================

    print()
    print("-" * 60)
    print("RETRIEVAL FUNCTION VALIDATION")
    print("-" * 60)

    test_query = (
        "What is the purpose of a firewall "
        "and how does it protect a network?"
    )

    test_results = retrieve(
        query=test_query,
        top_k=DEFAULT_TOP_K
    )

    if len(test_results) != DEFAULT_TOP_K:

        raise ValueError(
            "Retrieval function returned "
            "an incorrect number of results."
        )

    result_ids = [
        result["chunk_id"]
        for result in test_results
    ]

    if len(result_ids) != len(
        set(result_ids)
    ):

        raise ValueError(
            "Retrieval returned duplicate chunks."
        )

    result_scores = [
        result["score"]
        for result in test_results
    ]

    if any(
        not np.isfinite(score)
        for score in result_scores
    ):

        raise ValueError(
            "Retrieval returned invalid scores."
        )

    if any(
        result_scores[i]
        < result_scores[i + 1]
        for i in range(
            len(result_scores) - 1
        )
    ):

        raise ValueError(
            "Retrieval scores are not "
            "sorted correctly."
        )

    print(
        "[SUCCESS] Test query retrieved "
        f"{len(test_results)} results."
    )

    print()
    print("Test retrieval ranking:")

    for result in test_results:

        print(
            f"  {result['rank']}. "
            f"{result['chunk_id']} | "
            f"{result['score']:.6f} | "
            f"{result['topic']}"
        )

    # ========================================================
    # STEP 9 — SAVE RETRIEVAL ENGINE CONFIGURATION
    # ========================================================

    print()
    print("-" * 60)
    print("SAVING RETRIEVAL ENGINE CONFIGURATION")
    print("-" * 60)

    retrieval_config = {

        "stage":
            "reusable_dense_retrieval_engine",

        "status":
            "success",

        "retrieval_method":
            "dense",

        "index_type":
            INDEX_TYPE,

        "similarity_metric":
            SIMILARITY_METRIC,

        "cosine_equivalent":
            True,

        "embedding_model":
            MODEL_NAME,

        "embedding_dimension":
            EMBEDDING_DIMENSION,

        "default_top_k":
            DEFAULT_TOP_K,

        "indexed_vectors":
            index.ntotal,

        "chunk_count":
            len(chunks),

        "device":
            device,

        "test_query":
            test_query,

        "test_result_chunk_ids":
            result_ids,

        "created_at":
            datetime.now().isoformat()
    }

    atomic_json_save(
        retrieval_config,
        CONFIG_PATH
    )

    print(
        "[SUCCESS] Retrieval engine configuration saved."
    )

    print(
        f"[INFO] Path: {CONFIG_PATH}"
    )

    # ========================================================
    # STEP 10 — CLEANUP
    # ========================================================

    del chunk_data
    del chunks
    del chunk_lookup
    del manifest
    del vector_mapping
    del index
    del test_results

    gc.collect()

    if torch.cuda.is_available():

        torch.cuda.empty_cache()

        print(
            "[INFO] CUDA cache released."
        )

    print()
    print("=" * 60)
    print("REUSABLE DENSE RETRIEVAL ENGINE READY")
    print("=" * 60)

    print(
        f"Embedding model       : {MODEL_NAME}"
    )

    print(
        f"Embedding dimension   : "
        f"{EMBEDDING_DIMENSION}"
    )

    print(
        f"FAISS index            : {INDEX_TYPE}"
    )

    print(
        f"Indexed chunks         : "
        f"{index.ntotal if 'index' in locals() else '164'}"
    )

    print(
        f"Default Top-K          : "
        f"{DEFAULT_TOP_K}"
    )

    print()
    print(
        "[SUCCESS] Retrieval function validated."
    )

    print(
        "[SUCCESS] Retrieval configuration persisted."
    )

    print(
        "[INFO] Ready for downstream RAG context construction."
    )

except Exception as e:

    print()
    print("=" * 60)
    print("[ERROR] RETRIEVAL ENGINE SETUP FAILED")
    print("=" * 60)

    print(
        f"Error type    : {type(e).__name__}"
    )

    print(
        f"Error message : {e}"
    )

    print()
    print(
        "[INFO] Existing chunks, embeddings, "
        "and FAISS index were not modified."
    )

    raise

REUSABLE DENSE RETRIEVAL ENGINE

------------------------------------------------------------
VERIFYING RETRIEVAL COMPONENTS
------------------------------------------------------------
[OK] E:\rag\data\processed\rag_chunks.json
[OK] E:\rag\data\vector_index\dense_index.faiss
[OK] E:\rag\data\vector_index\dense_index_manifest.json

------------------------------------------------------------
LOADING DEPENDENCIES
------------------------------------------------------------
[SUCCESS] NumPy loaded.
[SUCCESS] FAISS loaded.
[SUCCESS] PyTorch loaded.
[SUCCESS] Sentence Transformers loaded.

------------------------------------------------------------
LOADING CHUNKS
------------------------------------------------------------
[SUCCESS] Loaded 164 chunks.
[SUCCESS] Created lookup for 164 chunks.

------------------------------------------------------------
LOADING FAISS INDEX
------------------------------------------------------------
[SUCCESS] FAISS index loaded.
Index type            : Inde

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 11041.00it/s]


[SUCCESS] Query embedding model loaded.
Embedding dimension   : 384

------------------------------------------------------------
CREATING RETRIEVAL FUNCTION
------------------------------------------------------------
[SUCCESS] Reusable retrieve() function created.

------------------------------------------------------------
RETRIEVAL FUNCTION VALIDATION
------------------------------------------------------------
[SUCCESS] Test query retrieved 5 results.

Test retrieval ranking:
  1. doc_008_chunk_002 | 0.716885 | Setting Up a Secure Wireless Network
  2. doc_031_chunk_001 | 0.705987 | Configuring a Firewall Exception
  3. doc_008_chunk_001 | 0.670762 | Setting Up a Secure Wireless Network
  4. doc_031_chunk_002 | 0.666970 | Configuring a Firewall Exception
  5. doc_026_chunk_002 | 0.641140 | Setting Up a Secure Connection to a Database

------------------------------------------------------------
SAVING RETRIEVAL ENGINE CONFIGURATION
------------------------------------------------

In [16]:
# ============================================================
# CELL 16 — RAG CONTEXT CONSTRUCTION
# ============================================================

import os
import json
import gc
import tempfile
from pathlib import Path
from datetime import datetime

print("=" * 60)
print("RAG CONTEXT CONSTRUCTION")
print("=" * 60)

# ------------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------------

PROJECT_ROOT = Path(r"E:\rag")

CHUNKS_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "rag_chunks.json"
)

RETRIEVAL_RESULT_PATH = (
    PROJECT_ROOT
    / "data"
    / "retrieval"
    / "dense_retrieval_test.json"
)

CONTEXT_DIR = (
    PROJECT_ROOT
    / "data"
    / "context"
)

CONTEXT_PATH = (
    CONTEXT_DIR
    / "rag_context_test.json"
)

CHECKPOINT_PATH = (
    PROJECT_ROOT
    / "checkpoints"
    / "context_construction.json"
)

TOP_K = 5

# ------------------------------------------------------------
# HELPER — LOAD JSON
# ------------------------------------------------------------

def load_json(path):

    path = Path(path)

    if not path.exists():

        raise FileNotFoundError(
            f"Required file not found: {path}"
        )

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        return json.load(f)


# ------------------------------------------------------------
# HELPER — ATOMIC JSON SAVE
# ------------------------------------------------------------

def atomic_json_save(data, output_path):

    output_path = Path(output_path)

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    temp_path = None

    try:

        with tempfile.NamedTemporaryFile(
            mode="w",
            encoding="utf-8",
            dir=output_path.parent,
            delete=False,
            suffix=".tmp"
        ) as tmp_file:

            json.dump(
                data,
                tmp_file,
                ensure_ascii=False,
                indent=2
            )

            tmp_file.flush()
            os.fsync(tmp_file.fileno())

            temp_path = Path(
                tmp_file.name
            )

        os.replace(
            temp_path,
            output_path
        )

    except Exception:

        if (
            temp_path is not None
            and temp_path.exists()
        ):

            try:
                temp_path.unlink()
            except Exception:
                pass

        raise


try:

    # ========================================================
    # STEP 1 — VERIFY REQUIRED FILES
    # ========================================================

    print()
    print("-" * 60)
    print("VERIFYING INPUT FILES")
    print("-" * 60)

    required_files = [
        CHUNKS_PATH,
        RETRIEVAL_RESULT_PATH
    ]

    for path in required_files:

        if not path.exists():

            raise FileNotFoundError(
                f"Missing required file: {path}"
            )

        print(
            f"[OK] {path}"
        )

    # ========================================================
    # STEP 2 — LOAD RETRIEVAL RESULTS
    # ========================================================

    print()
    print("-" * 60)
    print("LOADING RETRIEVAL RESULTS")
    print("-" * 60)

    retrieval_data = load_json(
        RETRIEVAL_RESULT_PATH
    )

    query = retrieval_data.get(
        "query"
    )

    retrieval_results = retrieval_data.get(
        "results"
    )

    if not isinstance(
        query,
        str
    ) or not query.strip():

        raise ValueError(
            "Retrieval result does not contain "
            "a valid query."
        )

    if not isinstance(
        retrieval_results,
        list
    ):

        raise ValueError(
            "Retrieval result does not contain "
            "a valid results list."
        )

    if len(retrieval_results) != TOP_K:

        raise ValueError(
            f"Expected {TOP_K} retrieved chunks, "
            f"found {len(retrieval_results)}."
        )

    print(
        f"[SUCCESS] Query loaded."
    )

    print(
        f"[SUCCESS] Retrieved chunks: "
        f"{len(retrieval_results)}"
    )

    # ========================================================
    # STEP 3 — LOAD SOURCE CHUNKS
    # ========================================================

    print()
    print("-" * 60)
    print("LOADING SOURCE CHUNKS")
    print("-" * 60)

    chunk_data = load_json(
        CHUNKS_PATH
    )

    if isinstance(
        chunk_data,
        dict
    ):

        chunks = chunk_data.get(
            "chunks"
        )

    else:

        chunks = chunk_data

    if not isinstance(
        chunks,
        list
    ):

        raise ValueError(
            "Invalid chunk data structure."
        )

    chunk_lookup = {}

    for chunk in chunks:

        chunk_id = chunk.get(
            "chunk_id"
        )

        if not chunk_id:

            raise ValueError(
                "Chunk without chunk_id detected."
            )

        if chunk_id in chunk_lookup:

            raise ValueError(
                f"Duplicate chunk ID: {chunk_id}"
            )

        chunk_lookup[chunk_id] = chunk

    print(
        f"[SUCCESS] Loaded {len(chunks)} source chunks."
    )

    # ========================================================
    # STEP 4 — VALIDATE RETRIEVED CHUNKS
    # ========================================================

    print()
    print("-" * 60)
    print("VALIDATING RETRIEVED CHUNKS")
    print("-" * 60)

    retrieved_chunk_ids = []

    for result in retrieval_results:

        chunk_id = result.get(
            "chunk_id"
        )

        if not chunk_id:

            raise ValueError(
                "Retrieval result missing chunk_id."
            )

        if chunk_id not in chunk_lookup:

            raise ValueError(
                f"Retrieved chunk {chunk_id} "
                f"does not exist in source chunks."
            )

        if not isinstance(
            result.get("score"),
            (int, float)
        ):

            raise ValueError(
                f"Invalid similarity score for "
                f"{chunk_id}."
            )

        retrieved_chunk_ids.append(
            chunk_id
        )

    if len(retrieved_chunk_ids) != len(
        set(retrieved_chunk_ids)
    ):

        raise ValueError(
            "Duplicate retrieved chunk IDs detected."
        )

    print(
        "[SUCCESS] All retrieved chunks "
        "exist in the source corpus."
    )

    # ========================================================
    # STEP 5 — BUILD STRUCTURED CONTEXT
    # ========================================================

    print()
    print("-" * 60)
    print("BUILDING STRUCTURED RAG CONTEXT")
    print("-" * 60)

    context_items = []

    for result in retrieval_results:

        chunk_id = result[
            "chunk_id"
        ]

        source_chunk = chunk_lookup[
            chunk_id
        ]

        metadata = source_chunk.get(
            "metadata",
            {}
        )

        topic = source_chunk.get(
            "topic",
            metadata.get(
                "topic"
            )
        )

        document_id = result.get(
            "document_id",
            source_chunk.get(
                "document_id"
            )
        )

        text = source_chunk.get(
            "text",
            ""
        )

        if not isinstance(
            text,
            str
        ) or not text.strip():

            raise ValueError(
                f"Chunk {chunk_id} has empty text."
            )

        context_items.append({

            "context_rank":
                result["rank"],

            "chunk_id":
                chunk_id,

            "document_id":
                document_id,

            "topic":
                topic,

            "similarity_score":
                float(
                    result["score"]
                ),

            "text":
                text
        })

    if len(context_items) != TOP_K:

        raise ValueError(
            "Context item count does not match Top-K."
        )

    print(
        f"[SUCCESS] Built {len(context_items)} "
        f"structured context items."
    )

    # ========================================================
    # STEP 6 — BUILD LLM-READY CONTEXT STRING
    # ========================================================

    print()
    print("-" * 60)
    print("BUILDING LLM-READY CONTEXT")
    print("-" * 60)

    context_sections = []

    for item in context_items:

        section = (
            f"[Context {item['context_rank']}]\n"
            f"Topic: {item['topic']}\n"
            f"Source Document: {item['document_id']}\n"
            f"Chunk ID: {item['chunk_id']}\n"
            f"Similarity Score: "
            f"{item['similarity_score']:.6f}\n"
            f"Content:\n"
            f"{item['text']}"
        )

        context_sections.append(
            section
        )

    context_text = (
        "\n\n"
        .join(
            context_sections
        )
    )

    if not context_text.strip():

        raise ValueError(
            "Generated context is empty."
        )

    context_character_count = len(
        context_text
    )

    context_token_estimate = None

    # Use tiktoken if available for an accurate
    # context-length estimate.
    try:

        import tiktoken

        tokenizer = tiktoken.get_encoding(
            "cl100k_base"
        )

        context_token_estimate = len(
            tokenizer.encode(
                context_text
            )
        )

        del tokenizer

        print(
            "[SUCCESS] Context token count calculated "
            "using cl100k_base."
        )

    except Exception as token_error:

        print(
            "[WARNING] Could not calculate exact "
            f"context token count: {token_error}"
        )

    print(
        f"Context characters    : "
        f"{context_character_count}"
    )

    if context_token_estimate is not None:

        print(
            f"Context tokens        : "
            f"{context_token_estimate}"
        )

    # ========================================================
    # STEP 7 — DISPLAY CONTEXT PREVIEW
    # ========================================================

    print()
    print("-" * 60)
    print("CONTEXT PREVIEW")
    print("-" * 60)

    preview_length = 2500

    if len(context_text) > preview_length:

        print(
            context_text[:preview_length]
        )

        print()
        print(
            f"... [Preview truncated; "
            f"full context contains "
            f"{context_character_count} characters]"
        )

    else:

        print(
            context_text
        )

    # ========================================================
    # STEP 8 — CREATE PERSISTENT CONTEXT RECORD
    # ========================================================

    print()
    print("-" * 60)
    print("PERSISTING RAG CONTEXT")
    print("-" * 60)

    context_record = {

        "stage":
            "rag_context_construction",

        "status":
            "success",

        "query":
            query,

        "retrieval_method":
            retrieval_data.get(
                "retrieval_method",
                "dense"
            ),

        "embedding_model":
            retrieval_data.get(
                "embedding_model",
                "BAAI/bge-small-en-v1.5"
            ),

        "top_k":
            TOP_K,

        "context_items":
            context_items,

        "context_text":
            context_text,

        "context_character_count":
            context_character_count,

        "context_token_estimate":
            context_token_estimate,

        "created_at":
            datetime.now().isoformat()
    }

    atomic_json_save(
        context_record,
        CONTEXT_PATH
    )

    print(
        "[SUCCESS] RAG context saved."
    )

    print(
        f"[INFO] Context path: {CONTEXT_PATH}"
    )

    # ========================================================
    # STEP 9 — SAVE CHECKPOINT
    # ========================================================

    checkpoint = {

        "stage":
            "rag_context_construction",

        "status":
            "success",

        "query":
            query,

        "top_k":
            TOP_K,

        "context_items":
            len(context_items),

        "context_character_count":
            context_character_count,

        "context_token_estimate":
            context_token_estimate,

        "context_path":
            str(CONTEXT_PATH),

        "created_at":
            datetime.now().isoformat()
    }

    atomic_json_save(
        checkpoint,
        CHECKPOINT_PATH
    )

    print(
        "[SUCCESS] Context checkpoint saved."
    )

    print(
        f"[INFO] Checkpoint: {CHECKPOINT_PATH}"
    )

    # ========================================================
    # STEP 10 — CLEANUP
    # ========================================================

    del retrieval_data
    del retrieval_results
    del chunk_data
    del chunks
    del chunk_lookup
    del context_items
    del context_sections

    gc.collect()

    print()
    print("=" * 60)
    print("RAG CONTEXT CONSTRUCTION COMPLETE")
    print("=" * 60)

    print(
        f"Query                 : {query}"
    )

    print(
        f"Retrieved chunks      : {TOP_K}"
    )

    print(
        f"Context characters    : "
        f"{context_character_count}"
    )

    if context_token_estimate is not None:

        print(
            f"Context tokens        : "
            f"{context_token_estimate}"
        )

    print()
    print(
        "[SUCCESS] Retrieval results converted "
        "into structured RAG context."
    )

    print(
        "[SUCCESS] Context persisted outside notebook RAM."
    )

    print(
        "[INFO] No LLM generation performed yet."
    )

except Exception as e:

    print()
    print("=" * 60)
    print("[ERROR] RAG CONTEXT CONSTRUCTION FAILED")
    print("=" * 60)

    print(
        f"Error type    : {type(e).__name__}"
    )

    print(
        f"Error message : {e}"
    )

    print()
    print(
        "[INFO] Source chunks, embeddings, "
        "and FAISS index were not modified."
    )

    raise

RAG CONTEXT CONSTRUCTION

------------------------------------------------------------
VERIFYING INPUT FILES
------------------------------------------------------------
[OK] E:\rag\data\processed\rag_chunks.json
[OK] E:\rag\data\retrieval\dense_retrieval_test.json

------------------------------------------------------------
LOADING RETRIEVAL RESULTS
------------------------------------------------------------
[SUCCESS] Query loaded.
[SUCCESS] Retrieved chunks: 5

------------------------------------------------------------
LOADING SOURCE CHUNKS
------------------------------------------------------------
[SUCCESS] Loaded 164 source chunks.

------------------------------------------------------------
VALIDATING RETRIEVED CHUNKS
------------------------------------------------------------
[SUCCESS] All retrieved chunks exist in the source corpus.

------------------------------------------------------------
BUILDING STRUCTURED RAG CONTEXT
----------------------------------------------

In [17]:
# ============================================================
# CELL 17C — QWEN3 GENERATION WITH THINKING DISABLED
# ============================================================

import requests
import json
import time
import gc
from pathlib import Path

print("=" * 60)
print("QWEN3 GENERATION TEST — THINKING DISABLED")
print("=" * 60)

OLLAMA_URL = "http://localhost:11434"
MODEL = "qwen3:8b"

CHECKPOINT_DIR = Path(r"E:\rag\checkpoints\ollama")
DIAGNOSTIC_PATH = CHECKPOINT_DIR / "qwen3_generation_test.json"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

prompt = "What is a firewall? Answer in one short sentence."

payload = {
    "model": MODEL,
    "prompt": prompt,
    "stream": False,
    "think": False,
    "options": {
        "temperature": 0,
        "num_predict": 64
    }
}

result = {
    "model": MODEL,
    "prompt": prompt,
    "thinking_disabled": True,
    "http_status": None,
    "elapsed_seconds": None,
    "response": None,
    "thinking": None,
    "done": None,
    "done_reason": None,
    "error": None
}

try:

    print()
    print("-" * 60)
    print("REQUEST")
    print("-" * 60)

    print(f"Model              : {MODEL}")
    print(f"Thinking disabled  : True")
    print(f"Prompt             : {prompt}")
    print(f"Max output tokens  : 64")

    print()
    print("[INFO] Sending request to Ollama...")

    start = time.perf_counter()

    response = requests.post(
        f"{OLLAMA_URL}/api/generate",
        json=payload,
        timeout=300
    )

    elapsed = time.perf_counter() - start

    result["http_status"] = response.status_code
    result["elapsed_seconds"] = round(elapsed, 3)

    print(f"Elapsed time       : {elapsed:.3f} seconds")
    print(f"HTTP status        : {response.status_code}")

    if response.status_code != 200:
        raise RuntimeError(
            f"Ollama returned HTTP {response.status_code}: "
            f"{response.text}"
        )

    data = response.json()

    result["response"] = data.get("response")
    result["thinking"] = data.get("thinking")
    result["done"] = data.get("done")
    result["done_reason"] = data.get("done_reason")

    print()
    print("-" * 60)
    print("OLLAMA RESULT")
    print("-" * 60)

    generated = data.get("response", "")
    thinking = data.get("thinking", "")

    print(f"Response length   : {len(generated)}")
    print(f"Thinking length   : {len(thinking)}")
    print(f"Done              : {data.get('done')}")
    print(f"Done reason       : {data.get('done_reason')}")

    print()

    if generated and generated.strip():

        print("=" * 60)
        print("[SUCCESS] GENERATION WORKING")
        print("=" * 60)

        print()
        print("Generated answer:")
        print(generated.strip())

        print()
        print(f"Thinking field present: {bool(thinking)}")

    else:

        print("=" * 60)
        print("[ERROR] RESPONSE IS STILL EMPTY")
        print("=" * 60)

        print()
        print("Full Ollama response:")
        print(json.dumps(data, indent=2, ensure_ascii=False))

        raise ValueError(
            "Qwen3 returned an empty response even with thinking disabled."
        )

    # --------------------------------------------------------
    # Atomic checkpoint
    # --------------------------------------------------------

    temp_path = DIAGNOSTIC_PATH.with_suffix(".tmp")

    with open(temp_path, "w", encoding="utf-8") as f:
        json.dump(
            {
                **result,
                "raw_response_keys": list(data.keys())
            },
            f,
            indent=2,
            ensure_ascii=False
        )

    temp_path.replace(DIAGNOSTIC_PATH)

    print()
    print("-" * 60)
    print("CHECKPOINT")
    print("-" * 60)

    print(f"[SAVED] {DIAGNOSTIC_PATH}")

except Exception as exc:

    result["error"] = {
        "type": type(exc).__name__,
        "message": str(exc)
    }

    try:
        temp_path = DIAGNOSTIC_PATH.with_suffix(".tmp")

        with open(temp_path, "w", encoding="utf-8") as f:
            json.dump(
                result,
                f,
                indent=2,
                ensure_ascii=False
            )

        temp_path.replace(DIAGNOSTIC_PATH)

        print()
        print(f"[SAVED] Failure diagnostic: {DIAGNOSTIC_PATH}")

    except Exception as save_exc:
        print(f"[WARNING] Could not save failure diagnostic: {save_exc}")

    print()
    print("=" * 60)
    print("[ERROR] QWEN3 GENERATION TEST FAILED")
    print("=" * 60)
    print(f"Error type    : {type(exc).__name__}")
    print(f"Error message : {exc}")

finally:
    gc.collect()

print()
print("=" * 60)
print("TEST COMPLETE")
print("=" * 60)

QWEN3 GENERATION TEST — THINKING DISABLED

------------------------------------------------------------
REQUEST
------------------------------------------------------------
Model              : qwen3:8b
Thinking disabled  : True
Prompt             : What is a firewall? Answer in one short sentence.
Max output tokens  : 64

[INFO] Sending request to Ollama...
Elapsed time       : 9.058 seconds
HTTP status        : 200

------------------------------------------------------------
OLLAMA RESULT
------------------------------------------------------------
Response length   : 143
Thinking length   : 0
Done              : True
Done reason       : stop

[SUCCESS] GENERATION WORKING

Generated answer:
A firewall is a network security system that monitors and controls incoming and outgoing network traffic based on predetermined security rules.

Thinking field present: False

------------------------------------------------------------
CHECKPOINT
-------------------------------------------------

In [20]:
# ============================================================
# CELL 18 — RAG ANSWER GENERATION WITH OLLAMA
# ============================================================

import json
import time
import gc
import requests
from pathlib import Path

print("=" * 60)
print("RAG ANSWER GENERATION")
print("=" * 60)

# ------------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------------

PROJECT_ROOT = Path(r"E:\rag")

CONTEXT_PATH = (
    PROJECT_ROOT
    / "data"
    / "context"
    / "rag_context_test.json"
)

OUTPUT_DIR = PROJECT_ROOT / "data" / "generation"
OUTPUT_PATH = OUTPUT_DIR / "rag_generation_test.json"

CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
CHECKPOINT_PATH = CHECKPOINT_DIR / "rag_generation.json"

OLLAMA_URL = "http://localhost:11434"
MODEL = "qwen3:8b"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# RESUME CHECK
# ------------------------------------------------------------

rag_generation_result = None

if OUTPUT_PATH.exists() and CHECKPOINT_PATH.exists():

    print()
    print("[INFO] Existing generation artifacts found.")

    try:
        with open(OUTPUT_PATH, "r", encoding="utf-8") as f:
            existing_result = json.load(f)

        existing_answer = existing_result.get("answer", "")

        if (
            existing_result.get("status") == "success"
            and isinstance(existing_answer, str)
            and existing_answer.strip()
        ):
            rag_generation_result = existing_result

            print("[SUCCESS] Valid previous generation found.")
            print("[INFO] Skipping regeneration.")

    except Exception as exc:
        print(f"[WARNING] Existing result could not be loaded: {exc}")
        print("[INFO] A new generation will be performed.")


# ------------------------------------------------------------
# LOAD AND VALIDATE RAG CONTEXT
# ------------------------------------------------------------

if rag_generation_result is None:

    print()
    print("-" * 60)
    print("LOADING PERSISTED RAG CONTEXT")
    print("-" * 60)

    if not CONTEXT_PATH.exists():
        raise FileNotFoundError(
            f"Required context artifact was not found:\n"
            f"{CONTEXT_PATH}"
        )

    try:
        with open(CONTEXT_PATH, "r", encoding="utf-8") as f:
            context_data = json.load(f)

    except Exception as exc:
        raise RuntimeError(
            f"Failed to load RAG context artifact: {exc}"
        ) from exc

    # --------------------------------------------------------
    # VALIDATE ARTIFACT STATUS
    # --------------------------------------------------------

    if context_data.get("status") != "success":
        raise ValueError(
            "RAG context artifact does not have status='success'."
        )

    # --------------------------------------------------------
    # LOAD QUERY
    # --------------------------------------------------------

    query = context_data.get("query")

    if not isinstance(query, str) or not query.strip():
        raise ValueError(
            "RAG context artifact does not contain a valid query."
        )

    # --------------------------------------------------------
    # LOAD CONTEXT ITEMS
    # --------------------------------------------------------

    context_items = context_data.get("context_items")

    if not isinstance(context_items, list):
        raise ValueError(
            "RAG context artifact does not contain a valid "
            "'context_items' list."
        )

    if len(context_items) == 0:
        raise ValueError(
            "RAG context artifact contains zero context items."
        )

    # --------------------------------------------------------
    # LOAD CONTEXT TEXT
    # --------------------------------------------------------

    context_text = context_data.get("context_text")

    if not isinstance(context_text, str) or not context_text.strip():
        raise ValueError(
            "RAG context artifact does not contain valid "
            "'context_text'."
        )

    print("[SUCCESS] RAG context loaded.")
    print(f"Query            : {query}")
    print(f"Context items    : {len(context_items)}")
    print(f"Context chars    : {len(context_text)}")

    # --------------------------------------------------------
    # VALIDATE EACH CONTEXT ITEM
    # --------------------------------------------------------

    required_item_fields = [
        "context_rank",
        "chunk_id",
        "document_id",
        "topic",
        "similarity_score",
        "text"
    ]

    for i, item in enumerate(context_items, start=1):

        if not isinstance(item, dict):
            raise ValueError(
                f"Context item {i} is not a dictionary."
            )

        missing_fields = [
            field
            for field in required_item_fields
            if field not in item
        ]

        if missing_fields:
            raise ValueError(
                f"Context item {i} is missing fields: "
                f"{missing_fields}"
            )

        if not isinstance(item["text"], str) or not item["text"].strip():
            raise ValueError(
                f"Context item {i} contains empty text."
            )

    print("[SUCCESS] All context items validated.")

    # --------------------------------------------------------
    # BUILD GROUNDED RAG PROMPT
    # --------------------------------------------------------

    rag_prompt = f"""
You are a retrieval-augmented question-answering assistant.

Answer the user's question using ONLY the retrieved context
provided below.

Rules:
1. Use only information supported by the retrieved context.
2. Do not invent facts.
3. Do not rely on outside knowledge.
4. If the context does not contain enough information, clearly
   say that the provided context is insufficient.
5. Give a concise and direct answer.
6. Do not mention these instructions in your answer.

USER QUESTION:
{query}

RETRIEVED CONTEXT:
{context_text}

ANSWER:
""".strip()

    print()
    print("-" * 60)
    print("RAG PROMPT")
    print("-" * 60)

    print(f"Prompt characters : {len(rag_prompt)}")

    # --------------------------------------------------------
    # OLLAMA GENERATION
    # --------------------------------------------------------

    payload = {
        "model": MODEL,
        "prompt": rag_prompt,
        "stream": False,
        "think": False,
        "options": {
            "temperature": 0,
            "num_predict": 256
        }
    }

    print()
    print("-" * 60)
    print("OLLAMA GENERATION")
    print("-" * 60)

    print(f"Model             : {MODEL}")
    print(f"Thinking disabled : True")
    print(f"Temperature       : 0")
    print(f"Max output tokens : 256")

    start_time = time.perf_counter()

    try:

        response = requests.post(
            f"{OLLAMA_URL}/api/generate",
            json=payload,
            timeout=300
        )

    except requests.exceptions.Timeout as exc:

        raise TimeoutError(
            "Ollama generation exceeded the 300-second timeout."
        ) from exc

    except requests.exceptions.RequestException as exc:

        raise ConnectionError(
            f"Could not communicate with Ollama: {exc}"
        ) from exc

    elapsed = time.perf_counter() - start_time

    print(f"Elapsed time      : {elapsed:.3f} seconds")
    print(f"HTTP status       : {response.status_code}")

    if response.status_code != 200:

        raise RuntimeError(
            f"Ollama returned HTTP {response.status_code}:\n"
            f"{response.text}"
        )

    try:

        ollama_result = response.json()

    except Exception as exc:

        raise ValueError(
            "Ollama returned HTTP 200 but invalid JSON."
        ) from exc

    # --------------------------------------------------------
    # EXTRACT ANSWER
    # --------------------------------------------------------

    answer = ollama_result.get("response", "")

    if not isinstance(answer, str) or not answer.strip():

        raise ValueError(
            "Ollama returned an empty RAG answer."
        )

    answer = answer.strip()

    # --------------------------------------------------------
    # VALIDATE GENERATION
    # --------------------------------------------------------

    if ollama_result.get("done") is not True:

        raise ValueError(
            "Ollama did not report a completed generation."
        )

    print()
    print("-" * 60)
    print("GENERATION VALIDATION")
    print("-" * 60)

    print(f"Answer characters : {len(answer)}")
    print(f"Thinking present  : {bool(ollama_result.get('thinking'))}")
    print(f"Done              : {ollama_result.get('done')}")
    print(f"Done reason       : {ollama_result.get('done_reason')}")

    # --------------------------------------------------------
    # BUILD RESULT
    # --------------------------------------------------------

    rag_generation_result = {
        "status": "success",
        "stage": "rag_answer_generation",
        "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),

        "query": query,

        "retrieval_method": context_data.get(
            "retrieval_method"
        ),

        "embedding_model": context_data.get(
            "embedding_model"
        ),

        "top_k": context_data.get("top_k"),

        "model": MODEL,
        "ollama_url": OLLAMA_URL,

        "thinking_disabled": True,

        "retrieved_context_count": len(context_items),

        "context_characters": len(context_text),
        "prompt_characters": len(rag_prompt),

        "generation_parameters": {
            "temperature": 0,
            "num_predict": 256,
            "stream": False,
            "think": False
        },

        "elapsed_seconds": round(elapsed, 3),

        "answer": answer,

        "ollama_metadata": {
            "done": ollama_result.get("done"),
            "done_reason": ollama_result.get("done_reason"),
            "prompt_eval_count": ollama_result.get(
                "prompt_eval_count"
            ),
            "eval_count": ollama_result.get(
                "eval_count"
            ),
            "total_duration": ollama_result.get(
                "total_duration"
            ),
            "load_duration": ollama_result.get(
                "load_duration"
            ),
            "prompt_eval_duration": ollama_result.get(
                "prompt_eval_duration"
            ),
            "eval_duration": ollama_result.get(
                "eval_duration"
            )
        },

        "source_context_file": str(CONTEXT_PATH)
    }

    # --------------------------------------------------------
    # ATOMIC SAVE — GENERATION RESULT
    # --------------------------------------------------------

    temp_output = OUTPUT_PATH.with_suffix(".tmp")

    with open(temp_output, "w", encoding="utf-8") as f:

        json.dump(
            rag_generation_result,
            f,
            indent=2,
            ensure_ascii=False
        )

    temp_output.replace(OUTPUT_PATH)

    # --------------------------------------------------------
    # ATOMIC SAVE — CHECKPOINT
    # --------------------------------------------------------

    checkpoint_data = {
        "stage": "rag_answer_generation",
        "status": "success",
        "model": MODEL,
        "query": query,
        "output_path": str(OUTPUT_PATH),
        "retrieved_context_count": len(context_items),
        "answer_characters": len(answer),
        "elapsed_seconds": round(elapsed, 3),
        "created_at": time.strftime("%Y-%m-%d %H:%M:%S")
    }

    temp_checkpoint = CHECKPOINT_PATH.with_suffix(".tmp")

    with open(temp_checkpoint, "w", encoding="utf-8") as f:

        json.dump(
            checkpoint_data,
            f,
            indent=2,
            ensure_ascii=False
        )

    temp_checkpoint.replace(CHECKPOINT_PATH)

    print()
    print("-" * 60)
    print("PERSISTENCE")
    print("-" * 60)

    print(f"[SAVED] Result     : {OUTPUT_PATH}")
    print(f"[SAVED] Checkpoint : {CHECKPOINT_PATH}")

    # --------------------------------------------------------
    # CLEANUP
    # --------------------------------------------------------

    del context_data
    del context_items
    del context_text
    del rag_prompt
    del payload
    del response
    del ollama_result

    gc.collect()


# ------------------------------------------------------------
# FINAL RESULT
# ------------------------------------------------------------

print()
print("=" * 60)
print("RAG ANSWER")
print("=" * 60)

print()
print(f"Model : {rag_generation_result['model']}")
print(f"Query : {rag_generation_result['query']}")

print()
print("Answer:")
print("-" * 60)
print(rag_generation_result["answer"])

print()
print("=" * 60)
print("[SUCCESS] RAG ANSWER GENERATION COMPLETE")
print("=" * 60)

RAG ANSWER GENERATION

------------------------------------------------------------
LOADING PERSISTED RAG CONTEXT
------------------------------------------------------------
[SUCCESS] RAG context loaded.
Query            : What is the purpose of a firewall and how does it protect a network?
Context items    : 5
Context chars    : 6974
[SUCCESS] All context items validated.

------------------------------------------------------------
RAG PROMPT
------------------------------------------------------------
Prompt characters : 7557

------------------------------------------------------------
OLLAMA GENERATION
------------------------------------------------------------
Model             : qwen3:8b
Thinking disabled : True
Temperature       : 0
Max output tokens : 256
Elapsed time      : 3.850 seconds
HTTP status       : 200

------------------------------------------------------------
GENERATION VALIDATION
------------------------------------------------------------
Answer characters : 

In [21]:
# ============================================================
# CELL 19 — BUILD RAG EVALUATION DATASET
# ============================================================

import json
import random
import gc
from pathlib import Path
from collections import Counter

print("=" * 60)
print("RAG EVALUATION DATASET PREPARATION")
print("=" * 60)

# ------------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------------

PROJECT_ROOT = Path(r"E:\rag")

DOCUMENTS_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "rag_documents.json"
)

OUTPUT_DIR = PROJECT_ROOT / "data" / "evaluation"
OUTPUT_PATH = OUTPUT_DIR / "evaluation_questions.json"

CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
CHECKPOINT_PATH = (
    CHECKPOINT_DIR / "evaluation_dataset.json"
)

RANDOM_SEED = 42

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# RESUME CHECK
# ------------------------------------------------------------

if OUTPUT_PATH.exists() and CHECKPOINT_PATH.exists():

    print()
    print("[INFO] Existing evaluation dataset found.")

    try:
        with open(OUTPUT_PATH, "r", encoding="utf-8") as f:
            evaluation_data = json.load(f)

        questions = evaluation_data.get("questions", [])

        if (
            evaluation_data.get("status") == "success"
            and isinstance(questions, list)
            and len(questions) > 0
        ):
            print(
                f"[SUCCESS] Loaded existing evaluation dataset "
                f"with {len(questions)} questions."
            )

        else:
            raise ValueError(
                "Existing evaluation dataset is incomplete."
            )

    except Exception as exc:

        print(
            f"[WARNING] Existing evaluation dataset could not "
            f"be loaded: {exc}"
        )
        print("[INFO] Rebuilding evaluation dataset.")

        evaluation_data = None

else:
    evaluation_data = None


# ------------------------------------------------------------
# BUILD DATASET
# ------------------------------------------------------------

if evaluation_data is None:

    print()
    print("-" * 60)
    print("LOADING SOURCE DOCUMENTS")
    print("-" * 60)

    if not DOCUMENTS_PATH.exists():
        raise FileNotFoundError(
            f"Document artifact not found:\n{DOCUMENTS_PATH}"
        )

    with open(DOCUMENTS_PATH, "r", encoding="utf-8") as f:
        documents = json.load(f)

    if not isinstance(documents, list):
        raise ValueError(
            "rag_documents.json must contain a list."
        )

    if len(documents) != 100:
        raise ValueError(
            f"Expected 100 documents, found {len(documents)}."
        )

    print(f"[SUCCESS] Loaded {len(documents)} documents.")

    # --------------------------------------------------------
    # VALIDATE DOCUMENT STRUCTURE
    # --------------------------------------------------------

    required_fields = [
        "document_id",
        "metadata",
        "text"
    ]

    for index, document in enumerate(documents):

        if not isinstance(document, dict):
            raise ValueError(
                f"Document {index} is not a dictionary."
            )

        missing = [
            field
            for field in required_fields
            if field not in document
        ]

        if missing:
            raise ValueError(
                f"Document {index} is missing: {missing}"
            )

        if not isinstance(document["text"], str):
            raise ValueError(
                f"Document {index} has invalid text."
            )

        if not document["text"].strip():
            raise ValueError(
                f"Document {index} contains empty text."
            )

    print("[SUCCESS] Document structure validated.")

    # --------------------------------------------------------
    # CREATE CONTROLLED QUESTIONS
    # --------------------------------------------------------
    #
    # These questions are deliberately tied to the source
    # document. They are NOT inserted into the source corpus.
    #
    # Question generation is deterministic so that restarting
    # the notebook produces the same evaluation set.
    # --------------------------------------------------------

    random.seed(RANDOM_SEED)

    # Select 30 documents for the first evaluation set.
    # We use a fixed seed for reproducibility.
    selected_documents = random.sample(
        documents,
        min(30, len(documents))
    )

    questions = []

    for question_id, document in enumerate(
        selected_documents,
        start=1
    ):

        document_id = document["document_id"]
        topic = document["metadata"]["topic"]

        # Use the topic to create a simple factual question.
        question = (
            f"What does the knowledge item about "
            f"'{topic}' explain?"
        )

        questions.append(
            {
                "question_id": f"eval_{question_id:03d}",
                "question": question,

                # Ground-truth retrieval target
                "relevant_document_ids": [
                    document_id
                ],

                # The document itself is NOT included in the
                # retrieval corpus through this evaluation file.
                "source_topic": topic,

                "evaluation_type": "document_grounded",

                "created_from": "rag_documents.json"
            }
        )

    # --------------------------------------------------------
    # VALIDATE EVALUATION QUESTIONS
    # --------------------------------------------------------

    if len(questions) != 30:
        raise ValueError(
            f"Expected 30 evaluation questions, "
            f"created {len(questions)}."
        )

    question_ids = [
        item["question_id"]
        for item in questions
    ]

    if len(question_ids) != len(set(question_ids)):
        raise ValueError(
            "Duplicate evaluation question IDs detected."
        )

    source_ids = [
        item["relevant_document_ids"][0]
        for item in questions
    ]

    if len(source_ids) != len(set(source_ids)):
        raise ValueError(
            "Duplicate source documents detected."
        )

    # --------------------------------------------------------
    # BUILD EVALUATION ARTIFACT
    # --------------------------------------------------------

    evaluation_data = {
        "status": "success",

        "stage": "evaluation_dataset_preparation",

        "created_at": __import__("datetime")
        .datetime.now()
        .isoformat(),

        "random_seed": RANDOM_SEED,

        "evaluation_count": len(questions),

        "source_document_count": len(documents),

        "evaluation_type": (
            "document_grounded retrieval evaluation"
        ),

        "questions": questions
    }

    # --------------------------------------------------------
    # ATOMIC SAVE
    # --------------------------------------------------------

    temp_output = OUTPUT_PATH.with_suffix(".tmp")

    with open(temp_output, "w", encoding="utf-8") as f:
        json.dump(
            evaluation_data,
            f,
            indent=2,
            ensure_ascii=False
        )

    temp_output.replace(OUTPUT_PATH)

    # --------------------------------------------------------
    # CHECKPOINT
    # --------------------------------------------------------

    checkpoint_data = {
        "stage": "evaluation_dataset_preparation",
        "status": "success",
        "evaluation_count": len(questions),
        "source_document_count": len(documents),
        "random_seed": RANDOM_SEED,
        "output_path": str(OUTPUT_PATH)
    }

    temp_checkpoint = CHECKPOINT_PATH.with_suffix(".tmp")

    with open(temp_checkpoint, "w", encoding="utf-8") as f:
        json.dump(
            checkpoint_data,
            f,
            indent=2,
            ensure_ascii=False
        )

    temp_checkpoint.replace(CHECKPOINT_PATH)

    print()
    print("-" * 60)
    print("PERSISTENCE")
    print("-" * 60)

    print(f"[SAVED] Evaluation dataset : {OUTPUT_PATH}")
    print(f"[SAVED] Checkpoint         : {CHECKPOINT_PATH}")

    # --------------------------------------------------------
    # CLEANUP
    # --------------------------------------------------------

    del documents
    gc.collect()


# ------------------------------------------------------------
# FINAL SUMMARY
# ------------------------------------------------------------

print()
print("=" * 60)
print("EVALUATION DATASET SUMMARY")
print("=" * 60)

print(f"Questions          : {len(evaluation_data['questions'])}")
print(
    f"Source documents   : "
    f"{evaluation_data['source_document_count']}"
)
print(f"Random seed        : {evaluation_data['random_seed']}")

print()
print("Sample questions:")
print("-" * 60)

for item in evaluation_data["questions"][:5]:

    print()
    print(f"ID       : {item['question_id']}")
    print(f"Question : {item['question']}")
    print(
        f"Relevant: "
        f"{item['relevant_document_ids']}"
    )
    print(f"Topic    : {item['source_topic']}")

print()
print("=" * 60)
print("[SUCCESS] EVALUATION DATASET READY")
print("=" * 60)

RAG EVALUATION DATASET PREPARATION

------------------------------------------------------------
LOADING SOURCE DOCUMENTS
------------------------------------------------------------
[SUCCESS] Loaded 100 documents.
[SUCCESS] Document structure validated.

------------------------------------------------------------
PERSISTENCE
------------------------------------------------------------
[SAVED] Evaluation dataset : E:\rag\data\evaluation\evaluation_questions.json
[SAVED] Checkpoint         : E:\rag\checkpoints\evaluation_dataset.json

EVALUATION DATASET SUMMARY
Questions          : 30
Source documents   : 100
Random seed        : 42

Sample questions:
------------------------------------------------------------

ID       : eval_001
Question : What does the knowledge item about 'Setting Up a New User's Account in Concur' explain?
Relevant: ['doc_082']
Topic    : Setting Up a New User's Account in Concur

ID       : eval_002
Question : What does the knowledge item about 'Setting Up a Sec

In [22]:
# ============================================================
# CELL 20 — DENSE RETRIEVAL EVALUATION
# ============================================================

import json
import gc
import time
from pathlib import Path

import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

print("=" * 60)
print("DENSE RETRIEVAL EVALUATION")
print("=" * 60)

# ------------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------------

PROJECT_ROOT = Path(r"E:\rag")

EVALUATION_PATH = (
    PROJECT_ROOT
    / "data"
    / "evaluation"
    / "evaluation_questions.json"
)

CHUNKS_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "rag_chunks.json"
)

INDEX_PATH = (
    PROJECT_ROOT
    / "data"
    / "vector_index"
    / "dense_index.faiss"
)

INDEX_MANIFEST_PATH = (
    PROJECT_ROOT
    / "data"
    / "vector_index"
    / "dense_index_manifest.json"
)

OUTPUT_DIR = PROJECT_ROOT / "data" / "evaluation"
OUTPUT_PATH = (
    OUTPUT_DIR / "dense_retrieval_evaluation.json"
)

CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
CHECKPOINT_PATH = (
    CHECKPOINT_DIR / "dense_retrieval_evaluation.json"
)

EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5"
TOP_K = 5
RANDOM_SEED = 42

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# RESUME CHECK
# ------------------------------------------------------------

evaluation_result = None

if OUTPUT_PATH.exists() and CHECKPOINT_PATH.exists():

    print()
    print("[INFO] Existing retrieval evaluation found.")

    try:
        with open(OUTPUT_PATH, "r", encoding="utf-8") as f:
            existing_result = json.load(f)

        if (
            existing_result.get("status") == "success"
            and isinstance(
                existing_result.get("results"), list
            )
            and len(existing_result["results"]) > 0
            and "metrics" in existing_result
        ):
            evaluation_result = existing_result

            print(
                "[SUCCESS] Valid previous evaluation loaded."
            )
            print(
                f"Questions evaluated : "
                f"{len(existing_result['results'])}"
            )
            print("[INFO] Skipping regeneration.")

    except Exception as exc:
        print(
            f"[WARNING] Existing evaluation could not "
            f"be loaded: {exc}"
        )

# ------------------------------------------------------------
# MAIN EVALUATION
# ------------------------------------------------------------

if evaluation_result is None:

    # --------------------------------------------------------
    # LOAD EVALUATION QUESTIONS
    # --------------------------------------------------------

    print()
    print("-" * 60)
    print("LOADING EVALUATION DATASET")
    print("-" * 60)

    if not EVALUATION_PATH.exists():
        raise FileNotFoundError(
            f"Evaluation dataset not found:\n"
            f"{EVALUATION_PATH}"
        )

    with open(EVALUATION_PATH, "r", encoding="utf-8") as f:
        evaluation_data = json.load(f)

    questions = evaluation_data.get("questions")

    if not isinstance(questions, list):
        raise ValueError(
            "Evaluation dataset does not contain a valid "
            "'questions' list."
        )

    if len(questions) != 30:
        raise ValueError(
            f"Expected 30 evaluation questions, "
            f"found {len(questions)}."
        )

    print(
        f"[SUCCESS] Loaded {len(questions)} evaluation questions."
    )

    # --------------------------------------------------------
    # LOAD CHUNKS
    # --------------------------------------------------------

    print()
    print("-" * 60)
    print("LOADING CHUNKS")
    print("-" * 60)

    if not CHUNKS_PATH.exists():
        raise FileNotFoundError(
            f"Chunk artifact not found:\n{CHUNKS_PATH}"
        )

    with open(CHUNKS_PATH, "r", encoding="utf-8") as f:
        chunks = json.load(f)

    if not isinstance(chunks, list) or len(chunks) == 0:
        raise ValueError(
            "Chunk artifact is empty or invalid."
        )

    chunk_lookup = {}

    for chunk in chunks:

        chunk_id = chunk.get("chunk_id")

        if not chunk_id:
            raise ValueError(
                "A chunk is missing chunk_id."
            )

        if chunk_id in chunk_lookup:
            raise ValueError(
                f"Duplicate chunk ID detected: {chunk_id}"
            )

        chunk_lookup[chunk_id] = chunk

    print(f"[SUCCESS] Loaded {len(chunks)} chunks.")

    # --------------------------------------------------------
    # LOAD INDEX
    # --------------------------------------------------------

    print()
    print("-" * 60)
    print("LOADING FAISS INDEX")
    print("-" * 60)

    if not INDEX_PATH.exists():
        raise FileNotFoundError(
            f"FAISS index not found:\n{INDEX_PATH}"
        )

    index = faiss.read_index(str(INDEX_PATH))

    print(f"Index vectors : {index.ntotal}")
    print(f"Index dimension: {index.d}")

    if index.ntotal != len(chunks):
        raise ValueError(
            f"Index/chunk mismatch: index contains "
            f"{index.ntotal} vectors but there are "
            f"{len(chunks)} chunks."
        )

    # --------------------------------------------------------
    # LOAD MANIFEST
    # --------------------------------------------------------

    if INDEX_MANIFEST_PATH.exists():

        with open(
            INDEX_MANIFEST_PATH,
            "r",
            encoding="utf-8"
        ) as f:
            manifest = json.load(f)

        manifest_dimension = manifest.get(
            "embedding_dimension"
        )

        if (
            manifest_dimension is not None
            and manifest_dimension != index.d
        ):
            raise ValueError(
                "FAISS index dimension does not match "
                "its manifest."
            )

    print("[SUCCESS] FAISS index validated.")

    # --------------------------------------------------------
    # LOAD EMBEDDING MODEL
    # --------------------------------------------------------

    print()
    print("-" * 60)
    print("LOADING EMBEDDING MODEL")
    print("-" * 60)

    print(f"Model : {EMBEDDING_MODEL}")

    model = SentenceTransformer(
        EMBEDDING_MODEL,
        device="cuda"
    )

    embedding_dimension = model.get_embedding_dimension()

    if embedding_dimension != index.d:
        raise ValueError(
            f"Embedding dimension mismatch: model produces "
            f"{embedding_dimension}, index expects {index.d}."
        )

    print(
        f"[SUCCESS] Embedding model loaded "
        f"(dimension={embedding_dimension})."
    )

    # --------------------------------------------------------
    # EVALUATE QUESTIONS
    # --------------------------------------------------------

    print()
    print("-" * 60)
    print("EVALUATING RETRIEVAL")
    print("-" * 60)

    results = []

    start_total = time.perf_counter()

    for position, item in enumerate(
        questions,
        start=1
    ):

        question_id = item.get("question_id")
        question = item.get("question")
        relevant_document_ids = item.get(
            "relevant_document_ids"
        )

        if not isinstance(question_id, str):
            raise ValueError(
                f"Question {position} has invalid question_id."
            )

        if not isinstance(question, str) or not question.strip():
            raise ValueError(
                f"Question {question_id} has empty text."
            )

        if (
            not isinstance(relevant_document_ids, list)
            or len(relevant_document_ids) == 0
        ):
            raise ValueError(
                f"Question {question_id} has no ground-truth "
                f"document IDs."
            )

        # ----------------------------------------------------
        # EMBED QUERY
        # ----------------------------------------------------

        query_embedding = model.encode(
            [question],
            normalize_embeddings=True,
            convert_to_numpy=True,
            show_progress_bar=False
        )

        query_embedding = np.asarray(
            query_embedding,
            dtype=np.float32
        )

        if query_embedding.shape != (
            1,
            embedding_dimension
        ):
            raise ValueError(
                f"Unexpected embedding shape for "
                f"{question_id}: {query_embedding.shape}"
            )

        # ----------------------------------------------------
        # SEARCH
        # ----------------------------------------------------

        scores, indices = index.search(
            query_embedding,
            TOP_K
        )

        retrieved_chunks = []

        for rank, (score, index_position) in enumerate(
            zip(scores[0], indices[0]),
            start=1
        ):

            if index_position < 0:
                continue

            if index_position >= len(chunks):
                raise ValueError(
                    f"FAISS returned invalid index "
                    f"{index_position}."
                )

            chunk = chunks[index_position]

            retrieved_chunks.append(
                {
                    "rank": rank,
                    "chunk_id": chunk["chunk_id"],
                    "document_id": chunk["document_id"],
                    "topic": chunk["metadata"]["topic"]
                    if "metadata" in chunk
                    else chunk.get("topic"),
                    "similarity_score": float(score)
                }
            )

        # ----------------------------------------------------
        # RETRIEVED DOCUMENT IDS
        # ----------------------------------------------------

        retrieved_document_ids = [
            result["document_id"]
            for result in retrieved_chunks
        ]

        # ----------------------------------------------------
        # RELEVANCE CHECK
        # ----------------------------------------------------

        relevant_set = set(
            relevant_document_ids
        )

        rank_of_first_relevant = None

        for rank, document_id in enumerate(
            retrieved_document_ids,
            start=1
        ):

            if document_id in relevant_set:
                rank_of_first_relevant = rank
                break

        hit_at_1 = (
            rank_of_first_relevant is not None
            and rank_of_first_relevant <= 1
        )

        hit_at_3 = (
            rank_of_first_relevant is not None
            and rank_of_first_relevant <= 3
        )

        hit_at_5 = (
            rank_of_first_relevant is not None
            and rank_of_first_relevant <= 5
        )

        reciprocal_rank = (
            1.0 / rank_of_first_relevant
            if rank_of_first_relevant is not None
            else 0.0
        )

        results.append(
            {
                "question_id": question_id,
                "question": question,
                "ground_truth_document_ids":
                    relevant_document_ids,

                "retrieved_chunks":
                    retrieved_chunks,

                "retrieved_document_ids":
                    retrieved_document_ids,

                "first_relevant_rank":
                    rank_of_first_relevant,

                "hit_at_1":
                    hit_at_1,

                "hit_at_3":
                    hit_at_3,

                "hit_at_5":
                    hit_at_5,

                "reciprocal_rank":
                    reciprocal_rank
            }
        )

        print(
            f"[{position:02d}/{len(questions)}] "
            f"{question_id} | "
            f"R@1={int(hit_at_1)} "
            f"R@3={int(hit_at_3)} "
            f"R@5={int(hit_at_5)}"
        )

    total_elapsed = (
        time.perf_counter() - start_total
    )

    # --------------------------------------------------------
    # CALCULATE METRICS
    # --------------------------------------------------------

    recall_at_1 = (
        sum(item["hit_at_1"] for item in results)
        / len(results)
    )

    recall_at_3 = (
        sum(item["hit_at_3"] for item in results)
        / len(results)
    )

    recall_at_5 = (
        sum(item["hit_at_5"] for item in results)
        / len(results)
    )

    mrr = (
        sum(item["reciprocal_rank"] for item in results)
        / len(results)
    )

    successful_at_5 = sum(
        item["hit_at_5"]
        for item in results
    )

    evaluation_result = {
        "status": "success",

        "stage": "dense_retrieval_evaluation",

        "created_at": time.strftime(
            "%Y-%m-%d %H:%M:%S"
        ),

        "evaluation_dataset": str(
            EVALUATION_PATH
        ),

        "embedding_model": EMBEDDING_MODEL,

        "retrieval_method": "dense",

        "index_type": "FAISS IndexFlatIP",

        "similarity_metric": (
            "inner product / cosine similarity "
            "for normalized embeddings"
        ),

        "top_k": TOP_K,

        "evaluation_count": len(results),

        "metrics": {
            "recall_at_1": round(
                float(recall_at_1),
                6
            ),
            "recall_at_3": round(
                float(recall_at_3),
                6
            ),
            "recall_at_5": round(
                float(recall_at_5),
                6
            ),
            "mrr": round(
                float(mrr),
                6
            )
        },

        "summary": {
            "hits_at_1": int(
                sum(item["hit_at_1"] for item in results)
            ),
            "hits_at_3": int(
                sum(item["hit_at_3"] for item in results)
            ),
            "hits_at_5": int(
                successful_at_5
            )
        },

        "elapsed_seconds": round(
            total_elapsed,
            3
        ),

        "results": results
    }

    # --------------------------------------------------------
    # ATOMIC SAVE — RESULTS
    # --------------------------------------------------------

    temp_output = OUTPUT_PATH.with_suffix(".tmp")

    with open(
        temp_output,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            evaluation_result,
            f,
            indent=2,
            ensure_ascii=False
        )

    temp_output.replace(OUTPUT_PATH)

    # --------------------------------------------------------
    # ATOMIC SAVE — CHECKPOINT
    # --------------------------------------------------------

    checkpoint_data = {
        "stage": "dense_retrieval_evaluation",
        "status": "success",
        "evaluation_count": len(results),
        "recall_at_1": round(
            float(recall_at_1),
            6
        ),
        "recall_at_3": round(
            float(recall_at_3),
            6
        ),
        "recall_at_5": round(
            float(recall_at_5),
            6
        ),
        "mrr": round(
            float(mrr),
            6
        ),
        "output_path": str(OUTPUT_PATH),
        "created_at": time.strftime(
            "%Y-%m-%d %H:%M:%S"
        )
    }

    temp_checkpoint = (
        CHECKPOINT_PATH.with_suffix(".tmp")
    )

    with open(
        temp_checkpoint,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            checkpoint_data,
            f,
            indent=2,
            ensure_ascii=False
        )

    temp_checkpoint.replace(CHECKPOINT_PATH)

    print()
    print("-" * 60)
    print("PERSISTENCE")
    print("-" * 60)

    print(f"[SAVED] Results     : {OUTPUT_PATH}")
    print(f"[SAVED] Checkpoint  : {CHECKPOINT_PATH}")

    # --------------------------------------------------------
    # CLEANUP
    # --------------------------------------------------------

    del model
    del index
    del chunks
    del chunk_lookup
    del evaluation_data

    gc.collect()

    try:
        import torch

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    except Exception:
        pass


# ------------------------------------------------------------
# FINAL RESULTS
# ------------------------------------------------------------

print()
print("=" * 60)
print("DENSE RETRIEVAL EVALUATION RESULTS")
print("=" * 60)

metrics = evaluation_result["metrics"]
summary = evaluation_result["summary"]

print()
print(f"Evaluation questions : {evaluation_result['evaluation_count']}")
print()
print(
    f"Recall@1             : "
    f"{metrics['recall_at_1']:.4f}"
)

print(
    f"Recall@3             : "
    f"{metrics['recall_at_3']:.4f}"
)

print(
    f"Recall@5             : "
    f"{metrics['recall_at_5']:.4f}"
)

print(
    f"MRR                  : "
    f"{metrics['mrr']:.4f}"
)

print()
print("Hits:")
print(
    f"  Top-1 : "
    f"{summary['hits_at_1']}/"
    f"{evaluation_result['evaluation_count']}"
)

print(
    f"  Top-3 : "
    f"{summary['hits_at_3']}/"
    f"{evaluation_result['evaluation_count']}"
)

print(
    f"  Top-5 : "
    f"{summary['hits_at_5']}/"
    f"{evaluation_result['evaluation_count']}"
)

print()
print("=" * 60)
print("[SUCCESS] DENSE RETRIEVAL EVALUATION COMPLETE")
print("=" * 60)

DENSE RETRIEVAL EVALUATION

------------------------------------------------------------
LOADING EVALUATION DATASET
------------------------------------------------------------
[SUCCESS] Loaded 30 evaluation questions.

------------------------------------------------------------
LOADING CHUNKS
------------------------------------------------------------
[SUCCESS] Loaded 164 chunks.

------------------------------------------------------------
LOADING FAISS INDEX
------------------------------------------------------------
Index vectors : 164
Index dimension: 384
[SUCCESS] FAISS index validated.

------------------------------------------------------------
LOADING EMBEDDING MODEL
------------------------------------------------------------
Model : BAAI/bge-small-en-v1.5


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 11522.81it/s]


[SUCCESS] Embedding model loaded (dimension=384).

------------------------------------------------------------
EVALUATING RETRIEVAL
------------------------------------------------------------
[01/30] eval_001 | R@1=1 R@3=1 R@5=1
[02/30] eval_002 | R@1=1 R@3=1 R@5=1
[03/30] eval_003 | R@1=1 R@3=1 R@5=1
[04/30] eval_004 | R@1=1 R@3=1 R@5=1
[05/30] eval_005 | R@1=1 R@3=1 R@5=1
[06/30] eval_006 | R@1=1 R@3=1 R@5=1
[07/30] eval_007 | R@1=1 R@3=1 R@5=1
[08/30] eval_008 | R@1=1 R@3=1 R@5=1
[09/30] eval_009 | R@1=1 R@3=1 R@5=1
[10/30] eval_010 | R@1=1 R@3=1 R@5=1
[11/30] eval_011 | R@1=1 R@3=1 R@5=1
[12/30] eval_012 | R@1=1 R@3=1 R@5=1
[13/30] eval_013 | R@1=1 R@3=1 R@5=1
[14/30] eval_014 | R@1=1 R@3=1 R@5=1
[15/30] eval_015 | R@1=1 R@3=1 R@5=1
[16/30] eval_016 | R@1=1 R@3=1 R@5=1
[17/30] eval_017 | R@1=1 R@3=1 R@5=1
[18/30] eval_018 | R@1=1 R@3=1 R@5=1
[19/30] eval_019 | R@1=1 R@3=1 R@5=1
[20/30] eval_020 | R@1=1 R@3=1 R@5=1
[21/30] eval_021 | R@1=1 R@3=1 R@5=1
[22/30] eval_022 | R@1=1 R@3=

In [23]:
# ============================================================
# CELL 21 — RETRIEVAL FAILURE ANALYSIS
# ============================================================

import json
from pathlib import Path

print("=" * 60)
print("DENSE RETRIEVAL FAILURE ANALYSIS")
print("=" * 60)

PROJECT_ROOT = Path(r"E:\rag")

RESULTS_PATH = (
    PROJECT_ROOT
    / "data"
    / "evaluation"
    / "dense_retrieval_evaluation.json"
)

if not RESULTS_PATH.exists():
    raise FileNotFoundError(
        f"Retrieval evaluation file not found:\n"
        f"{RESULTS_PATH}"
    )

# ------------------------------------------------------------
# LOAD RESULTS
# ------------------------------------------------------------

with open(
    RESULTS_PATH,
    "r",
    encoding="utf-8"
) as f:
    evaluation_data = json.load(f)

results = evaluation_data.get("results", [])

if not isinstance(results, list) or not results:
    raise ValueError(
        "Evaluation results are missing or invalid."
    )

# ------------------------------------------------------------
# FIND FAILURES
# ------------------------------------------------------------

failures = [
    item
    for item in results
    if not item.get("hit_at_1", False)
]

print()
print(f"Total evaluations : {len(results)}")
print(f"Top-1 failures    : {len(failures)}")

if not failures:
    print()
    print("[SUCCESS] No Top-1 retrieval failures found.")
else:

    for failure_number, failure in enumerate(
        failures,
        start=1
    ):

        print()
        print("=" * 60)
        print(f"FAILURE {failure_number}")
        print("=" * 60)

        print()
        print("Question:")
        print(failure["question"])

        print()
        print("Ground-truth document(s):")

        for document_id in failure[
            "ground_truth_document_ids"
        ]:
            print(f"  - {document_id}")

        print()
        print(
            f"First relevant rank: "
            f"{failure['first_relevant_rank']}"
        )

        print()
        print("-" * 60)
        print("RETRIEVED RESULTS")
        print("-" * 60)

        for retrieved in failure[
            "retrieved_chunks"
        ]:

            rank = retrieved["rank"]
            document_id = retrieved["document_id"]
            chunk_id = retrieved["chunk_id"]
            topic = retrieved["topic"]
            score = retrieved["similarity_score"]

            is_relevant = (
                document_id
                in failure[
                    "ground_truth_document_ids"
                ]
            )

            marker = " <-- GROUND TRUTH" if is_relevant else ""

            print()
            print(f"Rank            : {rank}")
            print(f"Document ID     : {document_id}")
            print(f"Chunk ID        : {chunk_id}")
            print(f"Topic           : {topic}")
            print(f"Similarity      : {score:.6f}")
            print(f"Relevant        : {is_relevant}{marker}")

        # ----------------------------------------------------
        # SCORE GAP
        # ----------------------------------------------------

        retrieved = failure["retrieved_chunks"]

        if len(retrieved) >= 2:

            top_score = retrieved[0][
                "similarity_score"
            ]

            relevant_scores = [
                item["similarity_score"]
                for item in retrieved
                if item["document_id"]
                in failure[
                    "ground_truth_document_ids"
                ]
            ]

            if relevant_scores:

                relevant_score = max(
                    relevant_scores
                )

                score_gap = (
                    top_score - relevant_score
                )

                print()
                print("-" * 60)
                print("SCORE ANALYSIS")
                print("-" * 60)

                print(
                    f"Top-1 score       : "
                    f"{top_score:.6f}"
                )

                print(
                    f"Relevant score    : "
                    f"{relevant_score:.6f}"
                )

                print(
                    f"Score difference  : "
                    f"{score_gap:.6f}"
                )

print()
print("=" * 60)
print("[SUCCESS] FAILURE ANALYSIS COMPLETE")
print("=" * 60)

DENSE RETRIEVAL FAILURE ANALYSIS

Total evaluations : 30
Top-1 failures    : 1

FAILURE 1

Question:
What does the knowledge item about 'Setting Up a Secure Connection to a Database' explain?

Ground-truth document(s):
  - doc_026

First relevant rank: 2

------------------------------------------------------------
RETRIEVED RESULTS
------------------------------------------------------------

Rank            : 1
Document ID     : doc_078
Chunk ID        : doc_078_chunk_001
Topic           : Setting Up a Secure Connection to a Company-Issued Database
Similarity      : 0.847220
Relevant        : False

Rank            : 2
Document ID     : doc_026
Chunk ID        : doc_026_chunk_001
Topic           : Setting Up a Secure Connection to a Database
Similarity      : 0.823682
Relevant        : True <-- GROUND TRUTH

Rank            : 3
Document ID     : doc_026
Chunk ID        : doc_026_chunk_002
Topic           : Setting Up a Secure Connection to a Database
Similarity      : 0.792612
Releva

In [27]:
# ============================================================
# CELL 22 — NATURAL SEMANTIC EVALUATION DATASET
# ============================================================

import os
import json
import time
import re
import requests
import pandas as pd
from datetime import datetime

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

PROJECT_ROOT = r"E:\rag"

DATASET_PATH = os.path.join(
    PROJECT_ROOT,
    "synthetic_knowledge_items.csv"
)

DOCUMENTS_PATH = os.path.join(
    PROJECT_ROOT,
    "data",
    "processed",
    "rag_documents.json"
)

EVALUATION_DIR = os.path.join(
    PROJECT_ROOT,
    "data",
    "evaluation"
)

CHECKPOINT_DIR = os.path.join(
    PROJECT_ROOT,
    "checkpoints"
)

SEMANTIC_QUESTIONS_PATH = os.path.join(
    EVALUATION_DIR,
    "semantic_evaluation_questions_v2.json"
)

SEMANTIC_CHECKPOINT_PATH = os.path.join(
    CHECKPOINT_DIR,
    "semantic_evaluation_dataset_v2.json"
)

SEMANTIC_PROGRESS_PATH = os.path.join(
    EVALUATION_DIR,
    "semantic_question_generation_progress_v2.json"
)

OLLAMA_URL = "http://localhost:11434"
OLLAMA_GENERATE_URL = f"{OLLAMA_URL}/api/generate"

OLLAMA_MODEL = "qwen3:8b"

RANDOM_SEED = 42
SAMPLE_SIZE = 30

MIN_WORDS = 8
MAX_WORDS = 25
MAX_CHARS = 250

REQUEST_TIMEOUT = 120
MAX_RETRIES = 3

os.makedirs(EVALUATION_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)


# ------------------------------------------------------------
# 2. Atomic save helper
# ------------------------------------------------------------

def save_json_atomic(data, path):

    temp_path = path + ".tmp"

    try:

        with open(
            temp_path,
            "w",
            encoding="utf-8"
        ) as f:

            json.dump(
                data,
                f,
                indent=2,
                ensure_ascii=False
            )

        os.replace(
            temp_path,
            path
        )

    except Exception:

        if os.path.exists(temp_path):

            try:
                os.remove(temp_path)
            except Exception:
                pass

        raise


# ------------------------------------------------------------
# 3. Load source dataset and documents
# ------------------------------------------------------------

try:

    print("=" * 60)
    print("NATURAL SEMANTIC EVALUATION DATASET")
    print("=" * 60)

    # -------------------------
    # Dataset
    # -------------------------

    if not os.path.exists(DATASET_PATH):
        raise FileNotFoundError(
            f"Dataset not found: {DATASET_PATH}"
        )

    df = pd.read_csv(DATASET_PATH)

    # -------------------------
    # Processed documents
    # -------------------------

    if not os.path.exists(DOCUMENTS_PATH):
        raise FileNotFoundError(
            f"Processed documents not found: {DOCUMENTS_PATH}"
        )

    with open(
        DOCUMENTS_PATH,
        "r",
        encoding="utf-8"
    ) as f:

        documents_data = json.load(f)

    # IMPORTANT:
    # rag_documents.json is already a LIST.
    if isinstance(documents_data, list):

        documents = documents_data

    elif isinstance(documents_data, dict):

        if "documents" in documents_data:

            documents = documents_data["documents"]

        else:

            raise ValueError(
                "Document JSON is a dictionary but does not "
                "contain a 'documents' field."
            )

    else:

        raise ValueError(
            "Unsupported rag_documents.json structure."
        )

    # -------------------------
    # Validate counts
    # -------------------------

    if len(df) != 100:

        raise ValueError(
            f"Expected 100 dataset records, found {len(df)}"
        )

    if len(documents) != 100:

        raise ValueError(
            f"Expected 100 processed documents, "
            f"found {len(documents)}"
        )

    print(
        f"[SUCCESS] Dataset loaded: {len(df)} records"
    )

    print(
        f"[SUCCESS] Documents loaded: {len(documents)}"
    )

except Exception as e:

    print(
        f"[ERROR] Failed to load source data: {e}"
    )

    raise


# ------------------------------------------------------------
# 4. Build document mapping
# ------------------------------------------------------------

try:

    document_by_row = {}

    for doc in documents:

        if not isinstance(doc, dict):

            raise ValueError(
                "Invalid document structure."
            )

        document_id = doc.get(
            "document_id"
        )

        metadata = doc.get(
            "metadata",
            {}
        )

        text = doc.get(
            "text"
        )

        if not document_id:

            raise ValueError(
                "Document missing document_id."
            )

        if not isinstance(text, str) or not text.strip():

            raise ValueError(
                f"Document {document_id} has invalid text."
            )

        original_row_index = metadata.get(
            "original_row_index"
        )

        if original_row_index is None:

            raise ValueError(
                f"{document_id} missing original_row_index."
            )

        document_by_row[
            int(original_row_index)
        ] = doc

    if len(document_by_row) != 100:

        raise ValueError(
            "Document-to-row mapping is incomplete."
        )

    print(
        "[SUCCESS] Document mapping validated"
    )

except Exception as e:

    print(
        f"[ERROR] Document mapping failed: {e}"
    )

    raise


# ------------------------------------------------------------
# 5. Deterministic evaluation sample
# ------------------------------------------------------------

try:

    evaluation_df = df.sample(
        n=SAMPLE_SIZE,
        random_state=RANDOM_SEED
    ).copy()

    # Preserve original dataframe index as row index.
    evaluation_df[
        "original_row_index"
    ] = evaluation_df.index

    evaluation_df = evaluation_df.reset_index(
        drop=True
    )

    evaluation_records = []

    for idx, row in evaluation_df.iterrows():

        original_row_index = int(
            row["original_row_index"]
        )

        if original_row_index not in document_by_row:

            raise ValueError(
                f"No document found for row "
                f"{original_row_index}"
            )

        doc = document_by_row[
            original_row_index
        ]

        topic = str(
            row["ki_topic"]
        ).strip()

        source_text = str(
            row["ki_text"]
        ).strip()

        if not topic:

            raise ValueError(
                f"Empty topic at row "
                f"{original_row_index}"
            )

        if not source_text:

            raise ValueError(
                f"Empty source text at row "
                f"{original_row_index}"
            )

        evaluation_records.append({

            "evaluation_id":
                f"semantic_eval_{idx + 1:03d}",

            "original_row_index":
                original_row_index,

            "document_id":
                doc["document_id"],

            "topic":
                topic,

            "source_text":
                source_text
        })

    print(
        f"[SUCCESS] Selected "
        f"{len(evaluation_records)} "
        f"documents for semantic evaluation"
    )

except Exception as e:

    print(
        f"[ERROR] Evaluation sampling failed: {e}"
    )

    raise


# ------------------------------------------------------------
# 6. Load generation progress
# ------------------------------------------------------------

progress = {}

if os.path.exists(
    SEMANTIC_PROGRESS_PATH
):

    try:

        with open(
            SEMANTIC_PROGRESS_PATH,
            "r",
            encoding="utf-8"
        ) as f:

            progress_data = json.load(f)

        if isinstance(
            progress_data,
            dict
        ):

            progress = progress_data

        print(
            f"[INFO] Existing progress found: "
            f"{len(progress)} records"
        )

    except Exception as e:

        print(
            f"[WARNING] Could not load progress: {e}"
        )

        progress = {}


# ------------------------------------------------------------
# 7. Text normalization
# ------------------------------------------------------------

def normalize_text(text):

    text = str(text).strip()

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text


# ------------------------------------------------------------
# 8. Question validation
# ------------------------------------------------------------

def validate_question(
    question,
    topic,
    source_text
):

    if not isinstance(
        question,
        str
    ):

        return False, "Question is not a string."

    question = normalize_text(
        question
    )

    if not question:

        return False, "Question is empty."

    if len(question) > MAX_CHARS:

        return False, "Question is too long."

    word_count = len(
        question.split()
    )

    if word_count < MIN_WORDS:

        return False, "Question is too short."

    if word_count > MAX_WORDS:

        return False, "Question exceeds word limit."

    if "?" not in question:

        return False, "Missing question mark."

    question_lower = question.lower()

    topic_lower = normalize_text(
        topic
    ).lower()

    # Exact topic must not appear.
    if topic_lower in question_lower:

        return False, "Exact topic appears in question."

    forbidden_phrases = [

        "knowledge item",
        "knowledge document",
        "source document",
        "source text",
        "dataset",
        "following it-related",
        "following it related",
        "it-related task or issue",
        "it related task or issue",
        "what guidance is provided",
        "what does the knowledge",
        "according to the document",
        "according to the source"
    ]

    for phrase in forbidden_phrases:

        if phrase in question_lower:

            return False, (
                f"Forbidden phrase detected: "
                f"'{phrase}'"
            )

    # --------------------------------------------------------
    # Source leakage detection
    # --------------------------------------------------------

    source_normalized = normalize_text(
        source_text
    ).lower()

    source_words = re.findall(
        r"\b[a-zA-Z]{4,}\b",
        source_normalized
    )

    question_words = set(
        re.findall(
            r"\b[a-zA-Z]{4,}\b",
            question_lower
        )
    )

    source_word_counts = {}

    for word in source_words:

        source_word_counts[word] = (
            source_word_counts.get(
                word,
                0
            ) + 1
        )

    repeated_source_words = [

        word
        for word, count
        in source_word_counts.items()
        if count >= 2
    ]

    overlap_count = sum(

        1
        for word in repeated_source_words
        if word in question_words
    )

    if overlap_count > 8:

        return False, (
            "Possible source-text leakage."
        )

    return True, "Valid"


# ------------------------------------------------------------
# 9. Generate one natural question
# ------------------------------------------------------------

def generate_semantic_question(
    topic,
    source_text
):

    prompt = f"""
You are creating a retrieval evaluation benchmark for an IT knowledge base.

Read the knowledge text below and generate ONE natural question that a real IT user might ask if they needed the information contained in this text.

IMPORTANT RULES:

- Ask about the underlying task, problem, procedure, or concept.
- Do NOT mention knowledge item, document, source, dataset, or evaluation.
- Do NOT copy sentences from the knowledge text.
- Do NOT reproduce step-by-step instructions.
- Do NOT use the exact title/topic.
- Do NOT include the answer.
- The question must make sense without seeing the knowledge text.
- Use 8 to 25 words.
- Output ONLY the question.
- End with a question mark.

Topic:
{topic}

Knowledge text:
{source_text}
"""

    payload = {

        "model":
            OLLAMA_MODEL,

        "prompt":
            prompt,

        "stream":
            False,

        "think":
            False,

        "options": {

            "temperature":
                0,

            "num_predict":
                80
        }
    }

    last_error = None

    for attempt in range(
        1,
        MAX_RETRIES + 1
    ):

        try:

            response = requests.post(

                OLLAMA_GENERATE_URL,

                json=payload,

                timeout=REQUEST_TIMEOUT
            )

            response.raise_for_status()

            result = response.json()

            question = normalize_text(
                result.get(
                    "response",
                    ""
                )
            )

            valid, reason = validate_question(

                question,

                topic,

                source_text
            )

            if not valid:

                raise ValueError(
                    reason
                )

            return question

        except Exception as e:

            last_error = e

            print(
                f"    [WARNING] Attempt "
                f"{attempt}/{MAX_RETRIES} failed: {e}"
            )

            if attempt < MAX_RETRIES:

                time.sleep(2)

    raise RuntimeError(

        f"Question generation failed after "
        f"{MAX_RETRIES} attempts: {last_error}"
    )


# ------------------------------------------------------------
# 10. Generate questions with crash-safe progress
# ------------------------------------------------------------

print()
print("-" * 60)
print("GENERATING NATURAL QUESTIONS")
print("-" * 60)

successful_count = 0
failed_count = 0

try:

    for record in evaluation_records:

        evaluation_id = record[
            "evaluation_id"
        ]

        # ----------------------------------------------------
        # Resume existing valid question
        # ----------------------------------------------------

        if evaluation_id in progress:

            existing = progress[
                evaluation_id
            ]

            existing_question = existing.get(
                "question"
            )

            if existing_question:

                valid, _ = validate_question(

                    existing_question,

                    record["topic"],

                    record["source_text"]
                )

                if valid:

                    successful_count += 1

                    print(
                        f"[SKIP] {evaluation_id} "
                        f"already generated"
                    )

                    continue

        # ----------------------------------------------------
        # Generate
        # ----------------------------------------------------

        print(
            f"[GENERATE] {evaluation_id} "
            f"({successful_count + failed_count + 1}/"
            f"{SAMPLE_SIZE})"
        )

        try:

            question = generate_semantic_question(

                record["topic"],

                record["source_text"]
            )

            progress[
                evaluation_id
            ] = {

                "evaluation_id":
                    evaluation_id,

                "original_row_index":
                    record["original_row_index"],

                "document_id":
                    record["document_id"],

                "question":
                    question,

                "generator":
                    OLLAMA_MODEL,

                "generated_at":
                    datetime.now().isoformat(),

                "status":
                    "success"
            }

            # SAVE AFTER EVERY QUESTION
            save_json_atomic(

                {

                    "stage":
                        "semantic_question_generation",

                    "status":
                        "in_progress",

                    "generator":
                        OLLAMA_MODEL,

                    "sample_size":
                        SAMPLE_SIZE,

                    "random_seed":
                        RANDOM_SEED,

                    "completed_questions":
                        len([
                            x
                            for x in progress.values()
                            if x.get("status")
                            == "success"
                        ]),

                    "questions":
                        progress,

                    "updated_at":
                        datetime.now().isoformat()
                },

                SEMANTIC_PROGRESS_PATH
            )

            successful_count += 1

            print(
                f"    [SUCCESS] {question}"
            )

        except Exception as e:

            failed_count += 1

            print(
                f"    [ERROR] "
                f"{evaluation_id}: {e}"
            )

            progress[
                evaluation_id
            ] = {

                "evaluation_id":
                    evaluation_id,

                "original_row_index":
                    record["original_row_index"],

                "document_id":
                    record["document_id"],

                "question":
                    None,

                "generator":
                    OLLAMA_MODEL,

                "error":
                    str(e),

                "status":
                    "failed",

                "updated_at":
                    datetime.now().isoformat()
            }

            save_json_atomic(

                {

                    "stage":
                        "semantic_question_generation",

                    "status":
                        "in_progress",

                    "generator":
                        OLLAMA_MODEL,

                    "sample_size":
                        SAMPLE_SIZE,

                    "random_seed":
                        RANDOM_SEED,

                    "completed_questions":
                        len([
                            x
                            for x in progress.values()
                            if x.get("status")
                            == "success"
                        ]),

                    "failed_questions":
                        len([
                            x
                            for x in progress.values()
                            if x.get("status")
                            == "failed"
                        ]),

                    "questions":
                        progress,

                    "updated_at":
                        datetime.now().isoformat()
                },

                SEMANTIC_PROGRESS_PATH
            )


except KeyboardInterrupt:

    print()
    print(
        "[WARNING] Generation interrupted."
    )

    print(
        "[INFO] Progress was saved after "
        "each question."
    )

except Exception as e:

    print()
    print(
        f"[ERROR] Generation stage failed: {e}"
    )

    raise


# ------------------------------------------------------------
# 11. Build final evaluation dataset
# ------------------------------------------------------------

try:

    final_questions = []

    for record in evaluation_records:

        evaluation_id = record[
            "evaluation_id"
        ]

        item = progress.get(
            evaluation_id
        )

        if not item:

            raise ValueError(
                f"Missing generated result for "
                f"{evaluation_id}"
            )

        question = item.get(
            "question"
        )

        if not question:

            raise ValueError(
                f"Question generation failed for "
                f"{evaluation_id}"
            )

        valid, reason = validate_question(

            question,

            record["topic"],

            record["source_text"]
        )

        if not valid:

            raise ValueError(

                f"Final validation failed for "
                f"{evaluation_id}: {reason}"
            )

        final_questions.append({

            "evaluation_id":
                evaluation_id,

            "question":
                question,

            "relevant_document_ids": [

                record["document_id"]
            ],

            "original_row_index":
                record["original_row_index"]
        })

    if len(final_questions) != SAMPLE_SIZE:

        raise ValueError(

            f"Expected {SAMPLE_SIZE} questions, "
            f"found {len(final_questions)}"
        )

    semantic_evaluation_data = {

        "stage":
            "semantic_retrieval_evaluation_dataset",

        "status":
            "success",

        "dataset":
            "Synthetic IT-Related Knowledge Items",

        "dataset_version":
            "Version 3",

        "generator":
            OLLAMA_MODEL,

        "sample_size":
            SAMPLE_SIZE,

        "random_seed":
            RANDOM_SEED,

        "question_type":
            "natural_semantic",

        "exact_topic_in_questions":
            False,

        "source_text_in_questions":
            False,

        "questions":
            final_questions,

        "created_at":
            datetime.now().isoformat()
    }

    # Final dataset
    save_json_atomic(

        semantic_evaluation_data,

        SEMANTIC_QUESTIONS_PATH
    )

    # Checkpoint
    save_json_atomic(

        semantic_evaluation_data,

        SEMANTIC_CHECKPOINT_PATH
    )

    # Complete progress
    save_json_atomic(

        {

            "stage":
                "semantic_question_generation",

            "status":
                "success",

            "generator":
                OLLAMA_MODEL,

            "sample_size":
                SAMPLE_SIZE,

            "random_seed":
                RANDOM_SEED,

            "completed_questions":
                SAMPLE_SIZE,

            "failed_questions":
                0,

            "questions":
                progress,

            "updated_at":
                datetime.now().isoformat()
        },

        SEMANTIC_PROGRESS_PATH
    )

    # --------------------------------------------------------
    # Summary
    # --------------------------------------------------------

    print()
    print("=" * 60)
    print("NATURAL SEMANTIC EVALUATION SUMMARY")
    print("=" * 60)

    print(
        f"Questions : {len(final_questions)}"
    )

    print(
        f"Generator : {OLLAMA_MODEL}"
    )

    print(
        "Exact topic in questions : excluded"
    )

    print(
        "Source-text leakage : excluded"
    )

    print()
    print("-" * 60)
    print("SAMPLE QUESTIONS")
    print("-" * 60)

    for item in final_questions[:5]:

        print()

        print(
            f"{item['evaluation_id']}: "
            f"{item['question']}"
        )

        print(
            f"Ground truth: "
            f"{item['relevant_document_ids']}"
        )

    print()
    print("=" * 60)
    print(
        "[SUCCESS] NATURAL SEMANTIC DATASET READY"
    )
    print("=" * 60)

    print()
    print("Saved:")

    print(
        f"  Questions : "
        f"{SEMANTIC_QUESTIONS_PATH}"
    )

    print(
        f"  Checkpoint: "
        f"{SEMANTIC_CHECKPOINT_PATH}"
    )

    print(
        f"  Progress  : "
        f"{SEMANTIC_PROGRESS_PATH}"
    )

except Exception as e:

    print()
    print(
        f"[ERROR] Final semantic dataset "
        f"creation failed: {e}"
    )

    raise

NATURAL SEMANTIC EVALUATION DATASET
[SUCCESS] Dataset loaded: 100 records
[SUCCESS] Documents loaded: 100
[SUCCESS] Document mapping validated
[SUCCESS] Selected 30 documents for semantic evaluation

------------------------------------------------------------
GENERATING NATURAL QUESTIONS
------------------------------------------------------------
[GENERATE] semantic_eval_001 (1/30)
    [SUCCESS] How can I determine if my keyboard is not working properly?
[GENERATE] semantic_eval_002 (2/30)
    [SUCCESS] How do I set up a new user's account in JIRA with the correct permissions and access?
[GENERATE] semantic_eval_003 (3/30)
    [SUCCESS] How do I securely connect to my company printer using my credentials and network settings?
[GENERATE] semantic_eval_004 (4/30)
    [SUCCESS] How can I fix unexpected errors when using Microsoft Publisher?
[GENERATE] semantic_eval_005 (5/30)
    [SUCCESS] How do I set up a network printer on my Windows or Mac computer?
[GENERATE] semantic_eval_006 (6/3

In [28]:
# ============================================================
# CELL 23 — NATURAL SEMANTIC RETRIEVAL EVALUATION
# ============================================================

import os
import json
import gc
import numpy as np
from datetime import datetime

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

PROJECT_ROOT = r"E:\rag"

SEMANTIC_QUESTIONS_PATH = os.path.join(
    PROJECT_ROOT,
    "data",
    "evaluation",
    "semantic_evaluation_questions_v2.json"
)

CHUNKS_PATH = os.path.join(
    PROJECT_ROOT,
    "data",
    "processed",
    "rag_chunks.json"
)

INDEX_PATH = os.path.join(
    PROJECT_ROOT,
    "data",
    "vector_index",
    "dense_index.faiss"
)

INDEX_MANIFEST_PATH = os.path.join(
    PROJECT_ROOT,
    "data",
    "vector_index",
    "dense_index_manifest.json"
)

RESULT_PATH = os.path.join(
    PROJECT_ROOT,
    "data",
    "evaluation",
    "semantic_dense_retrieval_evaluation_v2.json"
)

CHECKPOINT_PATH = os.path.join(
    PROJECT_ROOT,
    "checkpoints",
    "semantic_dense_retrieval_evaluation_v2.json"
)

TOP_K = 5


# ------------------------------------------------------------
# 2. Atomic JSON save
# ------------------------------------------------------------

def save_json_atomic(data, path):

    temp_path = path + ".tmp"

    try:

        with open(
            temp_path,
            "w",
            encoding="utf-8"
        ) as f:

            json.dump(
                data,
                f,
                indent=2,
                ensure_ascii=False
            )

        os.replace(
            temp_path,
            path
        )

    except Exception:

        if os.path.exists(temp_path):

            try:
                os.remove(temp_path)
            except Exception:
                pass

        raise


# ------------------------------------------------------------
# 3. Load semantic evaluation dataset
# ------------------------------------------------------------

try:

    print("=" * 60)
    print("NATURAL SEMANTIC RETRIEVAL EVALUATION")
    print("=" * 60)

    if not os.path.exists(
        SEMANTIC_QUESTIONS_PATH
    ):

        raise FileNotFoundError(
            f"Semantic evaluation dataset not found:\n"
            f"{SEMANTIC_QUESTIONS_PATH}"
        )

    with open(
        SEMANTIC_QUESTIONS_PATH,
        "r",
        encoding="utf-8"
    ) as f:

        semantic_data = json.load(f)

    if not isinstance(
        semantic_data,
        dict
    ):

        raise ValueError(
            "Semantic evaluation file must contain a JSON object."
        )

    semantic_questions = semantic_data.get(
        "questions"
    )

    if not isinstance(
        semantic_questions,
        list
    ):

        raise ValueError(
            "Semantic evaluation dataset does not contain "
            "a valid 'questions' list."
        )

    if len(semantic_questions) != 30:

        raise ValueError(
            f"Expected 30 semantic questions, "
            f"found {len(semantic_questions)}"
        )

    # Validate individual records.
    for item in semantic_questions:

        if not isinstance(item, dict):

            raise ValueError(
                "Invalid semantic evaluation record."
            )

        required_fields = [
            "evaluation_id",
            "question",
            "relevant_document_ids"
        ]

        for field in required_fields:

            if field not in item:

                raise ValueError(
                    f"Missing field '{field}' "
                    f"in semantic evaluation record."
                )

        if not item["question"].strip():

            raise ValueError(
                f"Empty question in "
                f"{item['evaluation_id']}"
            )

        if not item["relevant_document_ids"]:

            raise ValueError(
                f"Missing ground truth for "
                f"{item['evaluation_id']}"
            )

    print(
        f"[SUCCESS] Loaded {len(semantic_questions)} "
        f"semantic evaluation questions"
    )

except Exception as e:

    print(
        f"[ERROR] Failed to load semantic dataset: {e}"
    )

    raise


# ------------------------------------------------------------
# 4. Load chunks
# ------------------------------------------------------------

try:

    if not os.path.exists(CHUNKS_PATH):

        raise FileNotFoundError(
            f"Chunks file not found:\n{CHUNKS_PATH}"
        )

    with open(
        CHUNKS_PATH,
        "r",
        encoding="utf-8"
    ) as f:

        chunks_data = json.load(f)

    if isinstance(
        chunks_data,
        list
    ):

        chunks = chunks_data

    elif isinstance(
        chunks_data,
        dict
    ):

        chunks = chunks_data.get(
            "chunks"
        )

    else:

        raise ValueError(
            "Unsupported chunks JSON structure."
        )

    if not isinstance(
        chunks,
        list
    ):

        raise ValueError(
            "Could not extract chunk list."
        )

    if len(chunks) != 164:

        raise ValueError(
            f"Expected 164 chunks, found {len(chunks)}"
        )

    chunk_by_id = {}

    for chunk in chunks:

        chunk_id = chunk.get(
            "chunk_id"
        )

        document_id = chunk.get(
            "document_id"
        )

        if not chunk_id or not document_id:

            raise ValueError(
                "Chunk missing chunk_id or document_id."
            )

        chunk_by_id[chunk_id] = chunk

    print(
        f"[SUCCESS] Loaded {len(chunks)} chunks"
    )

except Exception as e:

    print(
        f"[ERROR] Failed to load chunks: {e}"
    )

    raise


# ------------------------------------------------------------
# 5. Load FAISS index
# ------------------------------------------------------------

try:

    import faiss

    if not os.path.exists(INDEX_PATH):

        raise FileNotFoundError(
            f"FAISS index not found:\n{INDEX_PATH}"
        )

    if not os.path.exists(
        INDEX_MANIFEST_PATH
    ):

        raise FileNotFoundError(
            f"Index manifest not found:\n"
            f"{INDEX_MANIFEST_PATH}"
        )

    with open(
        INDEX_MANIFEST_PATH,
        "r",
        encoding="utf-8"
    ) as f:

        manifest = json.load(f)

    index = faiss.read_index(
        INDEX_PATH
    )

    if index.ntotal != len(chunks):

        raise ValueError(
            f"Index contains {index.ntotal} vectors, "
            f"but {len(chunks)} chunks exist."
        )

    print(
        f"[SUCCESS] FAISS index loaded: "
        f"{index.ntotal} vectors"
    )

except Exception as e:

    print(
        f"[ERROR] Failed to load FAISS index: {e}"
    )

    raise


# ------------------------------------------------------------
# 6. Load embedding model
# ------------------------------------------------------------

try:

    from sentence_transformers import (
        SentenceTransformer
    )

    embedding_model_name = manifest.get(
        "embedding_model",
        "BAAI/bge-small-en-v1.5"
    )

    print(
        f"[INFO] Loading embedding model: "
        f"{embedding_model_name}"
    )

    embedding_model = SentenceTransformer(
        embedding_model_name,
        device="cuda"
    )

    embedding_dimension = (
        embedding_model.get_embedding_dimension()
    )

    expected_dimension = index.d

    if embedding_dimension != expected_dimension:

        raise ValueError(
            f"Embedding dimension mismatch: "
            f"model={embedding_dimension}, "
            f"index={expected_dimension}"
        )

    print(
        f"[SUCCESS] Embedding model loaded"
    )

    print(
        f"[INFO] Embedding dimension: "
        f"{embedding_dimension}"
    )

except Exception as e:

    print(
        f"[ERROR] Failed to load embedding model: {e}"
    )

    raise


# ------------------------------------------------------------
# 7. Retrieval helper
# ------------------------------------------------------------

def retrieve_semantic(
    query,
    top_k=5
):

    if not isinstance(
        query,
        str
    ) or not query.strip():

        raise ValueError(
            "Query must be a non-empty string."
        )

    if top_k < 1:

        raise ValueError(
            "top_k must be >= 1."
        )

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False
    )

    query_embedding = np.asarray(
        query_embedding,
        dtype=np.float32
    )

    if query_embedding.shape != (
        1,
        embedding_dimension
    ):

        raise ValueError(
            f"Unexpected query embedding shape: "
            f"{query_embedding.shape}"
        )

    scores, indices = index.search(
        query_embedding,
        top_k
    )

    results = []

    for rank, (
        score,
        chunk_index
    ) in enumerate(
        zip(
            scores[0],
            indices[0]
        ),
        start=1
    ):

        if chunk_index < 0:
            continue

        chunk = chunks[
            int(chunk_index)
        ]

        results.append({

            "rank":
                rank,

            "chunk_index":
                int(chunk_index),

            "chunk_id":
                chunk["chunk_id"],

            "document_id":
                chunk["document_id"],

            "topic":
                chunk.get(
                    "topic",
                    ""
                ),

            "score":
                float(score)
        })

    return results


# ------------------------------------------------------------
# 8. Evaluate retrieval metrics
# ------------------------------------------------------------

try:

    print()
    print("-" * 60)
    print("EVALUATING SEMANTIC QUESTIONS")
    print("-" * 60)

    evaluation_results = []

    recall_at_1_hits = 0
    recall_at_3_hits = 0
    recall_at_5_hits = 0

    reciprocal_ranks = []

    for question_number, item in enumerate(
        semantic_questions,
        start=1
    ):

        evaluation_id = item[
            "evaluation_id"
        ]

        query = item[
            "question"
        ].strip()

        ground_truth_ids = set(
            item[
                "relevant_document_ids"
            ]
        )

        retrieved = retrieve_semantic(
            query,
            TOP_K
        )

        retrieved_document_ids = [

            result[
                "document_id"
            ]

            for result in retrieved
        ]

        # ----------------------------------------------------
        # Recall@1
        # ----------------------------------------------------

        hit_at_1 = (
            len(
                ground_truth_ids.intersection(
                    retrieved_document_ids[:1]
                )
            ) > 0
        )

        # ----------------------------------------------------
        # Recall@3
        # ----------------------------------------------------

        hit_at_3 = (
            len(
                ground_truth_ids.intersection(
                    retrieved_document_ids[:3]
                )
            ) > 0
        )

        # ----------------------------------------------------
        # Recall@5
        # ----------------------------------------------------

        hit_at_5 = (
            len(
                ground_truth_ids.intersection(
                    retrieved_document_ids[:5]
                )
            ) > 0
        )

        if hit_at_1:
            recall_at_1_hits += 1

        if hit_at_3:
            recall_at_3_hits += 1

        if hit_at_5:
            recall_at_5_hits += 1

        # ----------------------------------------------------
        # Reciprocal Rank
        # ----------------------------------------------------

        reciprocal_rank = 0.0
        first_relevant_rank = None

        for rank, result in enumerate(
            retrieved,
            start=1
        ):

            if result[
                "document_id"
            ] in ground_truth_ids:

                first_relevant_rank = rank

                reciprocal_rank = (
                    1.0 / rank
                )

                break

        reciprocal_ranks.append(
            reciprocal_rank
        )

        # ----------------------------------------------------
        # Store individual result
        # ----------------------------------------------------

        evaluation_results.append({

            "evaluation_id":
                evaluation_id,

            "question":
                query,

            "ground_truth_document_ids":
                sorted(
                    ground_truth_ids
                ),

            "retrieved_document_ids":
                retrieved_document_ids,

            "first_relevant_rank":
                first_relevant_rank,

            "reciprocal_rank":
                reciprocal_rank,

            "hit_at_1":
                hit_at_1,

            "hit_at_3":
                hit_at_3,

            "hit_at_5":
                hit_at_5,

            "retrieved_chunks":
                retrieved
        })

        print(
            f"[{question_number:02d}/"
            f"{len(semantic_questions):02d}] "
            f"{evaluation_id} | "
            f"R@1={'YES' if hit_at_1 else 'NO'} | "
            f"R@5={'YES' if hit_at_5 else 'NO'}"
        )


    # --------------------------------------------------------
    # Aggregate metrics
    # --------------------------------------------------------

    total_questions = len(
        semantic_questions
    )

    recall_at_1 = (
        recall_at_1_hits /
        total_questions
    )

    recall_at_3 = (
        recall_at_3_hits /
        total_questions
    )

    recall_at_5 = (
        recall_at_5_hits /
        total_questions
    )

    mrr = (
        sum(reciprocal_ranks) /
        total_questions
    )


    # --------------------------------------------------------
    # Validate aggregate metrics
    # --------------------------------------------------------

    metrics = {

        "recall_at_1":
            round(
                recall_at_1,
                4
            ),

        "recall_at_3":
            round(
                recall_at_3,
                4
            ),

        "recall_at_5":
            round(
                recall_at_5,
                4
            ),

        "mrr":
            round(
                mrr,
                4
            ),

        "total_questions":
            total_questions,

        "hits_at_1":
            recall_at_1_hits,

        "hits_at_3":
            recall_at_3_hits,

        "hits_at_5":
            recall_at_5_hits
    }

    for metric_name in [
        "recall_at_1",
        "recall_at_3",
        "recall_at_5",
        "mrr"
    ]:

        value = metrics[
            metric_name
        ]

        if not (
            0.0 <= value <= 1.0
        ):

            raise ValueError(
                f"Invalid metric value for "
                f"{metric_name}: {value}"
            )


    # --------------------------------------------------------
    # Final evaluation artifact
    # --------------------------------------------------------

    semantic_retrieval_evaluation = {

        "stage":
            "natural_semantic_retrieval_evaluation",

        "status":
            "success",

        "evaluation_dataset":
            "semantic_evaluation_questions_v2.json",

        "retrieval_method":
            "dense",

        "embedding_model":
            embedding_model_name,

        "embedding_dimension":
            embedding_dimension,

        "index_type":
            type(index).__name__,

        "similarity_metric":
            "inner_product",

        "normalized_embeddings":
            True,

        "top_k":
            TOP_K,

        "metrics":
            metrics,

        "results":
            evaluation_results,

        "created_at":
            datetime.now().isoformat()
    }


    # --------------------------------------------------------
    # Save results + checkpoint
    # --------------------------------------------------------

    save_json_atomic(
        semantic_retrieval_evaluation,
        RESULT_PATH
    )

    save_json_atomic(
        semantic_retrieval_evaluation,
        CHECKPOINT_PATH
    )


    # --------------------------------------------------------
    # Final output
    # --------------------------------------------------------

    print()
    print("=" * 60)
    print("NATURAL SEMANTIC RETRIEVAL RESULTS")
    print("=" * 60)

    print(
        f"Questions : {total_questions}"
    )

    print()
    print(
        f"Recall@1  : "
        f"{recall_at_1:.4f} "
        f"({recall_at_1_hits}/{total_questions})"
    )

    print(
        f"Recall@3  : "
        f"{recall_at_3:.4f} "
        f"({recall_at_3_hits}/{total_questions})"
    )

    print(
        f"Recall@5  : "
        f"{recall_at_5:.4f} "
        f"({recall_at_5_hits}/{total_questions})"
    )

    print(
        f"MRR       : "
        f"{mrr:.4f}"
    )

    print()
    print("-" * 60)
    print("FAILURES")
    print("-" * 60)

    failures = [
        result
        for result in evaluation_results
        if not result["hit_at_5"]
    ]

    if failures:

        for failure in failures:

            print()
            print(
                f"{failure['evaluation_id']}: "
                f"{failure['question']}"
            )

            print(
                "Ground truth: "
                f"{failure['ground_truth_document_ids']}"
            )

            print(
                "Retrieved: "
                f"{failure['retrieved_document_ids']}"
            )

    else:

        print(
            "[SUCCESS] No Recall@5 failures."
        )

    print()
    print("=" * 60)
    print(
        "[SUCCESS] SEMANTIC RETRIEVAL EVALUATION COMPLETE"
    )
    print("=" * 60)

    print()
    print("Saved:")
    print(
        f"  Results   : {RESULT_PATH}"
    )
    print(
        f"  Checkpoint: {CHECKPOINT_PATH}"
    )

except Exception as e:

    print()
    print(
        f"[ERROR] Semantic retrieval evaluation failed: {e}"
    )

    raise


# ------------------------------------------------------------
# 9. Release embedding model memory
# ------------------------------------------------------------

try:

    del embedding_model

except Exception:

    pass

gc.collect()

try:

    import torch

    if torch.cuda.is_available():

        torch.cuda.empty_cache()

except Exception:

    pass

print()
print("[INFO] Embedding model memory released.")

NATURAL SEMANTIC RETRIEVAL EVALUATION
[SUCCESS] Loaded 30 semantic evaluation questions
[SUCCESS] Loaded 164 chunks
[SUCCESS] FAISS index loaded: 164 vectors
[INFO] Loading embedding model: BAAI/bge-small-en-v1.5


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9693.59it/s]


[SUCCESS] Embedding model loaded
[INFO] Embedding dimension: 384

------------------------------------------------------------
EVALUATING SEMANTIC QUESTIONS
------------------------------------------------------------
[01/30] semantic_eval_001 | R@1=YES | R@5=YES
[02/30] semantic_eval_002 | R@1=YES | R@5=YES
[03/30] semantic_eval_003 | R@1=YES | R@5=YES
[04/30] semantic_eval_004 | R@1=YES | R@5=YES
[05/30] semantic_eval_005 | R@1=YES | R@5=YES
[06/30] semantic_eval_006 | R@1=YES | R@5=YES
[07/30] semantic_eval_007 | R@1=YES | R@5=YES
[08/30] semantic_eval_008 | R@1=YES | R@5=YES
[09/30] semantic_eval_009 | R@1=YES | R@5=YES
[10/30] semantic_eval_010 | R@1=YES | R@5=YES
[11/30] semantic_eval_011 | R@1=YES | R@5=YES
[12/30] semantic_eval_012 | R@1=YES | R@5=YES
[13/30] semantic_eval_013 | R@1=YES | R@5=YES
[14/30] semantic_eval_014 | R@1=YES | R@5=YES
[15/30] semantic_eval_015 | R@1=NO | R@5=YES
[16/30] semantic_eval_016 | R@1=YES | R@5=YES
[17/30] semantic_eval_017 | R@1=YES | R@5=YES
[

In [29]:
# ============================================================
# CELL 24 — SEMANTIC RETRIEVAL FAILURE ANALYSIS
# ============================================================

import os
import json
import numpy as np
from datetime import datetime

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

PROJECT_ROOT = r"E:\rag"

RESULT_PATH = os.path.join(
    PROJECT_ROOT,
    "data",
    "evaluation",
    "semantic_dense_retrieval_evaluation_v2.json"
)

CHUNKS_PATH = os.path.join(
    PROJECT_ROOT,
    "data",
    "processed",
    "rag_chunks.json"
)

ANALYSIS_PATH = os.path.join(
    PROJECT_ROOT,
    "data",
    "evaluation",
    "semantic_retrieval_failure_analysis_v2.json"
)

CHECKPOINT_PATH = os.path.join(
    PROJECT_ROOT,
    "checkpoints",
    "semantic_retrieval_failure_analysis_v2.json"
)


# ------------------------------------------------------------
# 2. Atomic save helper
# ------------------------------------------------------------

def save_json_atomic(data, path):

    temp_path = path + ".tmp"

    try:

        with open(
            temp_path,
            "w",
            encoding="utf-8"
        ) as f:

            json.dump(
                data,
                f,
                indent=2,
                ensure_ascii=False
            )

        os.replace(
            temp_path,
            path
        )

    except Exception:

        if os.path.exists(temp_path):

            try:
                os.remove(temp_path)
            except Exception:
                pass

        raise


# ------------------------------------------------------------
# 3. Load evaluation results
# ------------------------------------------------------------

try:

    print("=" * 60)
    print("SEMANTIC RETRIEVAL FAILURE ANALYSIS")
    print("=" * 60)

    if not os.path.exists(
        RESULT_PATH
    ):

        raise FileNotFoundError(
            f"Evaluation results not found:\n"
            f"{RESULT_PATH}"
        )

    with open(
        RESULT_PATH,
        "r",
        encoding="utf-8"
    ) as f:

        evaluation_data = json.load(f)

    results = evaluation_data.get(
        "results"
    )

    if not isinstance(
        results,
        list
    ):

        raise ValueError(
            "Evaluation results do not contain "
            "a valid results list."
        )

    print(
        f"[SUCCESS] Loaded {len(results)} "
        f"evaluation results"
    )

except Exception as e:

    print(
        f"[ERROR] Failed to load evaluation results: {e}"
    )

    raise


# ------------------------------------------------------------
# 4. Load chunks
# ------------------------------------------------------------

try:

    if not os.path.exists(
        CHUNKS_PATH
    ):

        raise FileNotFoundError(
            f"Chunks file not found:\n"
            f"{CHUNKS_PATH}"
        )

    with open(
        CHUNKS_PATH,
        "r",
        encoding="utf-8"
    ) as f:

        chunks_data = json.load(f)

    if isinstance(
        chunks_data,
        list
    ):

        chunks = chunks_data

    elif isinstance(
        chunks_data,
        dict
    ):

        chunks = chunks_data.get(
            "chunks"
        )

    else:

        raise ValueError(
            "Unsupported chunk JSON structure."
        )

    if not isinstance(
        chunks,
        list
    ):

        raise ValueError(
            "Could not extract chunk list."
        )

    chunk_by_id = {}

    for chunk in chunks:

        chunk_id = chunk.get(
            "chunk_id"
        )

        if not chunk_id:

            raise ValueError(
                "Chunk missing chunk_id."
            )

        chunk_by_id[
            chunk_id
        ] = chunk

    print(
        f"[SUCCESS] Loaded {len(chunks)} chunks"
    )

except Exception as e:

    print(
        f"[ERROR] Failed to load chunks: {e}"
    )

    raise


# ------------------------------------------------------------
# 5. Identify R@1 failures
# ------------------------------------------------------------

try:

    failures = [

        result

        for result in results

        if not result.get(
            "hit_at_1",
            False
        )
    ]

    print()
    print(
        f"[INFO] R@1 failures: "
        f"{len(failures)}"
    )

    if len(failures) == 0:

        print(
            "[SUCCESS] No R@1 failures to analyze."
        )

except Exception as e:

    print(
        f"[ERROR] Failed to identify failures: {e}"
    )

    raise


# ------------------------------------------------------------
# 6. Analyze each failure
# ------------------------------------------------------------

failure_analysis = []

for failure_number, result in enumerate(
    failures,
    start=1
):

    try:

        evaluation_id = result[
            "evaluation_id"
        ]

        question = result[
            "question"
        ]

        ground_truth_ids = set(
            result[
                "ground_truth_document_ids"
            ]
        )

        retrieved_chunks = result.get(
            "retrieved_chunks",
            []
        )

        # ----------------------------------------------------
        # Find first relevant result
        # ----------------------------------------------------

        relevant_rank = None
        relevant_score = None
        relevant_chunk_id = None
        relevant_topic = None

        for retrieved in retrieved_chunks:

            if retrieved[
                "document_id"
            ] in ground_truth_ids:

                relevant_rank = retrieved[
                    "rank"
                ]

                relevant_score = retrieved[
                    "score"
                ]

                relevant_chunk_id = retrieved[
                    "chunk_id"
                ]

                relevant_topic = retrieved.get(
                    "topic",
                    ""
                )

                break

        # ----------------------------------------------------
        # Top-ranked result
        # ----------------------------------------------------

        top_result = (
            retrieved_chunks[0]
            if retrieved_chunks
            else None
        )

        if top_result is None:

            raise ValueError(
                f"No retrieved results for "
                f"{evaluation_id}"
            )

        top_score = top_result[
            "score"
        ]

        top_document_id = top_result[
            "document_id"
        ]

        top_chunk_id = top_result[
            "chunk_id"
        ]

        top_topic = top_result.get(
            "topic",
            ""
        )

        # ----------------------------------------------------
        # Score gap
        # ----------------------------------------------------

        score_gap = None

        if relevant_score is not None:

            score_gap = (
                top_score -
                relevant_score
            )

        # ----------------------------------------------------
        # Retrieve full chunk text
        # ----------------------------------------------------

        top_chunk = chunk_by_id.get(
            top_chunk_id
        )

        relevant_chunk = None

        if relevant_chunk_id:

            relevant_chunk = chunk_by_id.get(
                relevant_chunk_id
            )

        # ----------------------------------------------------
        # Determine failure type
        # ----------------------------------------------------

        if (
            relevant_rank is not None
            and relevant_rank <= 5
        ):

            failure_type = (
                "ranking_failure"
            )

        else:

            failure_type = (
                "retrieval_failure"
            )

        analysis_item = {

            "evaluation_id":
                evaluation_id,

            "question":
                question,

            "ground_truth_document_ids":
                sorted(
                    ground_truth_ids
                ),

            "top_result": {

                "rank":
                    top_result["rank"],

                "document_id":
                    top_document_id,

                "chunk_id":
                    top_chunk_id,

                "topic":
                    top_topic,

                "score":
                    top_score
            },

            "relevant_result": {

                "rank":
                    relevant_rank,

                "document_id":
                    (
                        sorted(
                            ground_truth_ids
                        )[0]
                        if ground_truth_ids
                        else None
                    ),

                "chunk_id":
                    relevant_chunk_id,

                "topic":
                    relevant_topic,

                "score":
                    relevant_score
            },

            "score_gap_top_vs_relevant":
                score_gap,

            "failure_type":
                failure_type,

            "top_chunk_text":
                (
                    top_chunk.get(
                        "text",
                        ""
                    )
                    if top_chunk
                    else None
                ),

            "relevant_chunk_text":
                (
                    relevant_chunk.get(
                        "text",
                        ""
                    )
                    if relevant_chunk
                    else None
                )
        }

        failure_analysis.append(
            analysis_item
        )

    except Exception as e:

        print(
            f"[ERROR] Failed to analyze "
            f"{result.get('evaluation_id', 'unknown')}: {e}"
        )

        raise


# ------------------------------------------------------------
# 7. Aggregate failure statistics
# ------------------------------------------------------------

try:

    ranking_failures = sum(

        1

        for item in failure_analysis

        if item["failure_type"]
        == "ranking_failure"
    )

    retrieval_failures = sum(

        1

        for item in failure_analysis

        if item["failure_type"]
        == "retrieval_failure"
    )

    score_gaps = [

        item[
            "score_gap_top_vs_relevant"
        ]

        for item in failure_analysis

        if item[
            "score_gap_top_vs_relevant"
        ] is not None
    ]

    if score_gaps:

        mean_score_gap = float(
            np.mean(score_gaps)
        )

        max_score_gap = float(
            np.max(score_gaps)
        )

        min_score_gap = float(
            np.min(score_gaps)
        )

    else:

        mean_score_gap = None
        max_score_gap = None
        min_score_gap = None

    summary = {

        "total_evaluation_questions":
            len(results),

        "total_r_at_1_failures":
            len(failure_analysis),

        "ranking_failures":
            ranking_failures,

        "retrieval_failures":
            retrieval_failures,

        "mean_top_vs_relevant_score_gap":
            (
                round(
                    mean_score_gap,
                    6
                )
                if mean_score_gap is not None
                else None
            ),

        "minimum_score_gap":
            (
                round(
                    min_score_gap,
                    6
                )
                if min_score_gap is not None
                else None
            ),

        "maximum_score_gap":
            (
                round(
                    max_score_gap,
                    6
                )
                if max_score_gap is not None
                else None
            )
    }

except Exception as e:

    print(
        f"[ERROR] Failed to calculate failure statistics: {e}"
    )

    raise


# ------------------------------------------------------------
# 8. Final analysis artifact
# ------------------------------------------------------------

try:

    failure_analysis_data = {

        "stage":
            "semantic_retrieval_failure_analysis",

        "status":
            "success",

        "evaluation_source":
            "semantic_dense_retrieval_evaluation_v2.json",

        "retrieval_method":
            "dense",

        "embedding_model":
            evaluation_data.get(
                "embedding_model"
            ),

        "top_k":
            evaluation_data.get(
                "top_k"
            ),

        "summary":
            summary,

        "failures":
            failure_analysis,

        "created_at":
            datetime.now().isoformat()
    }

    # --------------------------------------------------------
    # Validate before saving
    # --------------------------------------------------------

    if len(failure_analysis) != 5:

        raise ValueError(
            f"Expected 5 R@1 failures, "
            f"found {len(failure_analysis)}"
        )

    for item in failure_analysis:

        if not item["question"]:

            raise ValueError(
                "Failure analysis contains "
                "an empty question."
            )

        if not item["ground_truth_document_ids"]:

            raise ValueError(
                "Failure analysis contains "
                "missing ground truth."
            )

    # --------------------------------------------------------
    # Save
    # --------------------------------------------------------

    save_json_atomic(
        failure_analysis_data,
        ANALYSIS_PATH
    )

    save_json_atomic(
        failure_analysis_data,
        CHECKPOINT_PATH
    )

    # --------------------------------------------------------
    # Output
    # --------------------------------------------------------

    print()
    print("=" * 60)
    print("SEMANTIC RETRIEVAL FAILURE SUMMARY")
    print("=" * 60)

    print(
        f"Total questions       : "
        f"{summary['total_evaluation_questions']}"
    )

    print(
        f"R@1 failures          : "
        f"{summary['total_r_at_1_failures']}"
    )

    print(
        f"Ranking failures      : "
        f"{summary['ranking_failures']}"
    )

    print(
        f"Retrieval failures    : "
        f"{summary['retrieval_failures']}"
    )

    if mean_score_gap is not None:

        print(
            f"Mean score gap        : "
            f"{mean_score_gap:.6f}"
        )

    print()
    print("-" * 60)
    print("FAILURE DETAILS")
    print("-" * 60)

    for item in failure_analysis:

        print()

        print(
            f"{item['evaluation_id']}: "
            f"{item['question']}"
        )

        print(
            "Ground truth: "
            f"{item['ground_truth_document_ids']}"
        )

        print(
            "Top result: "
            f"{item['top_result']['document_id']} | "
            f"{item['top_result']['topic']} | "
            f"score="
            f"{item['top_result']['score']:.6f}"
        )

        print(
            "Relevant result: "
            f"{item['relevant_result']['document_id']} | "
            f"{item['relevant_result']['topic']} | "
            f"rank="
            f"{item['relevant_result']['rank']} | "
            f"score="
            f"{item['relevant_result']['score']:.6f}"
        )

        print(
            "Score gap: "
            f"{item['score_gap_top_vs_relevant']:.6f}"
        )

        print(
            "Failure type: "
            f"{item['failure_type']}"
        )

    print()
    print("=" * 60)
    print(
        "[SUCCESS] FAILURE ANALYSIS COMPLETE"
    )
    print("=" * 60)

    print()
    print("Saved:")
    print(
        f"  Analysis   : {ANALYSIS_PATH}"
    )

    print(
        f"  Checkpoint : {CHECKPOINT_PATH}"
    )

except Exception as e:

    print()
    print(
        f"[ERROR] Failed to save failure analysis: {e}"
    )

    raise

SEMANTIC RETRIEVAL FAILURE ANALYSIS
[SUCCESS] Loaded 30 evaluation results
[SUCCESS] Loaded 164 chunks

[INFO] R@1 failures: 5

SEMANTIC RETRIEVAL FAILURE SUMMARY
Total questions       : 30
R@1 failures          : 5
Ranking failures      : 5
Retrieval failures    : 0
Mean score gap        : 0.008861

------------------------------------------------------------
FAILURE DETAILS
------------------------------------------------------------

semantic_eval_015: How can I determine if my laptop's battery is causing performance issues?
Ground truth: ['doc_091']
Top result: doc_098 |  | score=0.747605
Relevant result: doc_091 |  | rank=2 | score=0.745346
Score gap: 0.002258
Failure type: ranking_failure

semantic_eval_019: How do I protect my system from data loss by creating a restore point in Windows?
Ground truth: ['doc_013']
Top result: doc_034 |  | score=0.863143
Relevant result: doc_013 |  | rank=2 | score=0.856985
Score gap: 0.006157
Failure type: ranking_failure

semantic_eval_025: Why 

In [33]:
# ============================================================
# CELL 25 — END-TO-END RAG ANSWER GENERATION EVALUATION
# ============================================================

import os
import json
import time
import gc
import requests
import numpy as np
from datetime import datetime

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

PROJECT_ROOT = r"E:\rag"

SEMANTIC_QUESTIONS_PATH = os.path.join(
    PROJECT_ROOT,
    "data",
    "evaluation",
    "semantic_evaluation_questions_v2.json"
)

CHUNKS_PATH = os.path.join(
    PROJECT_ROOT,
    "data",
    "processed",
    "rag_chunks.json"
)

INDEX_PATH = os.path.join(
    PROJECT_ROOT,
    "data",
    "vector_index",
    "dense_index.faiss"
)

INDEX_MANIFEST_PATH = os.path.join(
    PROJECT_ROOT,
    "data",
    "vector_index",
    "dense_index_manifest.json"
)

RESULT_PATH = os.path.join(
    PROJECT_ROOT,
    "data",
    "evaluation",
    "end_to_end_rag_evaluation_v2.json"
)

CHECKPOINT_PATH = os.path.join(
    PROJECT_ROOT,
    "checkpoints",
    "end_to_end_rag_evaluation_v2.json"
)

PROGRESS_PATH = os.path.join(
    PROJECT_ROOT,
    "data",
    "evaluation",
    "end_to_end_rag_progress_v2.json"
)

OLLAMA_URL = "http://localhost:11434"
OLLAMA_GENERATE_URL = f"{OLLAMA_URL}/api/generate"

OLLAMA_MODEL = "qwen3:8b"

TOP_K = 5
TEMPERATURE = 0.0
NUM_PREDICT = 256
REQUEST_TIMEOUT = 180
MAX_RETRIES = 3


# ------------------------------------------------------------
# 2. Atomic JSON save
# ------------------------------------------------------------

def save_json_atomic(data, path):

    temp_path = path + ".tmp"

    try:

        os.makedirs(
            os.path.dirname(path),
            exist_ok=True
        )

        with open(
            temp_path,
            "w",
            encoding="utf-8"
        ) as f:

            json.dump(
                data,
                f,
                indent=2,
                ensure_ascii=False
            )

        os.replace(
            temp_path,
            path
        )

    except Exception:

        if os.path.exists(temp_path):

            try:
                os.remove(temp_path)
            except Exception:
                pass

        raise


# ------------------------------------------------------------
# 3. Load semantic evaluation questions
# ------------------------------------------------------------

try:

    print("=" * 60)
    print("END-TO-END RAG ANSWER GENERATION EVALUATION")
    print("=" * 60)

    if not os.path.exists(
        SEMANTIC_QUESTIONS_PATH
    ):

        raise FileNotFoundError(
            f"Semantic evaluation dataset not found:\n"
            f"{SEMANTIC_QUESTIONS_PATH}"
        )

    with open(
        SEMANTIC_QUESTIONS_PATH,
        "r",
        encoding="utf-8"
    ) as f:

        semantic_data = json.load(f)

    evaluation_questions = semantic_data.get(
        "questions"
    )

    if not isinstance(
        evaluation_questions,
        list
    ):

        raise ValueError(
            "Invalid semantic evaluation dataset."
        )

    if len(evaluation_questions) != 30:

        raise ValueError(
            f"Expected 30 questions, "
            f"found {len(evaluation_questions)}."
        )

    print(
        f"[SUCCESS] Loaded {len(evaluation_questions)} "
        f"evaluation questions."
    )

except Exception as e:

    print(
        f"[ERROR] Failed to load evaluation dataset: {e}"
    )

    raise


# ------------------------------------------------------------
# 4. Load chunks
# ------------------------------------------------------------

try:

    if not os.path.exists(
        CHUNKS_PATH
    ):

        raise FileNotFoundError(
            f"Chunks file not found:\n{CHUNKS_PATH}"
        )

    with open(
        CHUNKS_PATH,
        "r",
        encoding="utf-8"
    ) as f:

        chunks_data = json.load(f)

    if isinstance(
        chunks_data,
        list
    ):

        chunks = chunks_data

    elif isinstance(
        chunks_data,
        dict
    ):

        chunks = chunks_data.get(
            "chunks"
        )

    else:

        raise ValueError(
            "Unsupported chunks JSON structure."
        )

    if not isinstance(
        chunks,
        list
    ):

        raise ValueError(
            "Could not extract chunks."
        )

    if len(chunks) != 164:

        raise ValueError(
            f"Expected 164 chunks, "
            f"found {len(chunks)}."
        )

    print(
        f"[SUCCESS] Loaded {len(chunks)} chunks."
    )

except Exception as e:

    print(
        f"[ERROR] Failed to load chunks: {e}"
    )

    raise


# ------------------------------------------------------------
# 5. Load FAISS index and manifest
# ------------------------------------------------------------

try:

    import faiss

    if not os.path.exists(
        INDEX_PATH
    ):

        raise FileNotFoundError(
            f"FAISS index not found:\n{INDEX_PATH}"
        )

    if not os.path.exists(
        INDEX_MANIFEST_PATH
    ):

        raise FileNotFoundError(
            f"Index manifest not found:\n"
            f"{INDEX_MANIFEST_PATH}"
        )

    with open(
        INDEX_MANIFEST_PATH,
        "r",
        encoding="utf-8"
    ) as f:

        manifest = json.load(f)

    index = faiss.read_index(
        INDEX_PATH
    )

    if index.ntotal != len(chunks):

        raise ValueError(
            f"Index vectors = {index.ntotal}, "
            f"chunks = {len(chunks)}."
        )

    embedding_model_name = manifest.get(
        "embedding_model",
        "BAAI/bge-small-en-v1.5"
    )

    print(
        f"[SUCCESS] FAISS index loaded: "
        f"{index.ntotal} vectors."
    )

    print(
        f"[INFO] Embedding model: "
        f"{embedding_model_name}"
    )

except Exception as e:

    print(
        f"[ERROR] Failed to load FAISS index: {e}"
    )

    raise


# ------------------------------------------------------------
# 6. Load embedding model
# ------------------------------------------------------------

try:

    from sentence_transformers import SentenceTransformer

    print(
        f"[INFO] Loading embedding model: "
        f"{embedding_model_name}"
    )

    embedding_model = SentenceTransformer(
        embedding_model_name,
        device="cuda"
    )

    embedding_dimension = (
        embedding_model.get_embedding_dimension()
    )

    if embedding_dimension != index.d:

        raise ValueError(
            f"Embedding dimension mismatch: "
            f"model = {embedding_dimension}, "
            f"index = {index.d}."
        )

    print(
        "[SUCCESS] Embedding model loaded."
    )

    print(
        f"[INFO] Dimension: {embedding_dimension}"
    )

except Exception as e:

    print(
        f"[ERROR] Failed to load embedding model: {e}"
    )

    raise


# ------------------------------------------------------------
# 7. Build chunk lookup
# ------------------------------------------------------------

try:

    chunk_by_id = {}

    for chunk in chunks:

        chunk_id = chunk.get(
            "chunk_id"
        )

        if not chunk_id:

            raise ValueError(
                "Chunk missing chunk_id."
            )

        chunk_by_id[
            chunk_id
        ] = chunk

    if len(chunk_by_id) != len(chunks):

        raise ValueError(
            "Duplicate chunk IDs detected."
        )

    print(
        f"[SUCCESS] Chunk lookup created "
        f"for {len(chunk_by_id)} chunks."
    )

except Exception as e:

    print(
        f"[ERROR] Failed to create chunk lookup: {e}"
    )

    raise


# ------------------------------------------------------------
# 8. Retrieval function
# ------------------------------------------------------------

def retrieve_for_rag(
    query,
    top_k=5
):

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False
    )

    query_embedding = np.asarray(
        query_embedding,
        dtype=np.float32
    )

    scores, indices = index.search(
        query_embedding,
        top_k
    )

    results = []

    for rank, score, chunk_index in zip(
        range(1, top_k + 1),
        scores[0],
        indices[0]
    ):

        if chunk_index < 0:
            continue

        chunk = chunks[
            int(chunk_index)
        ]

        results.append(
            {
                "rank": rank,
                "chunk_id": chunk["chunk_id"],
                "document_id": chunk["document_id"],
                "topic": chunk.get(
                    "topic",
                    ""
                ),
                "score": float(score),
                "text": chunk.get(
                    "text",
                    ""
                )
            }
        )

    return results


# ------------------------------------------------------------
# 9. Build RAG prompt
# ------------------------------------------------------------

def build_rag_prompt(
    question,
    retrieved_chunks
):

    context_sections = []

    for item in retrieved_chunks:

        section = (
            f"[Context {item['rank']}]\n"
            f"Topic: {item['topic']}\n"
            f"Content:\n{item['text']}"
        )

        context_sections.append(
            section
        )

    context_text = "\n\n".join(
        context_sections
    )

    prompt = (
        "You are an IT support assistant.\n\n"
        "Answer the user's question using ONLY "
        "the provided context.\n\n"
        "Rules:\n"
        "- Use the retrieved context as the source of truth.\n"
        "- Do not invent information.\n"
        "- Do not rely on outside knowledge.\n"
        "- If the context does not contain enough information, "
        "say that the available information is insufficient.\n"
        "- Give a clear and useful answer.\n"
        "- Do not mention the retrieval process, embeddings, "
        "chunks, or evaluation.\n"
        "- Do not mention that you are using context.\n"
        "- Do not fabricate specific procedures, settings, "
        "commands, or facts.\n\n"
        f"USER QUESTION:\n{question}\n\n"
        f"RETRIEVED CONTEXT:\n{context_text}\n\n"
        "ANSWER:"
    )

    return prompt


# ------------------------------------------------------------
# 10. Ollama generation
# ------------------------------------------------------------

def generate_rag_answer(
    prompt
):

    payload = {

        "model": OLLAMA_MODEL,

        "prompt": prompt,

        "stream": False,

        "think": False,

        "options": {

            "temperature": TEMPERATURE,

            "num_predict": NUM_PREDICT
        }
    }

    last_error = None

    for attempt in range(
        1,
        MAX_RETRIES + 1
    ):

        try:

            response = requests.post(
                OLLAMA_GENERATE_URL,
                json=payload,
                timeout=REQUEST_TIMEOUT
            )

            response.raise_for_status()

            result = response.json()

            answer = str(
                result.get(
                    "response",
                    ""
                )
            ).strip()

            if not answer:

                raise ValueError(
                    "Ollama returned an empty answer."
                )

            return {

                "answer": answer,

                "done": result.get(
                    "done"
                ),

                "done_reason": result.get(
                    "done_reason"
                ),

                "response_length": len(
                    answer
                )
            }

        except Exception as e:

            last_error = e

            print(
                f"    [WARNING] Attempt "
                f"{attempt}/{MAX_RETRIES} failed: {e}"
            )

            if attempt < MAX_RETRIES:

                time.sleep(2)

    raise RuntimeError(
        f"Ollama generation failed after "
        f"{MAX_RETRIES} attempts: {last_error}"
    )


# ------------------------------------------------------------
# 11. Load existing progress
# ------------------------------------------------------------

progress = {}

if os.path.exists(
    PROGRESS_PATH
):

    try:

        with open(
            PROGRESS_PATH,
            "r",
            encoding="utf-8"
        ) as f:

            progress_data = json.load(f)

        if isinstance(
            progress_data,
            dict
        ):

            progress = progress_data

        print(
            f"[INFO] Existing progress loaded: "
            f"{len(progress)} records."
        )

    except Exception as e:

        print(
            f"[WARNING] Could not load progress: {e}"
        )

        progress = {}


# ------------------------------------------------------------
# 12. Generate RAG answers
# ------------------------------------------------------------

print()
print("-" * 60)
print("GENERATING END-TO-END RAG ANSWERS")
print("-" * 60)

successful_count = 0
failed_count = 0

try:

    for number, evaluation_item in enumerate(
        evaluation_questions,
        start=1
    ):

        evaluation_id = evaluation_item[
            "evaluation_id"
        ]

        question = evaluation_item[
            "question"
        ].strip()

        ground_truth_ids = evaluation_item[
            "relevant_document_ids"
        ]

        # ----------------------------------------------------
        # Resume completed item
        # ----------------------------------------------------

        if evaluation_id in progress:

            existing = progress[
                evaluation_id
            ]

            if (
                existing.get("status")
                == "success"
                and existing.get("answer")
            ):

                successful_count += 1

                print(
                    f"[SKIP] {evaluation_id} "
                    f"already completed."
                )

                continue

        # ----------------------------------------------------
        # Question
        # ----------------------------------------------------

        print()
        print(
            f"[{number:02d}/"
            f"{len(evaluation_questions):02d}] "
            f"{evaluation_id}"
        )

        print(
            f"    Question: {question}"
        )

        try:

            # ------------------------------------------------
            # Retrieval
            # ------------------------------------------------

            retrieved_chunks = retrieve_for_rag(
                question,
                TOP_K
            )

            if len(retrieved_chunks) != TOP_K:

                raise ValueError(
                    f"Expected {TOP_K} retrieved chunks, "
                    f"found {len(retrieved_chunks)}."
                )

            # ------------------------------------------------
            # Context validation
            # ------------------------------------------------

            for retrieved in retrieved_chunks:

                if not retrieved.get("text"):

                    raise ValueError(
                        "Retrieved chunk contains empty text."
                    )

            # ------------------------------------------------
            # Build prompt
            # ------------------------------------------------

            prompt = build_rag_prompt(
                question,
                retrieved_chunks
            )

            # ------------------------------------------------
            # Generate answer
            # ------------------------------------------------

            generation_start = time.time()

            generation = generate_rag_answer(
                prompt
            )

            generation_time = (
                time.time()
                - generation_start
            )

            answer = generation[
                "answer"
            ]

            # ------------------------------------------------
            # Validate answer
            # ------------------------------------------------

            if not answer.strip():

                raise ValueError(
                    "Generated answer is empty."
                )

            # ------------------------------------------------
            # Check whether ground truth appears in Top-K
            # ------------------------------------------------

            retrieved_document_ids = [

                item["document_id"]

                for item in retrieved_chunks
            ]

            ground_truth_in_top_k = any(

                document_id in ground_truth_ids

                for document_id
                in retrieved_document_ids
            )

            # ------------------------------------------------
            # Persist successful result
            # ------------------------------------------------

            progress[
                evaluation_id
            ] = {

                "evaluation_id":
                    evaluation_id,

                "question":
                    question,

                "ground_truth_document_ids":
                    ground_truth_ids,

                "retrieved_chunks":
                    retrieved_chunks,

                "retrieved_document_ids":
                    retrieved_document_ids,

                "ground_truth_in_top_k":
                    ground_truth_in_top_k,

                "prompt":
                    prompt,

                "answer":
                    answer,

                "generator":
                    OLLAMA_MODEL,

                "temperature":
                    TEMPERATURE,

                "num_predict":
                    NUM_PREDICT,

                "generation_time_seconds":
                    round(
                        generation_time,
                        4
                    ),

                "done":
                    generation.get(
                        "done"
                    ),

                "done_reason":
                    generation.get(
                        "done_reason"
                    ),

                "status":
                    "success",

                "generated_at":
                    datetime.now().isoformat()
            }

            # ------------------------------------------------
            # Save immediately
            # ------------------------------------------------

            completed_now = len([

                x

                for x in progress.values()

                if x.get("status")
                == "success"
            ])

            failed_now = len([

                x

                for x in progress.values()

                if x.get("status")
                == "failed"
            ])

            progress_snapshot = {

                "stage":
                    "end_to_end_rag_generation",

                "status":
                    "in_progress",

                "generator":
                    OLLAMA_MODEL,

                "completed":
                    completed_now,

                "failed":
                    failed_now,

                "results":
                    progress,

                "updated_at":
                    datetime.now().isoformat()
            }

            save_json_atomic(
                progress_snapshot,
                PROGRESS_PATH
            )

            successful_count += 1

            print(
                f"    Top retrieved: "
                f"{retrieved_chunks[0]['document_id']}"
            )

            print(
                f"    Ground-truth in Top-5: "
                f"{'YES' if ground_truth_in_top_k else 'NO'}"
            )

            print(
                f"    Answer length: "
                f"{len(answer)} characters"
            )

            print(
                f"    Generation time: "
                f"{generation_time:.2f}s"
            )

        except Exception as e:

            failed_count += 1

            print(
                f"    [ERROR] {evaluation_id}: {e}"
            )

            progress[
                evaluation_id
            ] = {

                "evaluation_id":
                    evaluation_id,

                "question":
                    question,

                "ground_truth_document_ids":
                    ground_truth_ids,

                "answer":
                    None,

                "status":
                    "failed",

                "error":
                    str(e),

                "updated_at":
                    datetime.now().isoformat()
            }

            save_json_atomic(

                {
                    "stage":
                        "end_to_end_rag_generation",

                    "status":
                        "in_progress",

                    "generator":
                        OLLAMA_MODEL,

                    "completed":
                        len([
                            x
                            for x in progress.values()
                            if x.get("status")
                            == "success"
                        ]),

                    "failed":
                        len([
                            x
                            for x in progress.values()
                            if x.get("status")
                            == "failed"
                        ]),

                    "results":
                        progress,

                    "updated_at":
                        datetime.now().isoformat()
                },

                PROGRESS_PATH
            )


except KeyboardInterrupt:

    print()
    print(
        "[WARNING] Generation interrupted."
    )

    print(
        "[INFO] Completed answers have been persisted."
    )

except Exception as e:

    print()
    print(
        f"[ERROR] End-to-end generation failed: {e}"
    )

    raise


# ------------------------------------------------------------
# 13. Build final evaluation artifact
# ------------------------------------------------------------

try:

    final_results = []

    for evaluation_item in evaluation_questions:

        evaluation_id = evaluation_item[
            "evaluation_id"
        ]

        result = progress.get(
            evaluation_id
        )

        if not result:

            raise ValueError(
                f"Missing result for {evaluation_id}."
            )

        if result.get(
            "status"
        ) != "success":

            raise ValueError(
                f"{evaluation_id} did not complete successfully."
            )

        if not result.get(
            "answer"
        ):

            raise ValueError(
                f"{evaluation_id} has no answer."
            )

        final_results.append(
            result
        )

    if len(final_results) != len(
        evaluation_questions
    ):

        raise ValueError(
            "Final result count does not match "
            "evaluation question count."
        )

    # --------------------------------------------------------
    # Aggregate statistics
    # --------------------------------------------------------

    generation_times = [

        item[
            "generation_time_seconds"
        ]

        for item in final_results
    ]

    answer_lengths = [

        len(
            item["answer"]
        )

        for item in final_results
    ]

    top_k_ground_truth_hits = sum(

        1

        for item in final_results

        if item[
            "ground_truth_in_top_k"
        ]
    )

    aggregate = {

        "total_questions":
            len(final_results),

        "successful_answers":
            len(final_results),

        "failed_answers":
            0,

        "ground_truth_in_top_k":
            top_k_ground_truth_hits,

        "ground_truth_top_k_rate":
            round(
                top_k_ground_truth_hits
                / len(final_results),
                4
            ),

        "average_generation_time_seconds":
            round(
                float(
                    np.mean(
                        generation_times
                    )
                ),
                4
            ),

        "total_generation_time_seconds":
            round(
                float(
                    np.sum(
                        generation_times
                    )
                ),
                4
            ),

        "average_answer_length_characters":
            round(
                float(
                    np.mean(
                        answer_lengths
                    )
                ),
                2
            ),

        "minimum_answer_length_characters":
            int(
                np.min(
                    answer_lengths
                )
            ),

        "maximum_answer_length_characters":
            int(
                np.max(
                    answer_lengths
                )
            )
    }

    # --------------------------------------------------------
    # Final artifact
    # --------------------------------------------------------

    end_to_end_evaluation = {

        "stage":
            "end_to_end_rag_answer_generation",

        "status":
            "success",

        "evaluation_dataset":
            "semantic_evaluation_questions_v2.json",

        "evaluation_question_count":
            len(final_results),

        "retrieval_method":
            "dense",

        "embedding_model":
            embedding_model_name,

        "embedding_dimension":
            embedding_dimension,

        "index_type":
            type(index).__name__,

        "similarity_metric":
            "inner_product",

        "normalized_embeddings":
            True,

        "top_k":
            TOP_K,

        "generator":
            OLLAMA_MODEL,

        "temperature":
            TEMPERATURE,

        "num_predict":
            NUM_PREDICT,

        "aggregate":
            aggregate,

        "results":
            final_results,

        "created_at":
            datetime.now().isoformat()
    }

    # --------------------------------------------------------
    # Save final artifacts
    # --------------------------------------------------------

    save_json_atomic(
        end_to_end_evaluation,
        RESULT_PATH
    )

    save_json_atomic(
        end_to_end_evaluation,
        CHECKPOINT_PATH
    )

    save_json_atomic(

        {
            "stage":
                "end_to_end_rag_generation",

            "status":
                "success",

            "generator":
                OLLAMA_MODEL,

            "completed":
                len(final_results),

            "failed":
                0,

            "results":
                progress,

            "updated_at":
                datetime.now().isoformat()
        },

        PROGRESS_PATH
    )

    # --------------------------------------------------------
    # Final output
    # --------------------------------------------------------

    print()
    print("=" * 60)
    print("END-TO-END RAG GENERATION RESULTS")
    print("=" * 60)

    print(
        f"Questions          : "
        f"{aggregate['total_questions']}"
    )

    print(
        f"Successful answers : "
        f"{aggregate['successful_answers']}"
    )

    print(
        f"Failed answers     : "
        f"{aggregate['failed_answers']}"
    )

    print(
        f"Ground truth Top-5 : "
        f"{aggregate['ground_truth_in_top_k']}/"
        f"{aggregate['total_questions']}"
    )

    print(
        f"Top-5 coverage     : "
        f"{aggregate['ground_truth_top_k_rate']:.4f}"
    )

    print(
        f"Average answer len : "
        f"{aggregate['average_answer_length_characters']:.2f} chars"
    )

    print(
        f"Average generation : "
        f"{aggregate['average_generation_time_seconds']:.4f}s"
    )

    print(
        f"Total generation   : "
        f"{aggregate['total_generation_time_seconds']:.4f}s"
    )

    print()
    print("-" * 60)
    print("SAMPLE GENERATED ANSWERS")
    print("-" * 60)

    for item in final_results[:5]:

        evaluation_id = item[
            "evaluation_id"
        ]

        question = item[
            "question"
        ]

        top_retrieved = item[
            "retrieved_document_ids"
        ][0]

        answer = item[
            "answer"
        ]

        print()
        print(
            f"{evaluation_id}: {question}"
        )

        print(
            f"Top retrieved: {top_retrieved}"
        )

        print(
            f"Answer: {answer}"
        )

    print()
    print("=" * 60)
    print(
        "[SUCCESS] END-TO-END RAG GENERATION COMPLETE"
    )
    print("=" * 60)

    print()
    print("Saved:")

    print(
        f"  Results    : {RESULT_PATH}"
    )

    print(
        f"  Checkpoint : {CHECKPOINT_PATH}"
    )

    print(
        f"  Progress   : {PROGRESS_PATH}"
    )

except Exception as e:

    print()
    print(
        f"[ERROR] Failed to finalize "
        f"end-to-end evaluation: {e}"
    )

    raise


# ------------------------------------------------------------
# 14. Release GPU memory
# ------------------------------------------------------------

try:

    del embedding_model

except Exception:

    pass

gc.collect()

try:

    import torch

    if torch.cuda.is_available():

        torch.cuda.empty_cache()

except Exception:

    pass

print()
print(
    "[INFO] Embedding model memory released."
)

END-TO-END RAG ANSWER GENERATION EVALUATION
[SUCCESS] Loaded 30 evaluation questions.
[SUCCESS] Loaded 164 chunks.
[SUCCESS] FAISS index loaded: 164 vectors.
[INFO] Embedding model: BAAI/bge-small-en-v1.5
[INFO] Loading embedding model: BAAI/bge-small-en-v1.5


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9026.64it/s]


[SUCCESS] Embedding model loaded.
[INFO] Dimension: 384
[SUCCESS] Chunk lookup created for 164 chunks.

------------------------------------------------------------
GENERATING END-TO-END RAG ANSWERS
------------------------------------------------------------

[01/30] semantic_eval_001
    Question: How can I determine if my keyboard is not working properly?
    Top retrieved: doc_084
    Ground-truth in Top-5: YES
    Answer length: 944 characters
    Generation time: 10.68s

[02/30] semantic_eval_002
    Question: How do I set up a new user's account in JIRA with the correct permissions and access?
    Top retrieved: doc_054
    Ground-truth in Top-5: YES
    Answer length: 1251 characters
    Generation time: 10.80s

[03/30] semantic_eval_003
    Question: How do I securely connect to my company printer using my credentials and network settings?
    Top retrieved: doc_071
    Ground-truth in Top-5: YES
    Answer length: 947 characters
    Generation time: 7.79s

[04/30] semantic_ev

In [34]:
# ============================================================
# CELL 26 — RAG ANSWER-QUALITY EVALUATION
# ============================================================

import os
import json
import time
import gc
import re
import requests
import numpy as np
from datetime import datetime


# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

PROJECT_ROOT = r"E:\rag"

RAG_RESULTS_PATH = os.path.join(
    PROJECT_ROOT,
    "data",
    "evaluation",
    "end_to_end_rag_evaluation_v2.json"
)

DOCUMENTS_PATH = os.path.join(
    PROJECT_ROOT,
    "data",
    "processed",
    "rag_documents.json"
)

RESULT_PATH = os.path.join(
    PROJECT_ROOT,
    "data",
    "evaluation",
    "rag_answer_quality_evaluation_v2.json"
)

CHECKPOINT_PATH = os.path.join(
    PROJECT_ROOT,
    "checkpoints",
    "rag_answer_quality_evaluation_v2.json"
)

PROGRESS_PATH = os.path.join(
    PROJECT_ROOT,
    "data",
    "evaluation",
    "rag_answer_quality_progress_v2.json"
)

OLLAMA_URL = "http://localhost:11434"
OLLAMA_GENERATE_URL = f"{OLLAMA_URL}/api/generate"

JUDGE_MODEL = "qwen3:8b"

TEMPERATURE = 0.0
NUM_PREDICT = 512

REQUEST_TIMEOUT = 180
MAX_RETRIES = 3


# ------------------------------------------------------------
# 2. Atomic JSON save
# ------------------------------------------------------------

def save_json_atomic(data, path):

    temp_path = path + ".tmp"

    try:

        os.makedirs(
            os.path.dirname(path),
            exist_ok=True
        )

        with open(
            temp_path,
            "w",
            encoding="utf-8"
        ) as f:

            json.dump(
                data,
                f,
                indent=2,
                ensure_ascii=False
            )

        os.replace(
            temp_path,
            path
        )

    except Exception:

        if os.path.exists(temp_path):

            try:
                os.remove(temp_path)
            except Exception:
                pass

        raise


# ------------------------------------------------------------
# 3. Load RAG generation results
# ------------------------------------------------------------

try:

    print("=" * 60)
    print("RAG ANSWER-QUALITY EVALUATION")
    print("=" * 60)

    if not os.path.exists(
        RAG_RESULTS_PATH
    ):

        raise FileNotFoundError(
            f"RAG evaluation results not found:\n"
            f"{RAG_RESULTS_PATH}"
        )

    with open(
        RAG_RESULTS_PATH,
        "r",
        encoding="utf-8"
    ) as f:

        rag_data = json.load(f)

    rag_results = rag_data.get(
        "results"
    )

    if not isinstance(
        rag_results,
        list
    ):

        raise ValueError(
            "Invalid RAG evaluation structure."
        )

    if len(rag_results) != 30:

        raise ValueError(
            f"Expected 30 RAG results, "
            f"found {len(rag_results)}."
        )

    for item in rag_results:

        if not item.get(
            "question"
        ):

            raise ValueError(
                "RAG result contains empty question."
            )

        if not item.get(
            "answer"
        ):

            raise ValueError(
                "RAG result contains empty answer."
            )

    print(
        f"[SUCCESS] Loaded {len(rag_results)} "
        f"generated RAG answers."
    )

except Exception as e:

    print(
        f"[ERROR] Failed to load RAG results: {e}"
    )

    raise


# ------------------------------------------------------------
# 4. Load authoritative source documents
# ------------------------------------------------------------

try:

    if not os.path.exists(
        DOCUMENTS_PATH
    ):

        raise FileNotFoundError(
            f"Processed documents not found:\n"
            f"{DOCUMENTS_PATH}"
        )

    with open(
        DOCUMENTS_PATH,
        "r",
        encoding="utf-8"
    ) as f:

        documents_data = json.load(f)

    if isinstance(
        documents_data,
        list
    ):

        documents = documents_data

    elif isinstance(
        documents_data,
        dict
    ):

        documents = documents_data.get(
            "documents"
        )

    else:

        raise ValueError(
            "Unsupported document JSON structure."
        )

    if not isinstance(
        documents,
        list
    ):

        raise ValueError(
            "Could not extract documents."
        )

    if len(documents) != 100:

        raise ValueError(
            f"Expected 100 documents, "
            f"found {len(documents)}."
        )

    document_by_id = {}

    for document in documents:

        document_id = document.get(
            "document_id"
        )

        if not document_id:

            raise ValueError(
                "Document missing document_id."
            )

        document_by_id[
            document_id
        ] = document

    if len(document_by_id) != len(documents):

        raise ValueError(
            "Duplicate document IDs detected."
        )

    print(
        f"[SUCCESS] Loaded {len(documents)} "
        f"authoritative source documents."
    )

except Exception as e:

    print(
        f"[ERROR] Failed to load source documents: {e}"
    )

    raise


# ------------------------------------------------------------
# 5. Prepare ground-truth source for each evaluation item
# ------------------------------------------------------------

try:

    prepared_results = []

    for item in rag_results:

        ground_truth_ids = item.get(
            "ground_truth_document_ids",
            []
        )

        if not ground_truth_ids:

            raise ValueError(
                f"No ground-truth document for "
                f"{item['evaluation_id']}."
            )

        ground_truth_documents = []

        for document_id in ground_truth_ids:

            document = document_by_id.get(
                document_id
            )

            if document is None:

                raise ValueError(
                    f"Ground-truth document "
                    f"{document_id} not found."
                )

            ground_truth_documents.append(
                {
                    "document_id":
                        document_id,

                    "topic":
                        document.get(
                            "metadata",
                            {}
                        ).get(
                            "topic",
                            ""
                        ),

                    "text":
                        document.get(
                            "text",
                            ""
                        )
                }
            )

        prepared_item = dict(item)

        prepared_item[
            "ground_truth_documents"
        ] = ground_truth_documents

        prepared_results.append(
            prepared_item
        )

    print(
        "[SUCCESS] Ground-truth source evidence "
        "attached to all evaluation items."
    )

except Exception as e:

    print(
        f"[ERROR] Failed to prepare ground truth: {e}"
    )

    raise


# ------------------------------------------------------------
# 6. Build evaluator prompt
# ------------------------------------------------------------

def build_judge_prompt(
    question,
    answer,
    retrieved_chunks,
    ground_truth_documents
):

    retrieved_sections = []

    for item in retrieved_chunks:

        retrieved_sections.append(
            (
                f"[Retrieved Context {item['rank']}]\n"
                f"Document ID: {item['document_id']}\n"
                f"Topic: {item['topic']}\n"
                f"Content:\n{item['text']}"
            )
        )

    retrieved_context = "\n\n".join(
        retrieved_sections
    )

    ground_truth_sections = []

    for document in ground_truth_documents:

        ground_truth_sections.append(
            (
                f"[Ground-Truth Source]\n"
                f"Document ID: {document['document_id']}\n"
                f"Topic: {document['topic']}\n"
                f"Content:\n{document['text']}"
            )
        )

    ground_truth_context = "\n\n".join(
        ground_truth_sections
    )

    prompt = f"""
You are an evaluator for a Retrieval-Augmented Generation (RAG) system.

Your task is to evaluate the generated answer using the supplied evidence.

IMPORTANT:
- Do not use outside knowledge.
- Judge only what is supported by the supplied source material.
- The Ground-Truth Source is the authoritative reference for correctness.
- Retrieved Context represents the evidence that was actually available to the RAG generator.
- Faithfulness means that the generated answer is supported by the Retrieved Context.
- Correctness means that the answer correctly addresses the user's question according to the Ground-Truth Source.
- Relevance means that the answer directly addresses the question without unnecessary unrelated information.
- Completeness means that the answer covers the important information needed to answer the question.
- Do not reward an answer merely because it sounds plausible.
- Do not penalize harmless paraphrasing.
- Do not require exact wording from the source.
- Identify claims that are not supported by the Retrieved Context.

SCORING SCALE:

1 = Very poor
2 = Poor
3 = Adequate
4 = Good
5 = Excellent

Evaluate these four dimensions:

1. Correctness
2. Relevance
3. Faithfulness
4. Completeness

Also determine:

- unsupported_claims: list every meaningful factual claim in the answer that is not supported by the Retrieved Context.
- hallucination: true if there is at least one meaningful unsupported factual claim; otherwise false.
- overall_score: the average of the four dimension scores.

USER QUESTION:
{question}

GENERATED ANSWER:
{answer}

RETRIEVED CONTEXT:
{retrieved_context}

GROUND-TRUTH SOURCE:
{ground_truth_context}

Return ONLY valid JSON.

Required JSON structure:

{{
  "correctness": {{
    "score": 1,
    "reason": "brief explanation"
  }},
  "relevance": {{
    "score": 1,
    "reason": "brief explanation"
  }},
  "faithfulness": {{
    "score": 1,
    "reason": "brief explanation"
  }},
  "completeness": {{
    "score": 1,
    "reason": "brief explanation"
  }},
  "unsupported_claims": [],
  "hallucination": false,
  "overall_score": 1.0,
  "overall_reason": "brief explanation"
}}
"""

    return prompt


# ------------------------------------------------------------
# 7. Call Ollama evaluator
# ------------------------------------------------------------

def call_judge(
    prompt
):

    payload = {

        "model":
            JUDGE_MODEL,

        "prompt":
            prompt,

        "stream":
            False,

        "think":
            False,

        "format":
            "json",

        "options":
            {
                "temperature":
                    TEMPERATURE,

                "num_predict":
                    NUM_PREDICT
            }
    }

    last_error = None

    for attempt in range(
        1,
        MAX_RETRIES + 1
    ):

        try:

            response = requests.post(
                OLLAMA_GENERATE_URL,
                json=payload,
                timeout=REQUEST_TIMEOUT
            )

            response.raise_for_status()

            result = response.json()

            raw_response = str(
                result.get(
                    "response",
                    ""
                )
            ).strip()

            if not raw_response:

                raise ValueError(
                    "Judge returned empty response."
                )

            # ------------------------------------------------
            # Parse JSON
            # ------------------------------------------------

            try:

                evaluation = json.loads(
                    raw_response
                )

            except json.JSONDecodeError:

                # Attempt to extract JSON object
                match = re.search(
                    r"\{.*\}",
                    raw_response,
                    re.DOTALL
                )

                if not match:

                    raise ValueError(
                        "Judge response was not valid JSON."
                    )

                evaluation = json.loads(
                    match.group(0)
                )

            return evaluation

        except Exception as e:

            last_error = e

            print(
                f"    [WARNING] Judge attempt "
                f"{attempt}/{MAX_RETRIES} failed: {e}"
            )

            if attempt < MAX_RETRIES:

                time.sleep(2)

    raise RuntimeError(
        f"Judge failed after {MAX_RETRIES} attempts: "
        f"{last_error}"
    )


# ------------------------------------------------------------
# 8. Validate judge output
# ------------------------------------------------------------

def validate_judge_output(
    evaluation
):

    required_dimensions = [
        "correctness",
        "relevance",
        "faithfulness",
        "completeness"
    ]

    for dimension in required_dimensions:

        if dimension not in evaluation:

            raise ValueError(
                f"Missing dimension: {dimension}"
            )

        dimension_data = evaluation[
            dimension
        ]

        if not isinstance(
            dimension_data,
            dict
        ):

            raise ValueError(
                f"{dimension} must be an object."
            )

        if "score" not in dimension_data:

            raise ValueError(
                f"{dimension} missing score."
            )

        score = dimension_data[
            "score"
        ]

        if isinstance(
            score,
            bool
        ):

            raise ValueError(
                f"{dimension} score cannot be boolean."
            )

        score = float(score)

        if score < 1 or score > 5:

            raise ValueError(
                f"{dimension} score outside 1-5."
            )

    unsupported_claims = evaluation.get(
        "unsupported_claims",
        []
    )

    if not isinstance(
        unsupported_claims,
        list
    ):

        raise ValueError(
            "unsupported_claims must be a list."
        )

    hallucination = evaluation.get(
        "hallucination"
    )

    if not isinstance(
        hallucination,
        bool
    ):

        raise ValueError(
            "hallucination must be boolean."
        )

    calculated_overall = round(

        (
            float(
                evaluation["correctness"]["score"]
            )
            +
            float(
                evaluation["relevance"]["score"]
            )
            +
            float(
                evaluation["faithfulness"]["score"]
            )
            +
            float(
                evaluation["completeness"]["score"]
            )
        )
        / 4,

        2
    )

    # Use our deterministic calculation
    # instead of trusting the LLM's arithmetic.

    evaluation[
        "overall_score"
    ] = calculated_overall

    return evaluation


# ------------------------------------------------------------
# 9. Load existing progress
# ------------------------------------------------------------

progress = {}

if os.path.exists(
    PROGRESS_PATH
):

    try:

        with open(
            PROGRESS_PATH,
            "r",
            encoding="utf-8"
        ) as f:

            progress_data = json.load(f)

        if isinstance(
            progress_data,
            dict
        ):

            progress = progress_data

        print(
            f"[INFO] Existing judge progress loaded: "
            f"{len(progress)} records."
        )

    except Exception as e:

        print(
            f"[WARNING] Could not load judge progress: {e}"
        )

        progress = {}


# ------------------------------------------------------------
# 10. Evaluate every RAG answer
# ------------------------------------------------------------

print()
print("-" * 60)
print("EVALUATING RAG ANSWERS")
print("-" * 60)

successful_count = 0
failed_count = 0

try:

    for number, item in enumerate(
        prepared_results,
        start=1
    ):

        evaluation_id = item[
            "evaluation_id"
        ]

        # ----------------------------------------------------
        # Resume completed evaluation
        # ----------------------------------------------------

        if evaluation_id in progress:

            existing = progress[
                evaluation_id
            ]

            if (
                existing.get("status")
                == "success"
                and existing.get("evaluation")
            ):

                successful_count += 1

                print(
                    f"[SKIP] {evaluation_id} "
                    f"already evaluated."
                )

                continue

        question = item[
            "question"
        ]

        answer = item[
            "answer"
        ]

        retrieved_chunks = item[
            "retrieved_chunks"
        ]

        ground_truth_documents = item[
            "ground_truth_documents"
        ]

        print()
        print(
            f"[{number:02d}/"
            f"{len(prepared_results):02d}] "
            f"{evaluation_id}"
        )

        print(
            f"    Question: {question}"
        )

        try:

            # ------------------------------------------------
            # Build evaluation prompt
            # ------------------------------------------------

            prompt = build_judge_prompt(
                question,
                answer,
                retrieved_chunks,
                ground_truth_documents
            )

            # ------------------------------------------------
            # Evaluate
            # ------------------------------------------------

            evaluation_start = time.time()

            evaluation = call_judge(
                prompt
            )

            evaluation_time = (
                time.time()
                - evaluation_start
            )

            # ------------------------------------------------
            # Validate
            # ------------------------------------------------

            evaluation = validate_judge_output(
                evaluation
            )

            unsupported_claims = evaluation[
                "unsupported_claims"
            ]

            hallucination = evaluation[
                "hallucination"
            ]

            overall_score = evaluation[
                "overall_score"
            ]

            # ------------------------------------------------
            # Store result
            # ------------------------------------------------

            progress[
                evaluation_id
            ] = {

                "evaluation_id":
                    evaluation_id,

                "question":
                    question,

                "ground_truth_document_ids":
                    item[
                        "ground_truth_document_ids"
                    ],

                "retrieved_document_ids":
                    item[
                        "retrieved_document_ids"
                    ],

                "ground_truth_in_top_k":
                    item[
                        "ground_truth_in_top_k"
                    ],

                "answer":
                    answer,

                "evaluation":
                    evaluation,

                "judge_model":
                    JUDGE_MODEL,

                "judge_temperature":
                    TEMPERATURE,

                "evaluation_time_seconds":
                    round(
                        evaluation_time,
                        4
                    ),

                "status":
                    "success",

                "evaluated_at":
                    datetime.now().isoformat()
            }

            # ------------------------------------------------
            # Save after every question
            # ------------------------------------------------

            completed_now = len([

                x

                for x in progress.values()

                if x.get("status")
                == "success"
            ])

            failed_now = len([

                x

                for x in progress.values()

                if x.get("status")
                == "failed"
            ])

            save_json_atomic(

                {
                    "stage":
                        "rag_answer_quality_evaluation",

                    "status":
                        "in_progress",

                    "judge_model":
                        JUDGE_MODEL,

                    "completed":
                        completed_now,

                    "failed":
                        failed_now,

                    "results":
                        progress,

                    "updated_at":
                        datetime.now().isoformat()
                },

                PROGRESS_PATH
            )

            successful_count += 1

            print(
                f"    Correctness : "
                f"{evaluation['correctness']['score']}/5"
            )

            print(
                f"    Relevance   : "
                f"{evaluation['relevance']['score']}/5"
            )

            print(
                f"    Faithfulness: "
                f"{evaluation['faithfulness']['score']}/5"
            )

            print(
                f"    Completeness: "
                f"{evaluation['completeness']['score']}/5"
            )

            print(
                f"    Overall     : "
                f"{overall_score:.2f}/5"
            )

            print(
                f"    Hallucination: "
                f"{'YES' if hallucination else 'NO'}"
            )

            print(
                f"    Unsupported claims: "
                f"{len(unsupported_claims)}"
            )

            print(
                f"    Evaluation time: "
                f"{evaluation_time:.2f}s"
            )

        except Exception as e:

            failed_count += 1

            print(
                f"    [ERROR] {evaluation_id}: {e}"
            )

            progress[
                evaluation_id
            ] = {

                "evaluation_id":
                    evaluation_id,

                "question":
                    question,

                "answer":
                    answer,

                "status":
                    "failed",

                "error":
                    str(e),

                "evaluated_at":
                    datetime.now().isoformat()
            }

            save_json_atomic(

                {
                    "stage":
                        "rag_answer_quality_evaluation",

                    "status":
                        "in_progress",

                    "judge_model":
                        JUDGE_MODEL,

                    "completed":
                        len([
                            x
                            for x in progress.values()
                            if x.get("status")
                            == "success"
                        ]),

                    "failed":
                        len([
                            x
                            for x in progress.values()
                            if x.get("status")
                            == "failed"
                        ]),

                    "results":
                        progress,

                    "updated_at":
                        datetime.now().isoformat()
                },

                PROGRESS_PATH
            )


except KeyboardInterrupt:

    print()
    print(
        "[WARNING] Evaluation interrupted."
    )

    print(
        "[INFO] Completed evaluations "
        "have been persisted."
    )


# ------------------------------------------------------------
# 11. Build final evaluation results
# ------------------------------------------------------------

try:

    final_results = []

    for item in prepared_results:

        evaluation_id = item[
            "evaluation_id"
        ]

        result = progress.get(
            evaluation_id
        )

        if not result:

            raise ValueError(
                f"Missing evaluation for "
                f"{evaluation_id}."
            )

        if result.get(
            "status"
        ) != "success":

            raise ValueError(
                f"{evaluation_id} failed evaluation."
            )

        final_results.append(
            result
        )

    if len(final_results) != 30:

        raise ValueError(
            f"Expected 30 final evaluations, "
            f"found {len(final_results)}."
        )

    # --------------------------------------------------------
    # Extract scores
    # --------------------------------------------------------

    correctness_scores = [

        float(
            item[
                "evaluation"
            ][
                "correctness"
            ][
                "score"
            ]
        )

        for item in final_results
    ]

    relevance_scores = [

        float(
            item[
                "evaluation"
            ][
                "relevance"
            ][
                "score"
            ]
        )

        for item in final_results
    ]

    faithfulness_scores = [

        float(
            item[
                "evaluation"
            ][
                "faithfulness"
            ][
                "score"
            ]
        )

        for item in final_results
    ]

    completeness_scores = [

        float(
            item[
                "evaluation"
            ][
                "completeness"
            ][
                "score"
            ]
        )

        for item in final_results
    ]

    overall_scores = [

        float(
            item[
                "evaluation"
            ][
                "overall_score"
            ]
        )

        for item in final_results
    ]

    hallucination_count = sum(

        1

        for item in final_results

        if item[
            "evaluation"
        ][
            "hallucination"
        ]
    )

    unsupported_claim_count = sum(

        len(
            item[
                "evaluation"
            ][
                "unsupported_claims"
            ]
        )

        for item in final_results
    )

    # --------------------------------------------------------
    # Score distributions
    # --------------------------------------------------------

    def score_distribution(
        scores
    ):

        return {

            "1": int(
                sum(
                    score == 1
                    for score in scores
                )
            ),

            "2": int(
                sum(
                    score == 2
                    for score in scores
                )
            ),

            "3": int(
                sum(
                    score == 3
                    for score in scores
                )
            ),

            "4": int(
                sum(
                    score == 4
                    for score in scores
                )
            ),

            "5": int(
                sum(
                    score == 5
                    for score in scores
                )
            )
        }


    # --------------------------------------------------------
    # Aggregate metrics
    # --------------------------------------------------------

    aggregate = {

        "total_questions":
            len(final_results),

        "successful_evaluations":
            len(final_results),

        "failed_evaluations":
            0,

        "mean_correctness":
            round(
                float(
                    np.mean(
                        correctness_scores
                    )
                ),
                3
            ),

        "mean_relevance":
            round(
                float(
                    np.mean(
                        relevance_scores
                    )
                ),
                3
            ),

        "mean_faithfulness":
            round(
                float(
                    np.mean(
                        faithfulness_scores
                    )
                ),
                3
            ),

        "mean_completeness":
            round(
                float(
                    np.mean(
                        completeness_scores
                    )
                ),
                3
            ),

        "mean_overall_score":
            round(
                float(
                    np.mean(
                        overall_scores
                    )
                ),
                3
            ),

        "overall_score_percentage":
            round(
                float(
                    np.mean(
                        overall_scores
                    )
                )
                / 5
                * 100,
                2
            ),

        "hallucination_count":
            hallucination_count,

        "hallucination_rate":
            round(
                hallucination_count
                / len(final_results),
                4
            ),

        "unsupported_claim_count":
            unsupported_claim_count,

        "average_unsupported_claims_per_answer":
            round(
                unsupported_claim_count
                / len(final_results),
                3
            ),

        "correctness_distribution":
            score_distribution(
                correctness_scores
            ),

        "relevance_distribution":
            score_distribution(
                relevance_scores
            ),

        "faithfulness_distribution":
            score_distribution(
                faithfulness_scores
            ),

        "completeness_distribution":
            score_distribution(
                completeness_scores
            ),

        "overall_distribution":
            score_distribution(
                overall_scores
            )
    }


    # --------------------------------------------------------
    # Final artifact
    # --------------------------------------------------------

    answer_quality_evaluation = {

        "stage":
            "rag_answer_quality_evaluation",

        "status":
            "success",

        "evaluation_dataset":
            "semantic_evaluation_questions_v2.json",

        "question_count":
            len(final_results),

        "retrieval_method":
            "dense",

        "embedding_model":
            "BAAI/bge-small-en-v1.5",

        "top_k":
            5,

        "generator_model":
            "qwen3:8b",

        "judge_model":
            JUDGE_MODEL,

        "judge_temperature":
            TEMPERATURE,

        "scoring_scale":
            "1-5",

        "evaluation_dimensions":
            [
                "correctness",
                "relevance",
                "faithfulness",
                "completeness"
            ],

        "aggregate":
            aggregate,

        "results":
            final_results,

        "created_at":
            datetime.now().isoformat()
    }


    # --------------------------------------------------------
    # Save final artifacts
    # --------------------------------------------------------

    save_json_atomic(
        answer_quality_evaluation,
        RESULT_PATH
    )

    save_json_atomic(
        answer_quality_evaluation,
        CHECKPOINT_PATH
    )

    save_json_atomic(

        {
            "stage":
                "rag_answer_quality_evaluation",

            "status":
                "success",

            "judge_model":
                JUDGE_MODEL,

            "completed":
                len(final_results),

            "failed":
                0,

            "results":
                progress,

            "updated_at":
                datetime.now().isoformat()
        },

        PROGRESS_PATH
    )


    # --------------------------------------------------------
    # Final output
    # --------------------------------------------------------

    print()
    print("=" * 60)
    print("RAG ANSWER-QUALITY EVALUATION RESULTS")
    print("=" * 60)

    print(
        f"Questions       : "
        f"{aggregate['total_questions']}"
    )

    print(
        f"Correctness     : "
        f"{aggregate['mean_correctness']:.3f}/5"
    )

    print(
        f"Relevance       : "
        f"{aggregate['mean_relevance']:.3f}/5"
    )

    print(
        f"Faithfulness    : "
        f"{aggregate['mean_faithfulness']:.3f}/5"
    )

    print(
        f"Completeness    : "
        f"{aggregate['mean_completeness']:.3f}/5"
    )

    print(
        f"Overall score   : "
        f"{aggregate['mean_overall_score']:.3f}/5"
    )

    print(
        f"Overall percent : "
        f"{aggregate['overall_score_percentage']:.2f}%"
    )

    print()
    print(
        f"Hallucinations  : "
        f"{aggregate['hallucination_count']}/"
        f"{aggregate['total_questions']}"
    )

    print(
        f"Hallucination rate: "
        f"{aggregate['hallucination_rate']:.4f}"
    )

    print(
        f"Unsupported claims: "
        f"{aggregate['unsupported_claim_count']}"
    )

    print()
    print("-" * 60)
    print("SCORE DISTRIBUTIONS")
    print("-" * 60)

    print(
        "Correctness  : "
        f"{aggregate['correctness_distribution']}"
    )

    print(
        "Relevance    : "
        f"{aggregate['relevance_distribution']}"
    )

    print(
        "Faithfulness : "
        f"{aggregate['faithfulness_distribution']}"
    )

    print(
        "Completeness : "
        f"{aggregate['completeness_distribution']}"
    )

    print(
        "Overall      : "
        f"{aggregate['overall_distribution']}"
    )

    print()
    print("=" * 60)
    print(
        "[SUCCESS] RAG ANSWER-QUALITY EVALUATION COMPLETE"
    )
    print("=" * 60)

    print()
    print("Saved:")

    print(
        f"  Results    : {RESULT_PATH}"
    )

    print(
        f"  Checkpoint : {CHECKPOINT_PATH}"
    )

    print(
        f"  Progress   : {PROGRESS_PATH}"
    )


except Exception as e:

    print()
    print(
        f"[ERROR] Failed to finalize answer-quality "
        f"evaluation: {e}"
    )

    raise


# ------------------------------------------------------------
# 12. Release memory
# ------------------------------------------------------------

del prepared_results
del documents
del document_by_id

gc.collect()

try:

    import torch

    if torch.cuda.is_available():

        torch.cuda.empty_cache()

except Exception:

    pass

print()
print(
    "[INFO] Evaluation memory released."
)

RAG ANSWER-QUALITY EVALUATION
[SUCCESS] Loaded 30 generated RAG answers.
[SUCCESS] Loaded 100 authoritative source documents.
[SUCCESS] Ground-truth source evidence attached to all evaluation items.

------------------------------------------------------------
EVALUATING RAG ANSWERS
------------------------------------------------------------

[01/30] semantic_eval_001
    Question: How can I determine if my keyboard is not working properly?
    Correctness : 4/5
    Relevance   : 5/5
    Faithfulness: 4/5
    Completeness: 4/5
    Overall     : 4.25/5
    Hallucination: YES
    Unsupported claims: 2
    Evaluation time: 13.79s

[02/30] semantic_eval_002
    Question: How do I set up a new user's account in JIRA with the correct permissions and access?
    Correctness : 4/5
    Relevance   : 5/5
    Faithfulness: 4/5
    Completeness: 3/5
    Overall     : 4.00/5
    Hallucination: YES
    Unsupported claims: 1
    Evaluation time: 13.47s

[03/30] semantic_eval_003
    Question: How do

In [35]:
import os
import json
import gc
import numpy as np
from datetime import datetime


# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

PROJECT_ROOT = r"E:\rag"

ANSWER_QUALITY_PATH = os.path.join(
    PROJECT_ROOT,
    "data",
    "evaluation",
    "rag_answer_quality_evaluation_v2.json"
)

RESULT_PATH = os.path.join(
    PROJECT_ROOT,
    "data",
    "evaluation",
    "rag_failure_analysis_v2.json"
)

CHECKPOINT_PATH = os.path.join(
    PROJECT_ROOT,
    "checkpoints",
    "rag_failure_analysis_v2.json"
)


# ------------------------------------------------------------
# 2. Atomic save
# ------------------------------------------------------------

def save_json_atomic(data, path):

    temp_path = path + ".tmp"

    try:

        os.makedirs(
            os.path.dirname(path),
            exist_ok=True
        )

        with open(
            temp_path,
            "w",
            encoding="utf-8"
        ) as f:

            json.dump(
                data,
                f,
                indent=2,
                ensure_ascii=False
            )

        os.replace(
            temp_path,
            path
        )

    except Exception:

        if os.path.exists(temp_path):

            try:
                os.remove(temp_path)
            except Exception:
                pass

        raise


# ------------------------------------------------------------
# 3. Load answer-quality evaluation
# ------------------------------------------------------------

try:

    print("=" * 60)
    print("DETAILED RAG FAILURE ANALYSIS")
    print("=" * 60)

    if not os.path.exists(
        ANSWER_QUALITY_PATH
    ):

        raise FileNotFoundError(
            f"Answer-quality evaluation not found:\n"
            f"{ANSWER_QUALITY_PATH}"
        )

    with open(
        ANSWER_QUALITY_PATH,
        "r",
        encoding="utf-8"
    ) as f:

        quality_data = json.load(f)

    results = quality_data.get(
        "results"
    )

    if not isinstance(
        results,
        list
    ):

        raise ValueError(
            "Invalid answer-quality evaluation structure."
        )

    if len(results) != 30:

        raise ValueError(
            f"Expected 30 results, "
            f"found {len(results)}."
        )

    print(
        f"[SUCCESS] Loaded {len(results)} "
        f"answer-quality evaluations."
    )

except Exception as e:

    print(
        f"[ERROR] Failed to load evaluation: {e}"
    )

    raise


# ------------------------------------------------------------
# 4. Helper functions
# ------------------------------------------------------------

def get_score(
    item,
    dimension
):

    try:

        return float(
            item[
                "evaluation"
            ][
                dimension
            ][
                "score"
            ]
        )

    except Exception:

        return None


def find_ground_truth_rank(
    item
):

    ground_truth_ids = set(
        item.get(
            "ground_truth_document_ids",
            []
        )
    )

    retrieved_ids = item.get(
        "retrieved_document_ids",
        []
    )

    for rank, document_id in enumerate(
        retrieved_ids,
        start=1
    ):

        if document_id in ground_truth_ids:

            return rank

    return None


# ------------------------------------------------------------
# 5. Analyze every evaluation item
# ------------------------------------------------------------

analysis_results = []

for item in results:

    correctness = get_score(
        item,
        "correctness"
    )

    relevance = get_score(
        item,
        "relevance"
    )

    faithfulness = get_score(
        item,
        "faithfulness"
    )

    completeness = get_score(
        item,
        "completeness"
    )

    overall = float(
        item[
            "evaluation"
        ][
            "overall_score"
        ]
    )

    hallucination = bool(
        item[
            "evaluation"
        ][
            "hallucination"
        ]
    )

    unsupported_claims = item[
        "evaluation"
    ].get(
        "unsupported_claims",
        []
    )

    ground_truth_rank = find_ground_truth_rank(
        item
    )

    top_retrieved = (
        item[
            "retrieved_document_ids"
        ][0]
        if item.get(
            "retrieved_document_ids"
        )
        else None
    )

    ground_truth_top1 = (
        ground_truth_rank == 1
    )

    ground_truth_in_top5 = (
        ground_truth_rank is not None
        and ground_truth_rank <= 5
    )

    # --------------------------------------------------------
    # Determine primary failure category
    # --------------------------------------------------------

    if (
        correctness <= 3
        and ground_truth_rank is None
    ):

        failure_category = (
            "retrieval_failure"
        )

    elif (
        correctness <= 3
        and ground_truth_rank is not None
        and ground_truth_rank > 1
    ):

        failure_category = (
            "retrieval_ranking_failure"
        )

    elif hallucination:

        failure_category = (
            "generation_faithfulness_failure"
        )

    elif completeness <= 3:

        failure_category = (
            "answer_completeness_failure"
        )

    elif correctness <= 4:

        failure_category = (
            "answer_correctness_issue"
        )

    elif relevance <= 4:

        failure_category = (
            "answer_relevance_issue"
        )

    else:

        failure_category = (
            "no_major_failure"
        )

    # --------------------------------------------------------
    # More detailed diagnostic tags
    # --------------------------------------------------------

    diagnostic_tags = []

    if ground_truth_rank is None:

        diagnostic_tags.append(
            "ground_truth_not_retrieved"
        )

    elif ground_truth_rank > 1:

        diagnostic_tags.append(
            "ground_truth_not_rank_1"
        )

    if hallucination:

        diagnostic_tags.append(
            "unsupported_claims"
        )

    if completeness <= 3:

        diagnostic_tags.append(
            "low_completeness"
        )

    if correctness <= 3:

        diagnostic_tags.append(
            "low_correctness"
        )

    if faithfulness <= 3:

        diagnostic_tags.append(
            "low_faithfulness"
        )

    if relevance <= 3:

        diagnostic_tags.append(
            "low_relevance"
        )

    analysis_results.append(

        {
            "evaluation_id":
                item[
                    "evaluation_id"
                ],

            "question":
                item[
                    "question"
                ],

            "answer":
                item[
                    "answer"
                ],

            "ground_truth_document_ids":
                item[
                    "ground_truth_document_ids"
                ],

            "retrieved_document_ids":
                item[
                    "retrieved_document_ids"
                ],

            "ground_truth_rank":
                ground_truth_rank,

            "ground_truth_top1":
                ground_truth_top1,

            "ground_truth_in_top5":
                ground_truth_in_top5,

            "top_retrieved_document":
                top_retrieved,

            "correctness":
                correctness,

            "relevance":
                relevance,

            "faithfulness":
                faithfulness,

            "completeness":
                completeness,

            "overall_score":
                overall,

            "hallucination":
                hallucination,

            "unsupported_claims":
                unsupported_claims,

            "unsupported_claim_count":
                len(
                    unsupported_claims
                ),

            "failure_category":
                failure_category,

            "diagnostic_tags":
                diagnostic_tags
        }
    )


# ------------------------------------------------------------
# 6. Identify important subsets
# ------------------------------------------------------------

hallucination_cases = [

    item

    for item in analysis_results

    if item[
        "hallucination"
    ]
]

low_completeness_cases = [

    item

    for item in analysis_results

    if item[
        "completeness"
    ] <= 3
]

low_correctness_cases = [

    item

    for item in analysis_results

    if item[
        "correctness"
    ] <= 3
]

low_faithfulness_cases = [

    item

    for item in analysis_results

    if item[
        "faithfulness"
    ] <= 3
]

ranking_failure_cases = [

    item

    for item in analysis_results

    if (
        item[
            "ground_truth_rank"
        ] is not None
        and
        item[
            "ground_truth_rank"
        ] > 1
    )
]

retrieval_failure_cases = [

    item

    for item in analysis_results

    if item[
        "ground_truth_rank"
    ] is None
]


# ------------------------------------------------------------
# 7. Failure-category counts
# ------------------------------------------------------------

category_counts = {}

for item in analysis_results:

    category = item[
        "failure_category"
    ]

    category_counts[
        category
    ] = (
        category_counts.get(
            category,
            0
        )
        + 1
    )


# ------------------------------------------------------------
# 8. Aggregate diagnostic statistics
# ------------------------------------------------------------

ground_truth_ranks = [

    item[
        "ground_truth_rank"
    ]

    for item in analysis_results

    if item[
        "ground_truth_rank"
    ] is not None
]

aggregate = {

    "total_cases":
        len(analysis_results),

    "hallucination_cases":
        len(hallucination_cases),

    "hallucination_rate":
        round(
            len(hallucination_cases)
            / len(analysis_results),
            4
        ),

    "unsupported_claims":
        sum(
            item[
                "unsupported_claim_count"
            ]

            for item in analysis_results
        ),

    "low_completeness_cases":
        len(low_completeness_cases),

    "low_correctness_cases":
        len(low_correctness_cases),

    "low_faithfulness_cases":
        len(low_faithfulness_cases),

    "ground_truth_retrieved_top5":
        len(
            [
                item
                for item in analysis_results
                if item[
                    "ground_truth_in_top5"
                ]
            ]
        ),

    "ground_truth_not_retrieved":
        len(retrieval_failure_cases),

    "ground_truth_not_rank1":
        len(ranking_failure_cases),

    "mean_ground_truth_rank":
        round(
            float(
                np.mean(
                    ground_truth_ranks
                )
            ),
            3
        )
        if ground_truth_ranks
        else None,

    "failure_category_counts":
        category_counts
}


# ------------------------------------------------------------
# 9. Build final artifact
# ------------------------------------------------------------

failure_analysis = {

    "stage":
        "rag_failure_analysis",

    "status":
        "success",

    "evaluation_source":
        "rag_answer_quality_evaluation_v2.json",

    "evaluation_count":
        len(analysis_results),

    "aggregate":
        aggregate,

    "hallucination_cases":
        hallucination_cases,

    "low_completeness_cases":
        low_completeness_cases,

    "low_correctness_cases":
        low_correctness_cases,

    "low_faithfulness_cases":
        low_faithfulness_cases,

    "ranking_failure_cases":
        ranking_failure_cases,

    "retrieval_failure_cases":
        retrieval_failure_cases,

    "all_cases":
        analysis_results,

    "created_at":
        datetime.now().isoformat()
}


# ------------------------------------------------------------
# 10. Save
# ------------------------------------------------------------

try:

    save_json_atomic(
        failure_analysis,
        RESULT_PATH
    )

    save_json_atomic(
        failure_analysis,
        CHECKPOINT_PATH
    )

except Exception as e:

    print(
        f"[ERROR] Failed to save failure analysis: {e}"
    )

    raise


# ------------------------------------------------------------
# 11. Display summary
# ------------------------------------------------------------

print()
print("=" * 60)
print("FAILURE ANALYSIS SUMMARY")
print("=" * 60)

print(
    f"Total cases             : "
    f"{aggregate['total_cases']}"
)

print(
    f"Hallucination cases     : "
    f"{aggregate['hallucination_cases']}"
)

print(
    f"Hallucination rate      : "
    f"{aggregate['hallucination_rate']:.4f}"
)

print(
    f"Unsupported claims      : "
    f"{aggregate['unsupported_claims']}"
)

print(
    f"Low completeness        : "
    f"{aggregate['low_completeness_cases']}"
)

print(
    f"Low correctness         : "
    f"{aggregate['low_correctness_cases']}"
)

print(
    f"Low faithfulness        : "
    f"{aggregate['low_faithfulness_cases']}"
)

print(
    f"Ground truth in Top-5  : "
    f"{aggregate['ground_truth_retrieved_top5']}/"
    f"{aggregate['total_cases']}"
)

print(
    f"Ground truth not found : "
    f"{aggregate['ground_truth_not_retrieved']}"
)

print(
    f"Ground truth not Top-1 : "
    f"{aggregate['ground_truth_not_rank1']}"
)

print(
    f"Mean ground-truth rank : "
    f"{aggregate['mean_ground_truth_rank']}"
)


# ------------------------------------------------------------
# 12. Failure category table
# ------------------------------------------------------------

print()
print("-" * 60)
print("FAILURE CATEGORIES")
print("-" * 60)

for category, count in sorted(
    category_counts.items(),
    key=lambda x: (-x[1], x[0])
):

    print(
        f"{category:<38} : {count}"
    )


# ------------------------------------------------------------
# 13. Hallucination cases
# ------------------------------------------------------------

print()
print("-" * 60)
print("HALLUCINATION / UNSUPPORTED-CLAIM CASES")
print("-" * 60)

if hallucination_cases:

    for item in hallucination_cases:

        print()
        print(
            f"{item['evaluation_id']}"
        )

        print(
            f"Question: {item['question']}"
        )

        print(
            f"Ground-truth rank: "
            f"{item['ground_truth_rank']}"
        )

        print(
            f"Scores: "
            f"C={item['correctness']}, "
            f"R={item['relevance']}, "
            f"F={item['faithfulness']}, "
            f"Co={item['completeness']}, "
            f"O={item['overall_score']}"
        )

        print(
            "Unsupported claims:"
        )

        for claim_number, claim in enumerate(
            item[
                "unsupported_claims"
            ],
            start=1
        ):

            print(
                f"  {claim_number}. {claim}"
            )

else:

    print(
        "No hallucination cases detected."
    )


# ------------------------------------------------------------
# 14. Low-completeness cases
# ------------------------------------------------------------

print()
print("-" * 60)
print("LOW-COMPLETENESS CASES")
print("-" * 60)

if low_completeness_cases:

    for item in low_completeness_cases:

        print()
        print(
            f"{item['evaluation_id']} | "
            f"Completeness={item['completeness']}/5 | "
            f"Overall={item['overall_score']}/5"
        )

        print(
            f"Question: {item['question']}"
        )

else:

    print(
        "No low-completeness cases detected."
    )


# ------------------------------------------------------------
# 15. Low-correctness cases
# ------------------------------------------------------------

print()
print("-" * 60)
print("LOW-CORRECTNESS CASES")
print("-" * 60)

if low_correctness_cases:

    for item in low_correctness_cases:

        print()
        print(
            f"{item['evaluation_id']} | "
            f"Correctness={item['correctness']}/5 | "
            f"Ground-truth rank="
            f"{item['ground_truth_rank']}"
        )

        print(
            f"Question: {item['question']}"
        )

else:

    print(
        "No low-correctness cases detected."
    )


# ------------------------------------------------------------
# 16. Retrieval-ranking cases
# ------------------------------------------------------------

print()
print("-" * 60)
print("GROUND-TRUTH NOT RANKED FIRST")
print("-" * 60)

if ranking_failure_cases:

    for item in ranking_failure_cases:

        print(
            f"{item['evaluation_id']} -> "
            f"ground-truth rank "
            f"{item['ground_truth_rank']}"
        )

else:

    print(
        "All ground-truth documents ranked first."
    )


# ------------------------------------------------------------
# 17. Save completion metadata
# ------------------------------------------------------------

print()
print("=" * 60)
print("[SUCCESS] RAG FAILURE ANALYSIS COMPLETE")
print("=" * 60)

print()
print("Saved:")

print(
    f"  Results    : {RESULT_PATH}"
)

print(
    f"  Checkpoint : {CHECKPOINT_PATH}"
)


# ------------------------------------------------------------
# 18. Release memory
# ------------------------------------------------------------

del results
del analysis_results
del hallucination_cases
del low_completeness_cases
del low_correctness_cases
del low_faithfulness_cases
del ranking_failure_cases
del retrieval_failure_cases

gc.collect()

print()
print(
    "[INFO] Failure-analysis memory released."
)

DETAILED RAG FAILURE ANALYSIS
[SUCCESS] Loaded 30 answer-quality evaluations.

FAILURE ANALYSIS SUMMARY
Total cases             : 30
Hallucination cases     : 12
Hallucination rate      : 0.4000
Unsupported claims      : 13
Low completeness        : 11
Low correctness         : 2
Low faithfulness        : 2
Ground truth in Top-5  : 30/30
Ground truth not found : 0
Ground truth not Top-1 : 5
Mean ground-truth rank : 1.267

------------------------------------------------------------
FAILURE CATEGORIES
------------------------------------------------------------
no_major_failure                       : 18
generation_faithfulness_failure        : 11
retrieval_ranking_failure              : 1

------------------------------------------------------------
HALLUCINATION / UNSUPPORTED-CLAIM CASES
------------------------------------------------------------

semantic_eval_001
Question: How can I determine if my keyboard is not working properly?
Ground-truth rank: 1
Scores: C=4.0, R=5.0, F=4.0, 

In [36]:
# ============================================================
# CELL 28 — BASELINE EXPERIMENT SUMMARY & RESEARCH RECORD
# ============================================================

import os
import json
import gc
from datetime import datetime


# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

PROJECT_ROOT = r"E:\rag"

DATASET_PROFILE_PATH = os.path.join(
    PROJECT_ROOT,
    "checkpoints",
    "dataset_profile.json"
)

TOKENIZATION_PATH = os.path.join(
    PROJECT_ROOT,
    "checkpoints",
    "tokenization_analysis.json"
)

CHUNK_QUALITY_PATH = os.path.join(
    PROJECT_ROOT,
    "checkpoints",
    "chunk_quality_analysis.json"
)

EMBEDDING_MODEL_PATH = os.path.join(
    PROJECT_ROOT,
    "checkpoints",
    "embedding_model.json"
)

EMBEDDING_GENERATION_PATH = os.path.join(
    PROJECT_ROOT,
    "checkpoints",
    "embedding_generation.json"
)

VECTOR_INDEX_PATH = os.path.join(
    PROJECT_ROOT,
    "checkpoints",
    "vector_index_generation.json"
)

RETRIEVAL_EVALUATION_PATH = os.path.join(
    PROJECT_ROOT,
    "checkpoints",
    "semantic_dense_retrieval_evaluation_v2.json"
)

RETRIEVAL_FAILURE_PATH = os.path.join(
    PROJECT_ROOT,
    "checkpoints",
    "semantic_retrieval_failure_analysis_v2.json"
)

RAG_GENERATION_PATH = os.path.join(
    PROJECT_ROOT,
    "checkpoints",
    "end_to_end_rag_evaluation_v2.json"
)

ANSWER_QUALITY_PATH = os.path.join(
    PROJECT_ROOT,
    "checkpoints",
    "rag_answer_quality_evaluation_v2.json"
)

RAG_FAILURE_PATH = os.path.join(
    PROJECT_ROOT,
    "checkpoints",
    "rag_failure_analysis_v2.json"
)

RESULT_PATH = os.path.join(
    PROJECT_ROOT,
    "data",
    "evaluation",
    "baseline_experiment_record_v2.json"
)

CHECKPOINT_PATH = os.path.join(
    PROJECT_ROOT,
    "checkpoints",
    "baseline_experiment_record_v2.json"
)


# ------------------------------------------------------------
# 2. Atomic JSON save
# ------------------------------------------------------------

def save_json_atomic(data, path):

    temp_path = path + ".tmp"

    try:

        os.makedirs(
            os.path.dirname(path),
            exist_ok=True
        )

        with open(
            temp_path,
            "w",
            encoding="utf-8"
        ) as f:

            json.dump(
                data,
                f,
                indent=2,
                ensure_ascii=False
            )

        os.replace(
            temp_path,
            path
        )

    except Exception:

        if os.path.exists(temp_path):

            try:
                os.remove(temp_path)
            except Exception:
                pass

        raise


# ------------------------------------------------------------
# 3. JSON loader
# ------------------------------------------------------------

def load_json(
    path,
    name
):

    if not os.path.exists(path):

        raise FileNotFoundError(
            f"{name} not found:\n{path}"
        )

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        return json.load(f)


# ------------------------------------------------------------
# 4. Load all persisted experiment artifacts
# ------------------------------------------------------------

try:

    print("=" * 60)
    print("BASELINE EXPERIMENT SUMMARY")
    print("=" * 60)

    dataset_profile = load_json(
        DATASET_PROFILE_PATH,
        "Dataset profile"
    )

    tokenization = load_json(
        TOKENIZATION_PATH,
        "Tokenization analysis"
    )

    chunk_quality = load_json(
        CHUNK_QUALITY_PATH,
        "Chunk quality analysis"
    )

    embedding_model = load_json(
        EMBEDDING_MODEL_PATH,
        "Embedding model record"
    )

    embedding_generation = load_json(
        EMBEDDING_GENERATION_PATH,
        "Embedding generation record"
    )

    vector_index = load_json(
        VECTOR_INDEX_PATH,
        "Vector index record"
    )

    retrieval_evaluation = load_json(
        RETRIEVAL_EVALUATION_PATH,
        "Semantic retrieval evaluation"
    )

    retrieval_failure = load_json(
        RETRIEVAL_FAILURE_PATH,
        "Semantic retrieval failure analysis"
    )

    rag_generation = load_json(
        RAG_GENERATION_PATH,
        "RAG generation record"
    )

    answer_quality = load_json(
        ANSWER_QUALITY_PATH,
        "Answer-quality evaluation"
    )

    rag_failure = load_json(
        RAG_FAILURE_PATH,
        "RAG failure analysis"
    )

    print(
        "[SUCCESS] All baseline artifacts loaded."
    )

except Exception as e:

    print(
        f"[ERROR] Failed to load baseline artifacts: {e}"
    )

    raise


# ------------------------------------------------------------
# 5. Extract dataset information
# ------------------------------------------------------------

try:

    dataset_summary = {

        "dataset_name":
            "Synthetic IT-Related Knowledge Items",

        "dataset_version":
            "Version 3",

        "records":
            100,

        "columns":
            4,

        "baseline_text_fields":
            [
                "ki_topic",
                "ki_text"
            ],

        "excluded_text_fields":
            [
                "alt_ki_text",
                "bad_ki_text"
            ],

        "document_count":
            100,

        "notes":
            (
                "The primary knowledge text was used for the "
                "baseline corpus. Alternative and deliberately "
                "poor text versions were excluded from the "
                "baseline experiment."
            )
    }

    print(
        "[SUCCESS] Dataset summary prepared."
    )

except Exception as e:

    print(
        f"[ERROR] Dataset summary failed: {e}"
    )

    raise


# ------------------------------------------------------------
# 6. Extract chunking configuration
# ------------------------------------------------------------

try:

    chunking_summary = {

        "tokenizer":
            "cl100k_base",

        "chunk_size_tokens":
            500,

        "chunk_overlap_tokens":
            50,

        "chunk_step_tokens":
            450,

        "document_count":
            100,

        "chunk_count":
            164,

        "average_tokens_per_chunk":
            347.00,

        "median_tokens_per_chunk":
            474,

        "minimum_tokens_per_chunk":
            53,

        "maximum_tokens_per_chunk":
            500
    }

    print(
        "[SUCCESS] Chunking summary prepared."
    )

except Exception as e:

    print(
        f"[ERROR] Chunking summary failed: {e}"
    )

    raise


# ------------------------------------------------------------
# 7. Extract embedding configuration
# ------------------------------------------------------------

try:

    embedding_summary = {

        "model":
            "BAAI/bge-small-en-v1.5",

        "dimension":
            384,

        "device":
            "cuda",

        "normalized":
            True,

        "embedding_count":
            164,

        "batch_size":
            32
    }

    print(
        "[SUCCESS] Embedding summary prepared."
    )

except Exception as e:

    print(
        f"[ERROR] Embedding summary failed: {e}"
    )

    raise


# ------------------------------------------------------------
# 8. Extract vector-index configuration
# ------------------------------------------------------------

try:

    vector_index_summary = {

        "index_type":
            "FAISS IndexFlatIP",

        "vector_count":
            164,

        "dimension":
            384,

        "similarity_metric":
            "inner_product",

        "normalized_embeddings":
            True,

        "cosine_similarity_equivalent":
            True
    }

    print(
        "[SUCCESS] Vector-index summary prepared."
    )

except Exception as e:

    print(
        f"[ERROR] Vector-index summary failed: {e}"
    )

    raise


# ------------------------------------------------------------
# 9. Extract retrieval results
# ------------------------------------------------------------

try:

    retrieval_metrics = (
        retrieval_evaluation.get(
            "aggregate",
            retrieval_evaluation
        )
    )

    retrieval_failure_metrics = (
        retrieval_failure.get(
            "aggregate",
            retrieval_failure
        )
    )

    retrieval_summary = {

        "evaluation_type":
            "Natural semantic question retrieval",

        "evaluation_questions":
            30,

        "retrieval_method":
            "dense",

        "top_k":
            5,

        "recall_at_1":
            0.8333,

        "recall_at_3":
            0.9667,

        "recall_at_5":
            1.0000,

        "mrr":
            0.9067,

        "top_1_hits":
            25,

        "top_3_hits":
            29,

        "top_5_hits":
            30,

        "ranking_failure_cases":
            5,

        "retrieval_failure_cases":
            0,

        "mean_ground_truth_rank":
            retrieval_failure_metrics.get(
                "mean_ground_truth_rank"
            )
    }

    print(
        "[SUCCESS] Retrieval results prepared."
    )

except Exception as e:

    print(
        f"[ERROR] Retrieval summary failed: {e}"
    )

    raise


# ------------------------------------------------------------
# 10. Extract generation configuration/results
# ------------------------------------------------------------

try:

    generation_aggregate = rag_generation.get(
        "aggregate",
        {}
    )

    generation_summary = {

        "evaluation_questions":
            generation_aggregate.get(
                "total_questions",
                30
            ),

        "successful_answers":
            generation_aggregate.get(
                "successful_answers",
                30
            ),

        "failed_answers":
            generation_aggregate.get(
                "failed_answers",
                0
            ),

        "generator":
            "qwen3:8b",

        "thinking":
            False,

        "temperature":
            0.0,

        "num_predict":
            256,

        "top_k":
            5,

        "average_generation_time_seconds":
            generation_aggregate.get(
                "average_generation_time_seconds"
            ),

        "total_generation_time_seconds":
            generation_aggregate.get(
                "total_generation_time_seconds"
            ),

        "average_answer_length_characters":
            generation_aggregate.get(
                "average_answer_length_characters"
            )
    }

    print(
        "[SUCCESS] Generation summary prepared."
    )

except Exception as e:

    print(
        f"[ERROR] Generation summary failed: {e}"
    )

    raise


# ------------------------------------------------------------
# 11. Extract answer-quality results
# ------------------------------------------------------------

try:

    answer_quality_aggregate = answer_quality.get(
        "aggregate",
        {}
    )

    answer_quality_summary = {

        "evaluation_questions":
            answer_quality_aggregate.get(
                "total_questions",
                30
            ),

        "mean_correctness":
            answer_quality_aggregate.get(
                "mean_correctness"
            ),

        "mean_relevance":
            answer_quality_aggregate.get(
                "mean_relevance"
            ),

        "mean_faithfulness":
            answer_quality_aggregate.get(
                "mean_faithfulness"
            ),

        "mean_completeness":
            answer_quality_aggregate.get(
                "mean_completeness"
            ),

        "mean_overall_score":
            answer_quality_aggregate.get(
                "mean_overall_score"
            ),

        "overall_score_percentage":
            answer_quality_aggregate.get(
                "overall_score_percentage"
            ),

        "hallucination_count":
            answer_quality_aggregate.get(
                "hallucination_count"
            ),

        "hallucination_rate":
            answer_quality_aggregate.get(
                "hallucination_rate"
            ),

        "unsupported_claim_count":
            answer_quality_aggregate.get(
                "unsupported_claim_count"
            )
    }

    print(
        "[SUCCESS] Answer-quality summary prepared."
    )

except Exception as e:

    print(
        f"[ERROR] Answer-quality summary failed: {e}"
    )

    raise


# ------------------------------------------------------------
# 12. Extract failure-analysis results
# ------------------------------------------------------------

try:

    failure_aggregate = rag_failure.get(
        "aggregate",
        {}
    )

    failure_summary = {

        "total_cases":
            failure_aggregate.get(
                "total_cases",
                30
            ),

        "hallucination_cases":
            failure_aggregate.get(
                "hallucination_cases"
            ),

        "hallucination_rate":
            failure_aggregate.get(
                "hallucination_rate"
            ),

        "unsupported_claims":
            failure_aggregate.get(
                "unsupported_claims"
            ),

        "low_completeness_cases":
            failure_aggregate.get(
                "low_completeness_cases"
            ),

        "low_correctness_cases":
            failure_aggregate.get(
                "low_correctness_cases"
            ),

        "low_faithfulness_cases":
            failure_aggregate.get(
                "low_faithfulness_cases"
            ),

        "ground_truth_in_top5":
            failure_aggregate.get(
                "ground_truth_retrieved_top5"
            ),

        "ground_truth_not_retrieved":
            failure_aggregate.get(
                "ground_truth_not_retrieved"
            ),

        "ground_truth_not_rank1":
            failure_aggregate.get(
                "ground_truth_not_rank1"
            ),

        "mean_ground_truth_rank":
            failure_aggregate.get(
                "mean_ground_truth_rank"
            ),

        "failure_category_counts":
            failure_aggregate.get(
                "failure_category_counts",
                {}
            )
    }

    print(
        "[SUCCESS] Failure-analysis summary prepared."
    )

except Exception as e:

    print(
        f"[ERROR] Failure-analysis summary failed: {e}"
    )

    raise


# ------------------------------------------------------------
# 13. Research interpretation
# ------------------------------------------------------------

research_findings = {

    "retrieval_finding":
        (
            "The dense retriever achieved complete Top-5 "
            "coverage on the 30 natural semantic evaluation "
            "questions. The main retrieval limitation was "
            "ranking ambiguity rather than failure to retrieve "
            "the relevant document."
        ),

    "generation_finding":
        (
            "The RAG generator successfully produced answers "
            "for all 30 evaluation questions. Answer relevance "
            "was very high, while completeness was lower than "
            "the other evaluated dimensions."
        ),

    "faithfulness_finding":
        (
            "The baseline generated generally useful answers, "
            "but unsupported claims were identified in a "
            "subset of responses. This indicates that strong "
            "retrieval does not automatically guarantee fully "
            "grounded generation."
        ),

    "primary_baseline_limitation":
        (
            "The most visible answer-quality weakness is "
            "completeness, followed by unsupported additions "
            "in generated answers."
        ),

    "retrieval_generation_relationship":
        (
            "The baseline results demonstrate that retrieval "
            "coverage and generation quality are related but "
            "distinct components of RAG performance."
        ),

    "experimental_status":
        (
            "Baseline experiment completed. No retrieval, "
            "embedding, chunking, or generation parameters "
            "were modified after baseline evaluation."
        )
}


# ------------------------------------------------------------
# 14. Complete research record
# ------------------------------------------------------------

baseline_record = {

    "experiment":
        {
            "name":
                "Baseline Dense RAG",

            "version":
                "v2",

            "status":
                "completed",

            "purpose":
                (
                    "Establish a reproducible baseline for "
                    "dense Retrieval-Augmented Generation "
                    "using the synthetic IT knowledge corpus."
                )
        },

    "dataset":
        dataset_summary,

    "document_preparation":
        {
            "documents":
                100,

            "text_cleaning":
                (
                    "Conservative whitespace and line-ending "
                    "normalization."
                ),

            "original_dataset_modified":
                False
        },

    "chunking":
        chunking_summary,

    "embedding":
        embedding_summary,

    "vector_index":
        vector_index_summary,

    "retrieval":
        retrieval_summary,

    "generation":
        generation_summary,

    "answer_quality":
        answer_quality_summary,

    "failure_analysis":
        failure_summary,

    "research_findings":
        research_findings,

    "artifacts":

        {
            "documents":
                r"E:\rag\data\processed\rag_documents.json",

            "chunks":
                r"E:\rag\data\processed\rag_chunks.json",

            "embeddings":
                r"E:\rag\data\embeddings\rag_embeddings.json",

            "vector_index":
                r"E:\rag\data\vector_index\dense_index.faiss",

            "semantic_evaluation":
                r"E:\rag\data\evaluation\semantic_evaluation_questions_v2.json",

            "retrieval_evaluation":
                r"E:\rag\data\evaluation\semantic_dense_retrieval_evaluation_v2.json",

            "rag_generation":
                r"E:\rag\data\evaluation\end_to_end_rag_evaluation_v2.json",

            "answer_quality":
                r"E:\rag\data\evaluation\rag_answer_quality_evaluation_v2.json",

            "failure_analysis":
                r"E:\rag\data\evaluation\rag_failure_analysis_v2.json"
        },

    "reproducibility":

        {
            "retrieval_method":
                "dense",

            "embedding_model":
                "BAAI/bge-small-en-v1.5",

            "embedding_dimension":
                384,

            "chunk_size":
                500,

            "chunk_overlap":
                50,

            "top_k":
                5,

            "generator":
                "qwen3:8b",

            "temperature":
                0.0,

            "thinking":
                False,

            "judge_model":
                "qwen3:8b",

            "evaluation_questions":
                30
        },

    "created_at":
        datetime.now().isoformat()
}


# ------------------------------------------------------------
# 15. Save baseline record
# ------------------------------------------------------------

try:

    save_json_atomic(
        baseline_record,
        RESULT_PATH
    )

    save_json_atomic(
        baseline_record,
        CHECKPOINT_PATH
    )

except Exception as e:

    print(
        f"[ERROR] Failed to save baseline record: {e}"
    )

    raise


# ------------------------------------------------------------
# 16. Display final baseline
# ------------------------------------------------------------

print()
print("=" * 60)
print("BASELINE EXPERIMENT RESULTS")
print("=" * 60)

print()
print("DATASET")
print("-" * 60)

print(
    f"Documents             : "
    f"{dataset_summary['document_count']}"
)

print(
    f"Baseline records      : "
    f"{dataset_summary['records']}"
)

print()
print("CHUNKING")
print("-" * 60)

print(
    f"Chunks                : "
    f"{chunking_summary['chunk_count']}"
)

print(
    f"Chunk size            : "
    f"{chunking_summary['chunk_size_tokens']} tokens"
)

print(
    f"Overlap               : "
    f"{chunking_summary['chunk_overlap_tokens']} tokens"
)

print()
print("EMBEDDINGS")
print("-" * 60)

print(
    f"Model                 : "
    f"{embedding_summary['model']}"
)

print(
    f"Dimension             : "
    f"{embedding_summary['dimension']}"
)

print(
    f"Vectors               : "
    f"{embedding_summary['embedding_count']}"
)

print()
print("RETRIEVAL")
print("-" * 60)

print(
    f"Recall@1              : "
    f"{retrieval_summary['recall_at_1']:.4f}"
)

print(
    f"Recall@3              : "
    f"{retrieval_summary['recall_at_3']:.4f}"
)

print(
    f"Recall@5              : "
    f"{retrieval_summary['recall_at_5']:.4f}"
)

print(
    f"MRR                   : "
    f"{retrieval_summary['mrr']:.4f}"
)

print(
    f"Ground-truth rank avg : "
    f"{retrieval_summary['mean_ground_truth_rank']}"
)

print()
print("GENERATION")
print("-" * 60)

print(
    f"Generator             : "
    f"{generation_summary['generator']}"
)

print(
    f"Successful answers    : "
    f"{generation_summary['successful_answers']}/"
    f"{generation_summary['evaluation_questions']}"
)

print(
    f"Average generation   : "
    f"{generation_summary['average_generation_time_seconds']:.4f}s"
)

print()
print("ANSWER QUALITY")
print("-" * 60)

print(
    f"Correctness           : "
    f"{answer_quality_summary['mean_correctness']:.3f}/5"
)

print(
    f"Relevance             : "
    f"{answer_quality_summary['mean_relevance']:.3f}/5"
)

print(
    f"Faithfulness          : "
    f"{answer_quality_summary['mean_faithfulness']:.3f}/5"
)

print(
    f"Completeness          : "
    f"{answer_quality_summary['mean_completeness']:.3f}/5"
)

print(
    f"Overall               : "
    f"{answer_quality_summary['mean_overall_score']:.3f}/5"
)

print(
    f"Overall percentage    : "
    f"{answer_quality_summary['overall_score_percentage']:.2f}%"
)

print()
print("HALLUCINATION")
print("-" * 60)

print(
    f"Hallucination cases   : "
    f"{failure_summary['hallucination_cases']}/"
    f"{failure_summary['total_cases']}"
)

print(
    f"Hallucination rate    : "
    f"{failure_summary['hallucination_rate']:.4f}"
)

print(
    f"Unsupported claims    : "
    f"{failure_summary['unsupported_claims']}"
)

print()
print("FAILURE CATEGORIES")
print("-" * 60)

for category, count in sorted(
    failure_summary[
        "failure_category_counts"
    ].items(),
    key=lambda x: (-x[1], x[0])
):

    print(
        f"{category:<38}: {count}"
    )

print()
print("=" * 60)
print("[SUCCESS] BASELINE EXPERIMENT RECORD CREATED")
print("=" * 60)

print()
print(
    f"Results    : {RESULT_PATH}"
)

print(
    f"Checkpoint : {CHECKPOINT_PATH}"
)


# ------------------------------------------------------------
# 17. Release memory
# ------------------------------------------------------------

del dataset_profile
del tokenization
del chunk_quality
del embedding_model
del embedding_generation
del vector_index
del retrieval_evaluation
del retrieval_failure
del rag_generation
del answer_quality
del rag_failure

gc.collect()

print()
print(
    "[INFO] Baseline-analysis memory released.")

BASELINE EXPERIMENT SUMMARY
[SUCCESS] All baseline artifacts loaded.
[SUCCESS] Dataset summary prepared.
[SUCCESS] Chunking summary prepared.
[SUCCESS] Embedding summary prepared.
[SUCCESS] Vector-index summary prepared.
[SUCCESS] Retrieval results prepared.
[SUCCESS] Generation summary prepared.
[SUCCESS] Answer-quality summary prepared.
[SUCCESS] Failure-analysis summary prepared.

BASELINE EXPERIMENT RESULTS

DATASET
------------------------------------------------------------
Documents             : 100
Baseline records      : 100

CHUNKING
------------------------------------------------------------
Chunks                : 164
Chunk size            : 500 tokens
Overlap               : 50 tokens

EMBEDDINGS
------------------------------------------------------------
Model                 : BAAI/bge-small-en-v1.5
Dimension             : 384
Vectors               : 164

RETRIEVAL
------------------------------------------------------------
Recall@1              : 0.8333
Recall@3    